# Warehouse operations: analysis, traceability and historical scenarios

This notebook is a read-only proof of warehouse analysis using **Daily_Inbound_Report.xlsx** and **Daily_Outbound_Report.xlsx**. It supports receiving and dispatch investigation, recorded traceability, handover performance, data-quality checks and historical what-if comparisons. Handover to a forwarder is not evidence of delivery to the customer.

## Getting started

1. Select a Python/Jupyter kernel with pandas, openpyxl, ipywidgets and matplotlib available.
2. Run **Run All** from the top. ~~The two Excel files must be accessible at the paths shown in the import cell.~~ **→ Put both Excel files in a `data` folder next to this notebook (paths changed in the import cell).** Preparation cells build shared calculations; they are not interactive result screens.
3. Use the controls beside the analysis you want. Widgets require a running kernel. After editing a time field, press that section's Apply button; valid date selections update automatically. Where present, **Preview rows** sets the shared table page/preview size. Changing it does not change the analysed population.
4. Use each chart's Show checkbox to display or hide it. Inspect a result's line tables and event history, or use Section 5 to search a reference directly. Tables labelled 'first N' are previews rather than complete lists.

| Section | Use it to… |
|---|---|
| 1 | Check the loaded period, timestamp alignment and assumptions. |
| 2.1 | Review shipping activity, open orders, recorded readiness and dispatch records. |
| 2.2–2.3 | Compare receiving quantities and investigate dock waits. |
| 3–3.1 | Follow milk run IDs and inspect elapsed-time exceptions. |
| 4 | Review event activity and recent records. |
| 5–5.1 | Search records or evaluate the separate experimental ASN-candidate rule. |
| 6–6.1 | Analyse handover lateness and records excluded from measurement. |
| 7–7.1 | Investigate data quality and linked milk run quantity agreement. |
~~| 8 | See evidence-based attention checks and suggested follow-up. |~~ **→ | 8 | See the attention checks, now named IN-01, OUT-01, IN-03 and DQ-01 as in the prototype. |**
| 9 | Compare historical actual outcomes with constrained handover scenarios. |

## Shared interpretation rules

- **Dataset Today** is the date of the latest actual event after inbound alignment, not the computer date. Snapshot controls start at **09:00:00**. The historical scenario starts with the preceding seven planned days where available.
- **Snapshot, period and full-history scopes differ.** A snapshot limits observed events to a selected time. Period views include whole endpoint days. Full-history analyses use recorded outcomes across the loaded extract. Each section states its scope; independent controls do not silently change another section's time.
- **Rows, orders, ASNs, ASN lines and carton/LPN IDs are different units.** The same entity may occupy several source rows. Do not add counts with different units or treat source-row counts as shipments.
- **Quantity grain matters.** ~~Receiving expected quantity counts once per ASN/reference line;~~ **→ Receiving expected quantity counts once per WMS ASN# + Ref Line# + Functional Packing line;** received rows are summed. ~~Shipping activity and general order inspectors sum reported outbound rows, which can repeat an order-line quantity across cartons. Section 7.1 instead counts one consistent shipped quantity per order line. These measures are not interchangeable or independently verified physical totals.~~ **→ Shipping activity (2.1), the record inspector (5) and 7.1 now all count Qty Shipped once per order line (assumption WA-16, DSV to confirm in Q25). It is still not a verified physical total.** No unit-of-measure field is supplied.
- **A recorded ID link is not quantity reconciliation or verified stock provenance.** Missing counterparts may reflect extract coverage. Inferred candidates belong to the optional experiment, not confirmed operational links.
- **Future event timestamps and observed quantities are masked in snapshot evidence.** Static plans, classifications and assignments remain the extract's values because historical versions are unavailable. 'No observed receipt/handover' does not prove that the physical event never occurred. An undefined rate or unassessable result is not a healthy zero.
- **Original data is retained.** Duplicate patterns, discrepancies and dates are not automatically corrected. Readiness timestamps are proxies; no staffing, capacity or pickup schedule is inferred. Source-row numbers are imported DataFrame indices, not Excel worksheet row numbers.


## How to read this review version (16 Nov 2026)

This copy of **Feasure_Final** marks what changed and how each section relates to the prototype (`twin.html`). The original notebook is unchanged.

| Mark | Meaning |
|---|---|
| ~~struck text~~ **→ new text** | Text that was wrong, or does not match the prototype or the DSV rules. The replacement follows the arrow. |
| 🟦 In the prototype | This analysis already appears in the prototype (which screen and rule). |
| 🟧 Overlaps but differs | The prototype covers the same thing, but the rule, threshold or wording is not the same. Agree one version. |
| 🟨 Not in the prototype yet | Useful analysis the prototype does not show. Decide at the handover: add it, keep it notebook-only, or drop it. |
| 🟥 Changed in this version | Code was edited. Search the code for `REVIEW 16 Nov`. |

**In the code:** each changed line is kept as a comment starting `# [OLD]`, with a `# REVIEW 16 Nov - CHANGED:` comment above it giving the reason. The new line follows directly below. `NOT CHANGED, TO DO` marks lines we flagged but left alone.

**Summary of code changes**

| # | Change | Where | Why |
|---|---|---|---|
| 1 | File paths now read from a `data` folder | 1 import cell | Old paths only worked on one laptop |
| 2 | Inbound line key = WMS ASN# + Ref Line# + **Functional Packing** | 2.2, 5, 7, 8 | DSV A5. With the 2-part key 109 lines look exactly 2× expected; with the 3-part key all 36,533 match |
| 3 | Qty Shipped counted **once per order line** | 2.1 totals, groups, trend; 5 inspector | Row sums give 1,038,100,865 vs 695,808,651 (about 49% too high). Same grain as 7.1 and WA-16 |
| 4 | Rule IDs ASN-01, ORD-01, QTY-01, DATE-01 → IN-01, OUT-01, IN-03, DQ-01 | 8 | One naming scheme for notebook, deck and prototype |
| 5 | Visible "late" labels → "released after plan" | 6 | DSV, 11 Nov: the planned date alone does not prove lateness; treat it as a review signal |

These changes were checked against the full DSV files, but the notebook was **not re-run end to end** here (no Jupyter widgets in this environment). Please Run All once before relying on it.


In [ ]:

import pandas as pd
from pathlib import Path

# REVIEW 16 Nov - CHANGED: file paths.
# Why: the absolute C:/Users/warit/... paths only work on one laptop. Put both Excel files in a
# "data" folder next to this notebook so anyone in the team can Run All without editing code.
# [OLD] outbound = pd.read_excel("C:/Users/warit/OneDrive/SUTD/7th Term/Supply Chain/Digital Twin/Daily_Outbound_Report.xlsx")
# [OLD] inbound = pd.read_excel("C:/Users/warit/OneDrive/SUTD/7th Term/Supply Chain/Digital Twin/Daily_Inbound_Report.xlsx")
DATA_DIR = Path("data")
outbound = pd.read_excel(DATA_DIR / "Daily_Outbound_Report.xlsx")
inbound = pd.read_excel(DATA_DIR / "Daily_Inbound_Report.xlsx")



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Data context, coverage and assumptions

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Snapshot time, +8 h inbound shift and data coverage (DQ-05 "Clock drift (+8 h)", resolved on load).  

**Purpose:** Establish what the files cover before interpreting counts, history or scenario results.

The snapshot card reports the latest actual event, Dataset Today and loaded row counts. It is not a live-feed indicator or an independently recorded export time. Planned dates do not determine Today.

The coverage table shows each timestamp field's first/last observation, valid and missing/invalid source-row counts, and distinct IDs with a timestamp. The optional coverage bars show **first-to-last spans**, not continuous or complete coverage between those endpoints. Repeated timestamps across item rows can contribute several source rows to one event.

**Inbound alignment:** Dock and goods-receipt times in the prepared inbound copy are shifted **+8 hours**; original imported values remain available. Outbound times are not shifted. This is a working project assumption, not a timezone declaration verified by the files. Cross-report durations and receipt-based readiness depend on it.

**Assumption register:** Tick Show assumption register to inspect each topic, status, basis and practical limit. Confirmed refers to an explicit user/design instruction; Verified refers to a narrowly checked file calculation; Working is an analytical interpretation; Unresolved is information the extracts cannot establish. These are not blanket certifications of warehouse accuracy.

**Coverage limits:** Orders absent from the extract cannot be reconstructed. Open work and lateness near the end may be understated, while planned dates before the event coverage can be selectively represented by late handovers. Choose an interior historical cohort where appropriate and read the exclusion/sample counts. A period with no recorded events does not establish that no work occurred.


In [2]:
# "Today" is the latest actual event recorded in the reports, not the computer date.
# Apply the same inbound-time assumption used in the preparation below.
INBOUND_OFFSET_HOURS = 8
actual_times = []
for column in ["Dock in Date & Time", "GR Date & Time"]:
    actual_times.append(pd.to_datetime(inbound[column], errors="coerce", format="mixed")
                        + pd.Timedelta(hours=INBOUND_OFFSET_HOURS))
for column in ["Orderdate Date & Time", "Receiving of Shipping Carton into Cyclone Date & Time",
               "Packing Confirmation Timestamp", "Cargo Release Handover to Forwarder Date & Time"]:
    actual_times.append(pd.to_datetime(outbound[column], errors="coerce", format="mixed"))
recorded_times = pd.concat(actual_times).dropna()
if recorded_times.empty:
    raise ValueError("No valid actual event timestamps were found in the reports.")
DATA_LATEST_EVENT = recorded_times.max()
DATA_TODAY = DATA_LATEST_EVENT.normalize().date()
DEFAULT_TIME = "09:00:00"
AS_OF = pd.Timestamp(DATA_TODAY) + pd.Timedelta(hours=9)
RECEIPT_START = DATA_TODAY
RECEIPT_END = DATA_TODAY
ORDER_FILTER = "All open"
PREVIEW_ROWS = 10

def data_date_label(day):
    date = pd.Timestamp(day).date()
    return f"Today — {date}" if date == DATA_TODAY else f"{date} — History"

def parse_clock_time(value):
    from datetime import datetime
    for format_ in ["%H:%M:%S", "%H:%M"]:
        try:
            return datetime.strptime(value.strip(), format_).time()
        except ValueError:
            pass
    raise ValueError("Enter time as HH:MM or HH:MM:SS.")

print(f"Today in this dataset: {DATA_TODAY}")
print(f"Latest recorded actual event: {DATA_LATEST_EVENT}")
print("Defaults follow the reports whenever Run All reloads them. Planned dates are not used to define Today.")

print(f"Default snapshot time: {DEFAULT_TIME}")


Today in this dataset: 2026-08-24
Latest recorded actual event: 2026-08-24 23:48:37
Defaults follow the reports whenever Run All reloads them. Planned dates are not used to define Today.
Default snapshot time: 09:00:00


### 1.1 Timestamp and record preparation

The preparation cells parse timestamp fields, align inbound times and normalize identifiers in working copies. Several source rows can belong to the same order, ASN or carton, so entity counts use the corresponding identifiers.

| Prepared entity | Aggregation used |
|---|---|
| Order | Earliest recorded creation and planned date; latest recorded handover. Conflicting plans are checked separately in performance and attention analyses. |
| ASN | Earliest recorded dock time and latest recorded goods receipt. ASN-level views do not represent a separate partly-received state. |
| Carton/LPN | Identified by recorded IDs. Blank IDs cannot support exact carton tracing. |

Orderdate Date & Time represents **creation**, not a separately verified release-to-floor event. The field Cargo Release Handover to Forwarder Date & Time is interpreted as recorded handover. These preparation cells support later views; they do not change the Excel files.


In [3]:
from IPython.display import display

# Working assumption from the project: inbound timestamps need +8 hours.
ob = outbound.copy()
ib = inbound.copy()
created_col = "Orderdate Date & Time"
released_col = "Cargo Release Handover to Forwarder Date & Time"
scan_col = "Receiving of Shipping Carton into Cyclone Date & Time"
packed_col = "Packing Confirmation Timestamp"
for col in [created_col, released_col, scan_col, packed_col, "Planned Post Date"]:
    ob[col] = pd.to_datetime(ob[col], errors="coerce", format="mixed")
for col in ["Dock in Date & Time", "GR Date & Time"]:
    ib[col] = pd.to_datetime(ib[col], errors="coerce", format="mixed") + pd.Timedelta(hours=INBOUND_OFFSET_HOURS)

# Normalize join keys in copies. Import numeric IDs consistently on both sides.
def clean_id(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).replace("", pd.NA)
for col in ["Order Key", "Carton ID"]:
    ob[col] = clean_id(ob[col])
for col in ["WMS ASN#", "LPN"]:
    ib[col] = clean_id(ib[col])

orders = ob.groupby("Order Key").agg(
    created=(created_col, "min"), released=(released_col, "max"),
    planned=("Planned Post Date", "min"), flow=("DN Type", "first"),
).reset_index()
asns = ib.groupby("WMS ASN#").agg(
    dock=("Dock in Date & Time", "min"), receipt=("GR Date & Time", "max"),
).reset_index()

# One row per carton ID across both reports; blank IDs cannot be traced.
out_cartons = ob.groupby("Carton ID").agg(
    created=(created_col, "min"), scan=(scan_col, "min"),
    packed=(packed_col, "max"), released=(released_col, "max"),
)
in_cartons = ib.groupby("LPN").agg(
    dock=("Dock in Date & Time", "min"), receipt=("GR Date & Time", "max"),
)
cartons = out_cartons.join(in_cartons, how="outer")


### 1.2 Loaded snapshot and coverage views

The following controls display the snapshot card and coverage table. Tick Show extract coverage bars for the timestamp-span chart, or Show assumption register for the status/basis/limit table. Both displays are optional; the imported records and calculations are unaffected by visibility.


In [4]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from html import escape as context_escape

context_stream_specs=[("Inbound dock (+8h)",ib,"Dock in Date & Time","WMS ASN#"),
    ("Inbound receipt (+8h)",ib,"GR Date & Time","WMS ASN#"),
    ("Outbound creation",ob,created_col,"Order Key"),("Outbound packing",ob,packed_col,"Carton ID"),
    ("Outbound Cyclone scan",ob,scan_col,"Carton ID"),("Outbound handover",ob,released_col,"Order Key")]
context_coverage=pd.DataFrame([{"Event stream":name,"Excel timestamp":column,"Record ID field":idcol,"First observed":frame[column].min(),
    "Last observed":frame[column].max(),"Source rows":len(frame),"Valid timestamp rows":int(frame[column].notna().sum()),
    "Missing/invalid timestamp rows":int(frame[column].isna().sum()),"Distinct IDs with timestamp":frame.loc[frame[column].notna(),idcol].nunique()}
    for name,frame,column,idcol in context_stream_specs])
context_latest_out=pd.concat([ob[c] for c in [created_col,packed_col,scan_col,released_col]]).max()
context_snapshot=widgets.HTML(value="<div style='border:1px solid #888;padding:12px;max-width:950px'>"
    "<b>Historical data snapshot</b><br>Latest recorded actual event (adjusted alignment): <b>"+context_escape(str(DATA_LATEST_EVENT))+"</b><br>"
    "Dataset Today: "+str(DATA_TODAY)+"; default snapshot clock: 09:00:00<br>"
    "Inbound rows: "+f"{len(inbound):,}"+"; outbound rows: "+f"{len(outbound):,}"+"<br>"
    "Live feed: No. Last outbound actual event: "+context_escape(str(context_latest_out))+"</div>")
context_assumptions=pd.DataFrame([
    ("Today and 09:00 defaults","Confirmed","User instruction: latest actual event day, not the computer day.","Does not establish the extract export time."),
    ("Loaded row counts and coverage endpoints","Verified","Calculated directly from the two imported files and parsed timestamps.","A first/last span does not establish completeness between them."),
    ("Read-only source handling","Confirmed","Design: calculations use prepared copies; no source Excel write operations.","Derived values are not repairs to source data."),
    ("Inbound +8-hour alignment","Working",f"INBOUND_OFFSET_HOURS = {INBOUND_OFFSET_HOURS} in the preparation cell.","Not confirmed as an actual timezone conversion by these files."),
    ("Expected quantity once per ASN/reference line","Working","Repeated expected values are treated as a line total; received rows are summed.","Conflicting/missing expected quantities need review."),
    # REVIEW 16 Nov - CHANGED: all shipped-quantity views now use one grain.
    # [OLD] ("Outbound quantity grain","Working","Section 7.1 counts one consistent Qty Shipped per order line.","Older shipping-activity views sum source rows; these are different measures."),
    ("Outbound quantity grain","Working","Qty Shipped is counted once per Order Key + Order Line Number in 2.1, 5 and 7.1 (WA-16).","DSV to confirm (Q25). Row sums would be about 49% higher."),
    ("ID-based cross-report link","Working","Exact LPN/Carton ID equality is a recorded association.","Does not prove physical stock allocation or reconcile contents."),
    ("Definition of late","Working","A–D are explicit alternative rules; A is the default.","No confirmed customer/carrier SLA is available."),
    ("Order creation / work release","Working","Orderdate Date & Time is used as creation, not a separate verified floor-release event.","An earlier scenario cannot precede creation."),
    ("Readiness proxy","Working","Packing for non-milk; Cyclone scan, or unambiguous exact inbound receipt fallback, for milk.","A system timestamp does not establish labour/capacity availability."),
    ("Open-work completeness","Unresolved","The extract cannot enumerate missing orders that were still open.","Latest-day queues and lateness may be understated."),
    ("Historic assignments and planned-date changes","Unresolved","No versions of plans, carton assignments or forwarder fields are supplied.","Historical views retain the extract's static fields."),
    ("Physical capacity and collection schedule","Unresolved","No labour, dock-capacity, equipment or actual pickup-schedule data.","Scenario results are historical arithmetic, not a feasible operating schedule or forecast."),
    ("Section 5.1 inferred source ASN","Working","Optional SKU/lot latest-prior receipt experiment, separate from exact links.","Not used to manufacture scenario readiness links.")],columns=["Topic","Status","Basis","Practical limit"])
context_chart_show=widgets.Checkbox(value=False,description="Show extract coverage bars",indent=False)
context_register_show=widgets.Checkbox(value=False,description="Show assumption register",indent=False)
context_output=widgets.Output()
def refresh_context(change=None):
    def render():
        display(context_coverage)
        print("Coverage bars are observed timestamp spans, not proof of a continuous or complete event feed.")
        if context_chart_show.value:
            valid=context_coverage.loc[context_coverage["First observed"].notna()]
            fig,ax=plt.subplots(figsize=(11,4.5))
            for y,(_,row) in enumerate(valid.iterrows()):
                ax.plot([row["First observed"],row["Last observed"]],[y,y],linewidth=8,solid_capstyle="butt",color="#176b87",alpha=.65)
                ax.scatter([row["First observed"],row["Last observed"]],[y,y],s=20,color="#176b87")
            ax.set_yticks(range(len(valid)),valid["Event stream"]);ax.invert_yaxis()
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"));ax.tick_params(axis="x",rotation=25)
            ax.set_title("First-to-last observed timestamps by source field");ax.grid(axis="x",alpha=.2)
            fig.tight_layout();display(fig);plt.close(fig)
        if context_register_show.value:display(context_assumptions)
    # This cell precedes the shared output-replacement helper.
    from IPython.utils.capture import capture_output
    with capture_output() as captured:render()
    outputs=[]
    if captured.stdout:outputs.append({"output_type":"stream","name":"stdout","text":captured.stdout})
    if captured.stderr:outputs.append({"output_type":"stream","name":"stderr","text":captured.stderr})
    for item in captured.outputs:outputs.append({"output_type":"display_data","data":item.data,"metadata":item.metadata})
    context_output.outputs=tuple(outputs)
context_chart_show.observe(refresh_context,names="value");context_register_show.observe(refresh_context,names="value")
display(widgets.VBox([context_snapshot,widgets.HTML("<b>Coverage warning:</b> open work and late outcomes near the extract edges may be incomplete. These are historical files, not today's live operations."),
    context_chart_show,context_register_show,context_output]))
refresh_context()


## 2. Warehouse activity and outstanding work

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** The Inbound and Outbound team screens show the same split between activity and open work at a chosen time.  

This section separates **full-day activity** from **work outstanding at a selected historical time**. Outbound and inbound views have independent period/time controls. Shared preparation cells below calculate the models used by later sections; the interactive results begin at 2.1.

Dates offer recorded activity days, labelled Today or History. A day without events can still contain outstanding work even if it is absent from an availability-based dropdown. Counts printed beside a date are activity/source-row indicators, not evidence of complete coverage.


In [24]:
# Section 3 uses the prepared copies from section 2. Source reports stay unchanged.
kpi_out = ob.copy()
kpi_out["Qty Shipped"] = pd.to_numeric(kpi_out["Qty Shipped"], errors="raise")
if kpi_out["Qty Shipped"].isna().any():
    raise ValueError("Missing Qty Shipped values need review before totalling.")
kpi_out["is_milkrun"] = kpi_out["DN Type"].astype("string").str.contains("milkrun", case=False, na=False)
kpi_in = ib.copy()
kpi_in["is_milkrun"] = kpi_in["Shipment Type"].astype("string").str.contains("milkrun", case=False, na=False)
kpi_asns = kpi_in.groupby("WMS ASN#").agg(
    dock=("Dock in Date & Time", "min"), receipt=("GR Date & Time", "max"),
    shipment_type=("Shipment Type", "first"),
).reset_index()
# REVIEW I2 - CHANGED: count ASN lines on the 3-part key (DSV A5), same as the app's build_data.py.
# [OLD] lines=("Ref Line#", "nunique"),
lines3 = kpi_in.drop_duplicates(["WMS ASN#", "Ref Line#", "Functional Packing"]).groupby("WMS ASN#").size()
kpi_asns["lines"] = kpi_asns["WMS ASN#"].map(lines3).fillna(0).astype(int)

# Readiness uses distinct (order, carton) pairs, not repeated item rows.
kpi_order_cartons = kpi_out.dropna(subset=["Order Key", "Carton ID"]).groupby(["Order Key", "Carton ID"]).agg(
    created=(created_col, "min"), packed=(packed_col, "max"), scan=(scan_col, "min"),
    released=(released_col, "max"), is_milkrun=("is_milkrun", "max"),
).reset_index()
inbound_receipt_by_id = kpi_in.dropna(subset=["LPN"]).groupby("LPN")["GR Date & Time"].min()
kpi_order_cartons["ready_time"] = kpi_order_cartons.packed
milk_pairs = kpi_order_cartons.is_milkrun
kpi_order_cartons.loc[milk_pairs, "ready_time"] = kpi_order_cartons.loc[milk_pairs, "scan"].fillna(
    kpi_order_cartons.loc[milk_pairs, "Carton ID"].map(inbound_receipt_by_id))

kpi_cartons = kpi_out.dropna(subset=["Carton ID"]).groupby("Carton ID").agg(
    created=(created_col, "min"), released=(released_col, "max"),
    packed=(packed_col, "max"), is_milkrun=("is_milkrun", "max"),
).reset_index()

def join_ids(values):
    return ", ".join(sorted(set(values.dropna().astype(str))))

milk_in = kpi_in.loc[kpi_in.is_milkrun & kpi_in.LPN.notna()].groupby("LPN").agg(
    receipt=("GR Date & Time", "min"), ASNs=("WMS ASN#", join_ids),
).rename_axis("HU / carton ID").reset_index()
# The exact trace follows the documented direction: inbound to ABC, outbound from ABC.
# Other outbound milk-run directions can contribute to quantity share but not this cohort.
trace_outbound = kpi_out["DN Type"].astype("string").str.strip().str.casefold().eq("milkrun from abc")
milk_out = kpi_out.loc[trace_outbound & kpi_out["Carton ID"].notna()].groupby("Carton ID").agg(
    created=(created_col, "min"), released=(released_col, "max"), Orders=("Order Key", join_ids),
).rename_axis("HU / carton ID").reset_index()

def kpi_cohorts(at):
    t = pd.Timestamp(at)
    inbound_seen = milk_in.loc[milk_in.receipt <= t]
    outbound_seen = milk_out.loc[milk_out.created <= t]
    result = inbound_seen.merge(outbound_seen, on="HU / carton ID", how="outer", indicator=True)
    result["Cohort"] = result["_merge"].map({"both": "Exactly linked", "left_only": "Inbound only", "right_only": "Outbound only"}).astype(str)
    # Do not display an actual handover time until it has happened.
    result["Visible handover"] = result.released.where(result.released <= t)
    return result.drop(columns=["_merge", "released"])

def kpi_snapshot(at):
    t = pd.Timestamp(at)
    since = t - pd.Timedelta(hours=24)
    visible, opened = order_snapshot(t)
    dock = kpi_asns.loc[(kpi_asns.dock <= t) & (kpi_asns.receipt.isna() | (kpi_asns.receipt > t))].copy()
    dock["Waiting (hours)"] = (t - dock.dock).dt.total_seconds() / 3600
    dock = dock.sort_values("Waiting (hours)", ascending=False)
    packed = kpi_cartons.loc[~kpi_cartons.is_milkrun & (kpi_cartons.created <= t) &
        (kpi_cartons.packed <= t) & (kpi_cartons.released.isna() | (kpi_cartons.released > t))].copy()
    handovers = kpi_cartons.loc[kpi_cartons.released.gt(since) & kpi_cartons.released.le(t)].copy()
    created24 = orders.loc[orders.created.gt(since) & orders.created.le(t)].copy()
    released24 = orders.loc[orders.released.gt(since) & orders.released.le(t)].copy()
    cohort = kpi_cohorts(t)
    waiting_milk = cohort.loc[cohort.receipt.notna() & cohort["Visible handover"].isna()].copy()
    pairs = kpi_order_cartons.loc[kpi_order_cartons["Order Key"].isin(opened["Order Key"])].copy()
    pairs["ready"] = pairs.ready_time.le(t)
    ready_counts = pairs.groupby("Order Key").agg(Cartons=("Carton ID", "nunique"), Ready=("ready", "sum"))
    readiness = opened.merge(ready_counts, on="Order Key", how="left")
    readiness[["Cartons", "Ready"]] = readiness[["Cartons", "Ready"]].fillna(0).astype(int)
    readiness["Readiness"] = "Not all recorded cartons ready"
    readiness.loc[readiness.Cartons.eq(readiness.Ready) & readiness.Cartons.gt(0), "Readiness"] = "All recorded cartons ready"
    readiness.loc[readiness.Cartons.eq(0), "Readiness"] = "No carton IDs in extract"
    readiness["Ready / recorded cartons"] = readiness.Ready.astype(str) + " / " + readiness.Cartons.astype(str)
    values = {
        "ASNs at dock awaiting receipt": len(dock),
        "Milk HUs received, no observed handover": len(waiting_milk),
        "Orders created, awaiting handover": len(opened),
        "Packed non-milk cartons awaiting handover": len(packed),
        "Cartons handed over in last 24h": len(handovers),
        "Open orders": len(opened),
        "Open orders past planned day": int(opened.Priority.eq("Overdue").sum()),
        "Open orders due today": int(opened.Priority.eq("Due today").sum()),
        "Orders released in last 24h": len(released24),
        "Orders created in last 24h": len(created24),
        "Open orders with all recorded cartons ready": int(readiness.Readiness.eq("All recorded cartons ready").sum()),
    }
    return dict(values=values, dock=dock, opened=opened, packed=packed, handovers=handovers,
                created24=created24, released24=released24, cohort=cohort,
                waiting_milk=waiting_milk, readiness=readiness)

kpi_available_days = sorted(set(pd.concat([orders.created, orders.released, kpi_asns.dock, kpi_asns.receipt]).dropna().dt.date) | {DATA_TODAY})



In [6]:
# Shared filter semantics: only actual source values, including an explicit missing option.
ALL_FILTER = "__ALL_FILTER__"
MISSING_FILTER = "__MISSING_FILTER__"

def filter_options(values):
    clean=values.astype("string").str.strip().replace("",pd.NA)
    options=[("All",ALL_FILTER)]+[(str(value),str(value)) for value in sorted(clean.dropna().unique())]
    if clean.isna().any():options.append(("Missing / blank",MISSING_FILTER))
    return options

def source_filter(values,choice):
    if choice is None or choice==ALL_FILTER:return pd.Series(True,index=values.index)
    clean=values.astype("string").str.strip().replace("",pd.NA)
    return clean.isna() if choice==MISSING_FILTER else clean.eq(str(choice)).fillna(False)

def filter_widget(values,label,width="340px"):
    return widgets.Dropdown(options=filter_options(values),value=ALL_FILTER,description=label,
        style={"description_width":"initial"},layout=widgets.Layout(width=width))

def one_or_multiple(values):
    items=sorted(set(values.dropna().astype(str).str.strip())-{""})
    return items[0] if len(items)==1 else ("Multiple values" if items else "Missing / blank")

order_search_attributes=kpi_out.groupby("Order Key").agg(
    Final_forwarder=("Final Forwarder",one_or_multiple),
    Default_forwarder=("Default Forwarder",one_or_multiple),
    Country=("Country Code",one_or_multiple),
    External_orders=("External Order Key",join_ids),
    DN_types=("DN Type",one_or_multiple),
).reset_index()

def observed_trace_attributes(at):
    t=pd.Timestamp(at)
    incoming=kpi_in.loc[kpi_in.is_milkrun & kpi_in["GR Date & Time"].le(t) & kpi_in.LPN.notna()]
    outgoing=kpi_out.loc[trace_outbound & kpi_out[created_col].le(t) & kpi_out["Carton ID"].notna()]
    in_skus=incoming.groupby("LPN").SKU.agg(join_ids).rename("Inbound SKUs")
    out_attrs=outgoing.groupby("Carton ID").agg(**{"Outbound SKUs":("SKU",join_ids),"Final forwarders":("Final Forwarder",join_ids)})
    attrs=in_skus.to_frame().join(out_attrs,how="outer").rename_axis("HU / carton ID").reset_index()
    if attrs.empty:
        attrs["SKUs"]=pd.Series(index=attrs.index,dtype="string")
    else:
        attrs["SKUs"]=attrs[["Inbound SKUs","Outbound SKUs"]].fillna("").agg(lambda row:", ".join(sorted(set(token.strip() for value in row for token in value.split(",") if token.strip()))),axis=1)
    return attrs,incoming,outgoing


In [7]:
# Chart helpers shared by the existing operational views.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def display_operational_figure(fig):
    fig.tight_layout()
    display(fig)
    plt.close(fig)

def plot_group_quantity(table, column, title, ylabel="Reported quantity"):
    if table.empty:
        print("No matching records to chart.")
        return
    values = table[column].sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(9, max(3, len(values)*0.32+1)))
    ax.barh([str(x) for x in values.index], values.values, color="#176b87")
    ax.set_title(title + (" — top 15; full table below" if len(table)>15 else ""))
    ax.set_xlabel(ylabel); ax.grid(axis="x", alpha=0.2)
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
    display_operational_figure(fig)

# REVIEW 16 Nov - ADDED helper: Qty Shipped is an order-line total repeated on every carton row
# (all 10,684 multi-carton lines repeat the same value), so count it once per Order Key + Order Line Number.
# Same grain as Section 7.1, Assumption Log WA-16 and the prototype. DSV still to confirm (Q&A Log Q25).
def once_per_order_line(rows, time_column=None):
    work = rows.sort_values(time_column, kind="stable") if time_column else rows
    return work.drop_duplicates(["Order Key", "Order Line Number"], keep="first")

def shipping_group_totals(rows, column):
    # REVIEW 16 Nov - CHANGED: group totals count each order line once (helper above). Why: row sums double-count split lines.
    # [OLD] group = rows[column].astype("string").str.strip().replace("", pd.NA).fillna("Missing " + column)
    rows = once_per_order_line(rows)
    group = rows[column].astype("string").str.strip().replace("", pd.NA).fillna("Missing " + column)
    return rows.groupby(group)["Qty Shipped"].sum().rename_axis(column).to_frame("Shipped quantity [Qty Shipped]")

def shipping_time_totals(rows, frequency, basis):
    column = released_col if basis == "Handover time" else created_col
    # REVIEW 16 Nov - CHANGED: count each order line once, at its earliest timestamp. Why: row sums double-count split lines.
    # [OLD] valid = rows.loc[rows[column].notna()].copy()
    valid = once_per_order_line(rows.loc[rows[column].notna()], column).copy()
    step = "D" if frequency == "Daily" else "h"
    result = valid.groupby(valid[column].dt.floor(step))["Qty Shipped"].sum().sort_index()
    if not result.empty:
        result = result.reindex(pd.date_range(result.index.min(), result.index.max(), freq=step), fill_value=0)
    return result.rename_axis(f"{basis} [{column}]").to_frame("Shipped quantity [Qty Shipped]")

def zone_for_line(values):
    zones = sorted(set(values.dropna().astype(str).str.strip()) - {""})
    return zones[0] if len(zones)==1 else ("Multiple zones (unallocated)" if zones else "Unknown zone")

# REVIEW 16 Nov - CHANGED: inbound line key now includes Functional Packing (DSV A5).
# Why: rows with the same ASN# + Ref Line# but different Functional Packing (e.g. ITR vs ITY) are separate lines.
# On the 2-part key 109 lines look exactly 2x expected; on the 3-part key all 36,533 lines match.
# [OLD] inbound_zone_map = ib.groupby(["WMS ASN#", "Ref Line#"], dropna=False)["Putaway Zone"].agg(zone_for_line).rename("_ZoneGroup")
inbound_zone_map = ib.groupby(["WMS ASN#", "Ref Line#", "Functional Packing"], dropna=False)["Putaway Zone"].agg(zone_for_line).rename("_ZoneGroup")

def inbound_zone_totals(selected, summary):
    # REVIEW 16 Nov - CHANGED: merge on the same 3-part line key as receiving_totals (see zone map above).
    # [OLD] lines = summary["lines"].merge(inbound_zone_map.reset_index(), on=["WMS ASN#", "Ref Line#"], how="left", validate="one_to_one")
    lines = summary["lines"].merge(inbound_zone_map.reset_index(), on=["WMS ASN#", "Ref Line#", "Functional Packing"], how="left", validate="one_to_one")
    table = lines.groupby("_ZoneGroup", dropna=False).agg(expected=("expected", "sum"), received=("received", "sum"))
    return table.rename(columns={"expected":"Expected quantity [Qty Exp]", "received":"Received quantity [Qty Rcvd]"}).rename_axis("Putaway zone [Putaway Zone]")


In [8]:
# Timing population: exact milk-run ID links; ambiguous source associations stay flagged.
timing_in = kpi_in.loc[kpi_in.is_milkrun & kpi_in.LPN.notna()].groupby("LPN").agg(
    dock=("Dock in Date & Time","min"), receipt=("GR Date & Time","min"),
    ASN_count=("WMS ASN#","nunique"), ASNs=("WMS ASN#",join_ids)).rename_axis("HU / carton ID").reset_index()
timing_out = kpi_out.loc[trace_outbound & kpi_out["Carton ID"].notna()].groupby("Carton ID").agg(
    created=(created_col,"min"), handover=(released_col,"max"),
    Order_count=("Order Key","nunique"), Orders=("Order Key",join_ids)).rename_axis("HU / carton ID").reset_index()
timing_links = timing_in.merge(timing_out,on="HU / carton ID",how="inner",validate="one_to_one")
TIMING_STAGES = {
    "Dock to goods receipt": ("dock","receipt"),
    "Goods receipt to order creation": ("receipt","created"),
    "Order creation to handover": ("created","handover"),
    "Dock to handover": ("dock","handover"),
}

def process_timing(at):
    t=pd.Timestamp(at)
    # Both sides must have become observable; no future order association is revealed.
    population=timing_links.loc[timing_links.receipt.le(t)&timing_links.created.le(t)].copy()
    summaries=[]; records={}
    for stage,(start,end) in TIMING_STAGES.items():
        table=population.copy()
        table["Status"]="Included"
        table.loc[table[start].isna()|table[start].gt(t),"Status"]="Start event not observed by selected time"
        table.loc[table[end].isna()|table[end].gt(t),"Status"]="End event not observed by selected time"
        negative=(table[end]-table[start]).dt.total_seconds().lt(0)
        table.loc[negative,"Status"]="Negative timestamp sequence"
        table.loc[table.ASN_count.ne(1)|table.Order_count.ne(1),"Status"]="Ambiguous ASN or order association"
        table["Duration (hours)"]=((table[end]-table[start]).dt.total_seconds()/3600).where(table.Status.eq("Included"))
        for column in ["dock","receipt","created","handover"]:
            table[column]=table[column].where(table[column].le(t))
        valid=table.loc[table.Status.eq("Included"),"Duration (hours)"]
        p90 = valid.quantile(0.9) if len(valid) else float("nan")
        table["Benchmark P90 (hours)"] = p90
        table["Current wait (hours)"] = float("nan")
        pending = table.Status.eq("End event not observed by selected time") & table[start].notna()
        table.loc[pending,"Current wait (hours)"] = (t-table.loc[pending,start]).dt.total_seconds()/3600
        compared = table["Duration (hours)"].fillna(table["Current wait (hours)"])
        table["Beyond P90"] = compared.notna() & pd.notna(p90) & compared.gt(p90)
        table["Hours beyond P90"] = (compared-p90).clip(lower=0) if pd.notna(p90) else float("nan")
        table["Flag"] = table.Status
        table.loc[compared.notna(),"Flag"] = "Within P90"
        if pd.isna(p90):table.loc[compared.notna(),"Flag"] = "No completed benchmark"
        table.loc[table["Beyond P90"] & ~pending,"Flag"] = "Completed > P90"
        table.loc[table["Beyond P90"] & pending,"Flag"] = "Still waiting > P90"
        summaries.append({"Stage":stage,"Included HUs":len(valid),"Excluded / not yet observed":len(table)-len(valid),
            "Median (hours)":valid.median() if len(valid) else float("nan"),
            "P90 (hours)":p90,"Completed > P90":int((table["Beyond P90"] & ~pending).sum()),"Still waiting > P90":int((table["Beyond P90"] & pending).sum())})
        records[stage]=table
    return pd.DataFrame(summaries).set_index("Stage"),records


In [9]:
# Reusable record inspector for Feasure06. Prepared copies retain source columns.
import ipywidgets as widgets
from IPython.display import display

def canonical_record_id(value):
    if pd.isna(value):return ""
    text=str(value).strip()
    if text.endswith(".0") and text[:-2].isdigit():text=text[:-2]
    return (text.lstrip("0") or "0") if text.isdigit() else text

record_ib=ib.copy();record_ob=ob.copy()
for source,target in [("WMS ASN#","_ASN"),("Ref Line#","_Line"),("LPN","_HU")]:
    record_ib[target]=record_ib[source].map(canonical_record_id)
for source,target in [("Order Key","_Order"),("Order Line Number","_Line"),("Carton ID","_HU")]:
    record_ob[target]=record_ob[source].map(canonical_record_id)
# REVIEW 16 Nov - ADDED: Functional Packing is part of the inbound line key (DSV A5); kept as _FP for line-level grouping.
record_ib["_FP"]=record_ib["Functional Packing"].astype("string").str.strip().fillna("")
for frame in [record_ib,record_ob]:
    frame["_SKU"]=frame.SKU.astype("string").str.strip().fillna("")
    frame["_Lot"]=frame["Vendor Lot"].astype("string").str.strip().fillna("")
record_days=sorted(set(recorded_times.dt.date)|{DATA_TODAY})

def key_parts(value):
    text=str(value).replace(" / ","|").replace("/","|")
    return [canonical_record_id(part) for part in text.split("|")]

def normal_receipt_index():
    source=record_ib.loc[record_ib._SKU.ne("")&record_ib._Lot.ne("")&record_ib._ASN.ne("")&record_ib["GR Date & Time"].notna(),
        ["_SKU","_Lot","_ASN","GR Date & Time"]].drop_duplicates()
    raw={}
    for sku,lot,asn,when in source.itertuples(index=False,name=None):
        raw.setdefault((sku,lot),{}).setdefault(pd.Timestamp(when),set()).add(asn)
    return {key:(pd.DatetimeIndex(sorted(times)),[tuple(sorted(times[when])) for when in sorted(times)]) for key,times in raw.items()}

normal_receipts=normal_receipt_index()

def infer_receipt_candidate(sku,lot,created):
    if pd.isna(created):return {"Tag":"Absent","Candidates":(),"Receipt time":pd.NaT,"Reason":"Order creation time is missing."}
    if pd.isna(sku) or pd.isna(lot) or not str(sku).strip() or not str(lot).strip():
        return {"Tag":"Absent","Candidates":(),"Receipt time":pd.NaT,"Reason":"SKU or vendor lot is missing."}
    key=(str(sku).strip(),str(lot).strip())
    if key not in normal_receipts:return {"Tag":"Absent","Candidates":(),"Receipt time":pd.NaT,"Reason":"No receipt for this SKU and lot exists in the extract."}
    times,asns=normal_receipts[key]
    position=times.searchsorted(pd.Timestamp(created),side="right")-1
    if position<0:return {"Tag":"Absent","Candidates":(),"Receipt time":pd.NaT,"Reason":"No receipt for this SKU and lot precedes order creation."}
    candidates=asns[position]
    return {"Tag":"Inferred candidate" if len(candidates)==1 else "Ambiguous inference","Candidates":candidates,
        "Receipt time":times[position],"Reason":"Latest prior receipt of the same SKU and vendor lot; physical stock source is unconfirmed." if len(candidates)==1 else "Several ASNs share the latest eligible receipt timestamp; all candidates retained."}

def order_inference_rows(outgoing):
    normal=outgoing.loc[outgoing["DN Type"].astype("string").str.strip().str.casefold().eq("normal")]
    records=[]
    for (order,line,sku,lot),rows in normal.groupby(["_Order","_Line","_SKU","_Lot"],dropna=False):
        created=rows[created_col].min();candidate=infer_receipt_candidate(sku,lot,created)
        records.append({"Order Key":order,"Order line [Order Line Number]":line,"SKU":sku,"Vendor Lot":lot,
            "Order created [Orderdate Date & Time]":created,"Link tag":candidate["Tag"],"Candidate ASNs":", ".join(candidate["Candidates"]),
            "Candidate receipt (+8h) [GR Date & Time]":candidate["Receipt time"],"Reason":candidate["Reason"]})
    return pd.DataFrame(records,columns=["Order Key","Order line [Order Line Number]","SKU","Vendor Lot","Order created [Orderdate Date & Time]",
        "Link tag","Candidate ASNs","Candidate receipt (+8h) [GR Date & Time]","Reason"])

def record_payload(kind,key,at=None,experimental=False):
    t=None if at is None else pd.Timestamp(at)
    parts=key_parts(key);root=parts[0] if parts else ""
    if not root:raise ValueError("Enter a record identifier.")
    line=parts[1] if len(parts)>1 else ""
    if kind in ["ASN line","Order line"] and not line:raise ValueError("A line needs its parent ID and reference line, for example parent / line.")
    incoming=record_ib.iloc[:0].copy();outgoing=record_ob.iloc[:0].copy()
    visible_in=record_ib if t is None else record_ib.loc[record_ib["GR Date & Time"].le(t)]
    visible_out=record_ob if t is None else record_ob.loc[record_ob[[created_col,scan_col,packed_col,released_col]].le(t).any(axis=1)]
    if kind in ["ASN","ASN line"]:
        incoming=record_ib.loc[record_ib._ASN.eq(root)].copy()
        if kind=="ASN line":incoming=incoming.loc[incoming._Line.eq(line)]
        if t is not None:incoming=incoming.loc[incoming["Dock in Date & Time"].le(t)|incoming["GR Date & Time"].le(t)]
        linked_rows=incoming if t is None else incoming.loc[incoming["GR Date & Time"].le(t)]
        units=set(linked_rows._HU)-{""}
        outgoing=visible_out.loc[visible_out._HU.isin(units)].copy()
    elif kind in ["Order","Order line"]:
        outgoing=visible_out.loc[visible_out._Order.eq(root)].copy()
        if kind=="Order line":outgoing=outgoing.loc[outgoing._Line.eq(line)]
        incoming=visible_in.loc[visible_in._HU.isin(set(outgoing._HU)-{""})].copy()
    else:
        incoming=visible_in.loc[visible_in._HU.eq(root)].copy()
        outgoing=visible_out.loc[visible_out._HU.eq(root)].copy()
    related=[]
    def add(target,id_,tag,reason):
        if not str(id_):return
        if target==kind and str(id_)=="|".join(parts):return
        related.append({"Target type":target,"Target ID":str(id_),"Link tag":tag,"Evidence / reason":reason})
    for asn in incoming._ASN.unique():add("ASN",asn,"Recorded membership","Recorded on the inbound source rows; cross-report association uses a shared carton/LPN ID.")
    for asn,ref in incoming[["_ASN","_Line"]].drop_duplicates().itertuples(index=False,name=None):
        add("ASN line",asn+"|"+ref,"Recorded membership","Reference line recorded under this ASN.")
    observed_incoming=incoming if t is None else incoming.loc[incoming["GR Date & Time"].le(t)]
    in_units=set(observed_incoming._HU)-{""};out_units=set(outgoing._HU)-{""}
    for unit in sorted(in_units|out_units):
        add("Handling unit / carton",unit,"Exact identifier" if unit in in_units&out_units else "Recorded membership",
            "Inbound LPN equals outbound Carton ID." if unit in in_units&out_units else "Identifier appears on the selected source record.")
    for order in outgoing._Order.unique():add("Order",order,"Exact identifier" if not incoming.empty else "Recorded membership",
        "Reached through an exact shared carton/LPN ID." if not incoming.empty else "Order recorded on the outbound source rows.")
    for order,ref in outgoing[["_Order","_Line"]].drop_duplicates().itertuples(index=False,name=None):
        add("Order line",order+"|"+ref,"Recorded membership","Order line recorded in the outbound report.")
    observed_orders=outgoing if t is None else outgoing.loc[outgoing[created_col].le(t)]
    inference=order_inference_rows(observed_orders) if experimental and kind in ["Order","Order line"] else pd.DataFrame()
    if not inference.empty:
        for _,row in inference.iterrows():
            for asn in str(row["Candidate ASNs"]).split(","):
                if asn.strip():add("ASN",asn.strip(),row["Link tag"],row["Reason"])
    relations=pd.DataFrame(related,columns=["Target type","Target ID","Link tag","Evidence / reason"]).drop_duplicates()
    exact_units=in_units&out_units
    if exact_units:link_tag="Exact identifier"
    elif not incoming.empty and not outgoing.empty:link_tag="Recorded membership"
    else:link_tag="Absent cross-report link"
    asn_count=incoming._ASN.nunique();order_count=outgoing._Order.nunique()
    ambiguity="Multiple ASN associations retained" if kind=="Handling unit / carton" and asn_count>1 else ""
    if kind=="Handling unit / carton" and order_count>1:ambiguity=(ambiguity+"; " if ambiguity else "")+"Multiple order associations retained"
    if incoming.empty and outgoing.empty:reason="No source record is observable for this identifier at the selected time; use Full reports to inspect the full extract."
    elif incoming.empty:reason="No inbound LPN is observed for these carton IDs. Normal-flow records usually lack a shared cross-report identifier; absence is not proof of missing stock."
    elif outgoing.empty:reason="No outbound carton/order is observed for these LPNs at the selected time; the cause is not established."
    else:reason="Source rows are connected only through recorded IDs. Exact identity does not allocate quantities or prove that all contents reconcile."
    events=[]
    for frame,fields,side in [(incoming,[("Dock in Date & Time","Dock-in"),("GR Date & Time","Goods receipt")],"Inbound (+8h)"),
        (outgoing,[(created_col,"Order creation"),(scan_col,"Receipt / Cyclone scan"),(packed_col,"Packing"),(released_col,"Handover")],"Outbound")]:
        for column,event in fields:
            for _,row in frame.loc[frame[column].notna()].iterrows():
                when=pd.Timestamp(row[column])
                if t is not None and when>t:continue
                entity=row["_ASN"] if side.startswith("Inbound") else (row["_Order"] if column==created_col else row["_HU"] or row["_Order"])
                events.append({"Aligned event time":when,"Event":event,"Source":side,"Record ID":entity,
                    "Order context":row.get("_Order", ""),"Source timestamp":column})
    timeline=pd.DataFrame(events,columns=["Aligned event time","Event","Source","Record ID","Order context","Source timestamp"]).drop_duplicates()
    if not timeline.empty:timeline=timeline.sort_values(["Aligned event time","Event"])
    facts=[("Record",kind+" "+str(key).replace("|"," / ")),("Link tag",link_tag),("Ambiguity",ambiguity or "None identified"),
        ("Incoming ASNs",asn_count),("Outgoing orders",order_count),("Shared exact HU IDs",len(exact_units))]
    if kind in ["ASN","ASN line"] and not incoming.empty:
        # REVIEW 16 Nov - CHANGED: expected is counted once per 3-part line; on the 2-part key the second packing's expected quantity was dropped.
        # [OLD] quantity=incoming.groupby(["_ASN","_Line"]).agg(expected=("Qty Exp","first"),variants=("Qty Exp","nunique"))
        quantity=incoming.groupby(["_ASN","_Line","_FP"]).agg(expected=("Qty Exp","first"),variants=("Qty Exp","nunique"))
        expected=quantity.expected.sum() if quantity.variants.eq(1).all() else float("nan")
        seen=incoming if t is None else incoming.loc[incoming["GR Date & Time"].le(t)]
        facts += [("Expected quantity on recorded ASN lines [Qty Exp]",expected),("Received quantity observed [Qty Rcvd]",seen["Qty Rcvd"].sum())]
        facts += [("Shipment types [Shipment Type]",join_ids(incoming["Shipment Type"])),
                  ("Shipment references [Shipment Ref]",join_ids(incoming["Shipment Ref"])),
                  ("Putaway zones [Putaway Zone]",join_ids(incoming["Putaway Zone"]))]
    if kind in ["Order","Order line"] and not outgoing.empty:
        handed=outgoing if t is None else outgoing.loc[outgoing[released_col].le(t)]
        # REVIEW 16 Nov - CHANGED: count Qty Shipped once per order line (same grain as 2.1 and 7.1).
        # Why: an order line split over 3 cartons repeats its quantity on 3 rows, so a row sum triples it.
        # [OLD] facts += [("Shipped quantity observed [Qty Shipped]",handed["Qty Shipped"].sum())]
        facts += [("Shipped quantity observed, once per order line [Qty Shipped]",handed.drop_duplicates(["_Order","_Line"])["Qty Shipped"].sum())]
        facts += [("Flows [DN Type]",join_ids(outgoing["DN Type"])),
                  ("Customer orders [External Order Key]",join_ids(outgoing["External Order Key"])),
                  ("Planned dates [Planned Post Date]",join_ids(outgoing["Planned Post Date"].dt.strftime("%Y-%m-%d"))),
                  ("Final forwarders [Final Forwarder]",join_ids(outgoing["Final Forwarder"])),
                  ("Recorded carton IDs",len(set(outgoing._HU)-{""}))]
    if kind=="Handling unit / carton":
        seen=incoming if t is None else incoming.loc[incoming["GR Date & Time"].le(t)]
        handed=outgoing if t is None else outgoing.loc[outgoing[released_col].le(t)]
        # REVIEW 16 Nov - NOT CHANGED, TO DO: Qty Shipped is an order-line total, so there is no true per-carton shipped quantity.
        # Suggest relabelling as "Order-line quantity on this carton's rows" or hiding it. Line kept as it was for now.
        facts += [("Reported received quantity for this LPN [Qty Rcvd]",seen["Qty Rcvd"].sum()),("Reported shipped quantity for this carton [Qty Shipped]",handed["Qty Shipped"].sum())]
    if not incoming.empty or not outgoing.empty:
        facts += [("SKUs [SKU]",join_ids(pd.concat([incoming.SKU,outgoing.SKU]))),
                  ("Vendor lots [Vendor Lot]",join_ids(pd.concat([incoming["Vendor Lot"],outgoing["Vendor Lot"]])))]
    return dict(incoming=incoming,outgoing=outgoing,relations=relations,timeline=timeline,facts=pd.DataFrame(facts,columns=["Fact","Value"]),
        reason=reason,inference=inference,tag=link_tag,ambiguity=ambiguity)

def source_records_for_display(frame,side,at):
    shown=frame[[column for column in frame.columns if not column.startswith("_")]].copy()
    if at is not None:
        t=pd.Timestamp(at)
        if side=="Inbound":
            observed=frame["GR Date & Time"].le(t)
            shown["Qty Rcvd"]=shown["Qty Rcvd"].where(observed)
            shown["LPN"]=shown.LPN.where(observed)
            time_columns=["Dock in Date & Time","GR Date & Time"]
        else:
            shown["Qty Shipped"]=shown["Qty Shipped"].where(frame[released_col].le(t))
            time_columns=[created_col,scan_col,packed_col,released_col]
        for column in time_columns:shown[column]=shown[column].where(shown[column].le(t))
    if side=="Inbound":shown=shown.rename(columns={"Dock in Date & Time":"Docked (+8h) [Dock in Date & Time]","GR Date & Time":"Received (+8h) [GR Date & Time]"})
    return shown

def inspector_record_tables(payload,at):
    incoming=payload["incoming"].copy();outgoing=payload["outgoing"].copy()
    tables={}
    if not incoming.empty:
        incoming["_received_visible"]=incoming["Qty Rcvd"] if at is None else incoming["Qty Rcvd"].where(incoming["GR Date & Time"].le(at))
        # REVIEW 16 Nov - CHANGED: one row per 3-part line, so split packings are not shown as one line received 2x.
        # [OLD] lines=incoming.groupby(["_ASN","_Line"],dropna=False).agg(
        lines=incoming.groupby(["_ASN","_Line","_FP"],dropna=False).agg(
            SKU=("SKU",join_ids),lot=("Vendor Lot",join_ids),expected=("Qty Exp","first"),variants=("Qty Exp","nunique"),
            received=("_received_visible",lambda values:values.sum(min_count=1)),receipt=("GR Date & Time","max"),
            carton=("_HU",join_ids)).reset_index()
        lines["expected"]=lines.expected.where(lines.variants.le(1))
        lines["Receipt status"]=lines.receipt.map(lambda t:"Receipt observed" if pd.notna(t) and (at is None or t<=at) else "No receipt observed")
        if at is not None:lines["receipt"]=lines.receipt.where(lines.receipt.le(at))
        # LPNs are not observable before receipt confirmation.
        if at is not None:
            # REVIEW 16 Nov - CHANGED: same 3-part key as the lines table.
            # [OLD] observed=incoming.loc[incoming["GR Date & Time"].le(at)].groupby(["_ASN","_Line"])._HU.agg(join_ids)
            observed=incoming.loc[incoming["GR Date & Time"].le(at)].groupby(["_ASN","_Line","_FP"])._HU.agg(join_ids)
            # REVIEW 16 Nov - CHANGED: look-up uses the 3-part key.
            # [OLD] lines["carton"]=[observed.get((r._ASN,r._Line),"") for _,r in lines.iterrows()]
            lines["carton"]=[observed.get((r._ASN,r._Line,r._FP),"") for _,r in lines.iterrows()]
        tables["ASN lines"]=lines.drop(columns="variants").rename(columns={"_ASN":"ASN [WMS ASN#]","_Line":"Line [Ref Line#]","_FP":"Packing [Functional Packing]","SKU":"SKUs [SKU]",
            "lot":"Lots [Vendor Lot]","expected":"Expected [Qty Exp]","received":"Observed received [Qty Rcvd]",
            "receipt":"Received (+8h) [GR Date & Time]","carton":"LPNs [LPN]"})
    if not outgoing.empty:
        outgoing["_shipped_visible"]=outgoing["Qty Shipped"] if at is None else outgoing["Qty Shipped"].where(outgoing[released_col].le(at))
        lines=outgoing.groupby(["_Order","_Line"],dropna=False).agg(SKU=("SKU",join_ids),lot=("Vendor Lot",join_ids),
            # REVIEW 16 Nov - CHANGED: shipped per order line = the repeated line value (max), not the sum over its carton rows.
            # [OLD] flow=("DN Type",join_ids),carton=("_HU",join_ids),shipped=("_shipped_visible",lambda values:values.sum(min_count=1)),handover=(released_col,"max")).reset_index()
            flow=("DN Type",join_ids),carton=("_HU",join_ids),shipped=("_shipped_visible","max"),handover=(released_col,"max")).reset_index()
        if at is not None:lines["handover"]=lines.handover.where(lines.handover.le(at))
        tables["Order lines"]=lines.rename(columns={"_Order":"Order [Order Key]","_Line":"Line [Order Line Number]","SKU":"SKUs [SKU]",
            "lot":"Lots [Vendor Lot]","flow":"Flow [DN Type]","carton":"Cartons [Carton ID]","shipped":"Observed shipped [Qty Shipped]",
            "handover":"Handover ["+released_col+"]"})
    links=payload["relations"].loc[~payload["relations"]["Target type"].isin(["ASN line","Order line"])].copy()
    if not links.empty:tables["Related ASNs, orders and cartons"]=links
    return tables

class RecordInspector:
    """Read-only summary, line tables and history; no repeated search/settings form."""
    def __init__(self,scope="As of selected time"):
        self.current=None;self.payload=None;self.busy=False;self.line_tables={}
        self.title=widgets.HTML();self.summary=widgets.Output();self.related_output=widgets.Output();self.timeline_output=widgets.Output()
        self.related_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Related table page:",style={"description_width":"initial"})
        self.timeline_page=widgets.BoundedIntText(value=1,min=1,max=1,description="History page:",style={"description_width":"initial"})
        self.widget=widgets.VBox([self.title,self.summary,widgets.HTML("<b>Related records and lines</b>"),self.related_page,self.related_output,
            widgets.HTML("<b>Recorded event history</b>"),self.timeline_page,self.timeline_output])
        for control in [self.related_page,self.timeline_page,preview_input]:control.observe(self.refresh,names="value")
        self.clear()
    def clear(self):
        self.current=None;self.payload=None;self.line_tables={};self.widget.layout.display="none"
        for output in [self.summary,self.related_output,self.timeline_output]:output.outputs=()
        self.title.value=""
    def open(self,kind,key,at=None,full=False,origin="",push=True):
        self.busy=True
        self.current=(kind,str(key),None if full else pd.Timestamp(AS_OF if at is None else at))
        self.related_page.value=1;self.timeline_page.value=1;self.busy=False
        self.widget.layout.display="flex";self.refresh()
    def refresh(self,change=None):
        if self.busy or self.current is None:return
        self.busy=True
        try:
            from html import escape
            kind,key,at=self.current
            self.payload=record_payload(kind,key,at,experimental=False)
            self.line_tables=inspector_record_tables(self.payload,at)
            scope="Full report history" if at is None else "History up to "+str(at)
            self.title.value="<b>"+escape(kind+" "+key.replace("|"," / "))+"</b> · "+escape(scope)
            self.related_page.max=max(1,max(((len(t)+preview_input.value-1)//preview_input.value for t in self.line_tables.values()),default=1))
            self.related_page.value=min(self.related_page.value,self.related_page.max)
            self.timeline_page.max=max(1,(len(self.payload["timeline"])+preview_input.value-1)//preview_input.value)
            self.timeline_page.value=min(self.timeline_page.value,self.timeline_page.max)
            self.related_page.layout.display="" if self.related_page.max>1 else "none"
            self.timeline_page.layout.display="" if self.timeline_page.max>1 else "none"
            def summary():
                wanted=["Link tag","Flows [DN Type]","Customer orders [External Order Key]","Planned dates [Planned Post Date]",
                    "Shipped quantity observed [Qty Shipped]","Recorded carton IDs","Shipment types [Shipment Type]",
                    "Expected quantity on recorded ASN lines [Qty Exp]","Received quantity observed [Qty Rcvd]",
                    "Reported received quantity for this LPN [Qty Rcvd]","Reported shipped quantity for this carton [Qty Shipped]",
                    "Incoming ASNs","Outgoing orders"]
                facts=self.payload["facts"].loc[lambda f:f.Fact.isin(wanted)].copy()
                incoming=self.payload["incoming"]
                if not incoming.empty:
                    observed=incoming if at is None else incoming.loc[incoming["GR Date & Time"].le(at)]
                    if observed["Qty Rcvd"].notna().sum()==0:
                        facts.loc[facts.Fact.isin(["Received quantity observed [Qty Rcvd]","Reported received quantity for this LPN [Qty Rcvd]"]),"Value"]="Not recorded / not yet observed"
                facts["Fact"]=facts.Fact.replace({"Link tag":"Cross-report ID link"})
                display(facts)
                print(self.payload["reason"])
                if self.payload["ambiguity"]:print("Association warning: "+self.payload["ambiguity"])
            def related():
                if not self.line_tables:print("No related lines or records observed in this scope.");return
                start=(self.related_page.value-1)*preview_input.value
                for label,table in self.line_tables.items():
                    print(f"{label}: {len(table):,} records")
                    if start>=len(table):print("No more rows in this table on this page.")
                    else:display(table.iloc[start:start+preview_input.value])
                print("Missing quantities mean not recorded or not yet observed, not confirmed zero. Related IDs can be searched in Section 5.")
            def timeline():
                start=(self.timeline_page.value-1)*preview_input.value
                table=self.payload["timeline"].iloc[start:start+preview_input.value].rename(columns={"Aligned event time":"Event time","Source timestamp":"Excel timestamp"})
                if table.empty:print("No recorded events in this scope.")
                else:display(table[["Event time","Event","Record ID","Order context","Excel timestamp"]])
                print("Inbound times include +8h. Future events are hidden in as-of views. ID links do not verify contents or quantity reconciliation.")
            replace_output(self.summary,summary);replace_output(self.related_output,related);replace_output(self.timeline_output,timeline)
        finally:self.busy=False

def evidence_options(picker,records):
    previous=picker.value
    picker.options=records
    values=[value for label,value in records]
    picker.value=previous if previous in values else (values[0] if values else None)
def enrich_link_registry(full,at):
    t=pd.Timestamp(at)
    incoming=record_ib.loc[record_ib["Shipment Type"].astype("string").str.contains("milkrun",case=False,na=False)&record_ib["GR Date & Time"].le(t)&record_ib._HU.ne("")]
    outgoing=record_ob.loc[record_ob["DN Type"].astype("string").str.strip().str.casefold().eq("milkrun from abc")&record_ob[created_col].le(t)&record_ob._HU.ne("")]
    metadata=[]
    for unit,rows in incoming.groupby("_HU"):
        asns=rows._ASN.nunique();times=rows.loc[rows["Dock in Date & Time"].le(t),"Dock in Date & Time"].dropna().unique()
        dock=pd.Timestamp(times[0]) if asns==1 and len(times)==1 else pd.NaT
        day="Multiple ASN associations" if asns>1 else (dock.strftime("%Y-%m-%d") if pd.notna(dock) else "Unknown dock time")
        metadata.append({"HU / carton ID":unit,"Dock (+8h) [Dock in Date & Time]":dock,"ASN associations":asns,"Inbound dock day":day})
    in_meta=pd.DataFrame(metadata,columns=["HU / carton ID","Dock (+8h) [Dock in Date & Time]","ASN associations","Inbound dock day"])
    metadata=[]
    for unit,rows in outgoing.groupby("_HU"):
        orders_=rows._Order.nunique();times=rows[created_col].dropna().unique()
        day="Multiple order associations" if orders_>1 else (pd.Timestamp(times[0]).strftime("%Y-%m-%d") if len(times)==1 else "Multiple creation timestamps")
        metadata.append({"HU / carton ID":unit,"Order associations":orders_,"Outbound order day":day})
    out_meta=pd.DataFrame(metadata,columns=["HU / carton ID","Order associations","Outbound order day"])
    result=full.merge(in_meta,on="HU / carton ID",how="left",validate="one_to_one").merge(out_meta,on="HU / carton ID",how="left",validate="one_to_one")
    result["ASN associations"]=result["ASN associations"].fillna(0).astype(int)
    result["Order associations"]=result["Order associations"].fillna(0).astype(int)
    result["Link tag"]=result.Cohort.map({"Exactly linked":"Exact identifier","Inbound only":"Absent at selected time","Outbound only":"Absent at selected time"})
    result.loc[result["ASN associations"].gt(1)&result.Cohort.eq("Exactly linked"),"Link tag"]="Exact ID / multiple ASN associations"
    result.loc[result["ASN associations"].gt(1)&result.Cohort.ne("Exactly linked"),"Link tag"]="Absent counterpart / multiple ASN associations"
    result["Absence / association reason"]="Recorded LPN equals Carton ID. Quantity reconciliation is a separate check."
    all_in=record_ib.loc[record_ib._HU.ne("")&record_ib["Shipment Type"].astype("string").str.contains("milkrun",case=False,na=False)]
    all_out=record_ob.loc[record_ob._HU.ne("")&record_ob["DN Type"].astype("string").str.strip().str.casefold().eq("milkrun from abc")]
    raw_receipt=all_in.groupby("_HU")["GR Date & Time"].min()
    raw_created=all_out.groupby("_HU")[created_col].min()
    raw_scan=record_ob.loc[record_ob["DN Type"].astype("string").str.strip().str.casefold().eq("milkrun from abc")].groupby("_HU")[scan_col].min()
    first_receipt=record_ib.loc[record_ib["Shipment Type"].astype("string").str.contains("milkrun",case=False,na=False),"GR Date & Time"].min()
    for index,row in result.loc[result.Cohort.ne("Exactly linked")].iterrows():
        unit=row["HU / carton ID"]
        if row.Cohort=="Outbound only":
            if unit in raw_receipt.index:
                receipt=raw_receipt.loc[unit]
                reason="Inbound record exists but receipt time is unavailable." if pd.isna(receipt) else "Inbound receipt is not yet observed at the selected time."
            else:
                scan=raw_scan.get(unit,pd.NaT)
                reason="Cyclone scan predates the inbound receipt coverage; earlier receipt is not available in this extract." if pd.notna(scan) and scan<=t and pd.notna(first_receipt) and scan<first_receipt else "No matching inbound LPN in the file; cause unknown."
        else:
            if unit in raw_created.index:
                created=raw_created.loc[unit]
                reason="Outbound record exists but order creation time is unavailable." if pd.isna(created) else "Outbound order is not yet observed at the selected time."
            else:reason="No outbound carton in the file; cause unknown."
        result.loc[index,"Absence / association reason"]=reason
    result["Receipt to observed handover (hours)"]=(result["Visible handover"]-result.receipt).dt.total_seconds()/3600
    result["Elapsed since receipt; no handover observed (hours)"]=((t-result.receipt).dt.total_seconds()/3600).where(result.receipt.notna()&result["Visible handover"].isna())
    _,timing=process_timing(t)
    flag_table=timing["Dock to handover"][["HU / carton ID","Flag","Benchmark P90 (hours)","Hours beyond P90"]]
    result=result.merge(flag_table,on="HU / carton ID",how="left",validate="one_to_one")
    result["Flag"]=result.Flag.fillna("No eligible exact-link stage benchmark")
    return result

def linkage_coverage_tables(registry):
    tables=[]
    for date_column,eligible in [("Inbound dock day",registry.receipt.notna()),("Outbound order day",registry.created.notna())]:
        source=registry.loc[eligible].copy()
        source["Coverage"]=source.Cohort.map(lambda value:"Exactly linked" if value=="Exactly linked" else "No counterpart observed")
        table=pd.crosstab(source[date_column],source.Coverage).reindex(columns=["Exactly linked","No counterpart observed"],fill_value=0)
        tables.append(table)
    return tuple(tables)

def draw_linkage_coverage(registry):
    incoming,outgoing=linkage_coverage_tables(registry)
    if incoming.empty and outgoing.empty:
        print("No milk run units observed by this time to chart.");return
    fig,axes=plt.subplots(2,1,figsize=(10,7))
    for ax,table,title in [(axes[0],incoming,"Received inbound HUs by dock day (+8h)"),(axes[1],outgoing,"Outbound cartons by order creation day")]:
        if table.empty:ax.text(.5,.5,"No observed units",ha="center",va="center");ax.set_axis_off();continue
        table.plot.bar(stacked=True,ax=ax,color=["#279b77","#b5bec5"],width=.8)
        ax.set_title(title);ax.set_ylabel("Distinct units");ax.set_xlabel("");ax.tick_params(axis="x",labelrotation=35,labelsize=8)
        ax.legend(fontsize=8);ax.grid(axis="y",alpha=.2)
    fig.suptitle("Linkage coverage at the selected time — an unmatched record is not a missing-stock alarm",fontsize=10)
    display_operational_figure(fig)


### 2.1 Outbound shipping activity and outstanding work

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Open orders, orders past their planned day and due today (OUT-01, OUT-03, OUT-06). The prototype counts **orders, lines and cartons**, not units.  
> 🟨 **Not in the prototype yet:** Shipped-quantity totals, grouped charts and the Milkrun share of quantity. The prototype deliberately shows no units total (Issue 1, option D).  
> 🟥 **Changed in this version:** Shipped totals, grouped totals and the trend now count Qty Shipped once per order line.  

**Purpose:** Review recorded shipped quantities for a period, then separately inspect open orders and dispatch work at an as-of time.

**Shipping period controls:** Set Ship from/to, DN Type, Default Forwarder, Loc and Urgent Flag. These filters change shipping totals, charts and records together. The period uses Cargo Release Handover to Forwarder Date & Time and includes both endpoint days. Choose Grouped totals, Shipping trend or Source shipping rows; use Ship page to browse results.

- Grouped charts/tables use DN Type, Default Forwarder or Loc. Charts may show only the top 15 categories; the table retains all groups.
- Shipping trend supports daily/hourly frequency. Handover time is the movement date. Order creation time instead groups the **orders shipped within the selected handover period** by their creation time; it is not a shipping-event date.
- Milk run share is the filtered shipped quantity on DN Type containing 'milkrun' divided by all filtered shipped quantity. With a milk-only filter it can be 100%; with no quantity the rate is undefined. All source outbound rows carry Urgent Flag Y, so its Y quantity repeats the total.

~~**Quantity caution:** Shipping totals sum Qty Shipped over source rows. A consistent order-line quantity can repeat across cartons, so these are reported-row totals, not a deduplicated physical shipment quantity.~~ **→ **Quantity rule:** shipping totals now count Qty Shipped once per Order Key + Order Line Number, because the same line quantity repeats on every carton row. This is assumption WA-16 (DSV to confirm, Q25).** ~~Use the explicit order-line grain in Section 7.1 for content reconciliation.~~ **→ 2.1 and 7.1 now use the same grain.** There is no supplied outbound expected-quantity measure or customer-delivery confirmation.

**Outstanding-work controls:** Choose Date, enter Time and press Apply date/time. Open means created by that time with no final handover observed. Overdue, due today and due later compare the planned calendar day with this selected day. Orders without a planned date have their own category.

| Display | Interpretation |
|---|---|
| Now / 24h earlier / Change | Full-population counts of open orders, due work, recorded readiness, waiting cartons/HUs and recent activity. Change is a net difference, not a count of completions. |
| Order evidence | Narrow with status, readiness, DN type, Final Forwarder or order/customer-order text. These filters do not redefine the summary counts. Order page and Preview rows control the displayed slice. |
| Order types / carton stages / release evidence | Recorded classifications and process states at the selected time. Stage precedence is dock, receipt/scan, creation, packing, then handover. Blank carton IDs cannot contribute identified-carton counts; future actual outcomes are hidden. |
| Dispatch records | An independent paged list of packed cartons awaiting handover, milk HUs without observed handover, or handovers/creations in the last 24h. Dispatch view and Records page affect this list, not the order filters. |

Readiness counts use recorded order/carton pairs: packing for non-milk and Cyclone scan, with linked receipt fallback, for milk. 'All recorded cartons ready' is limited to IDs present in the extract and is not a verified required-carton inventory. Missing outbound coverage can make a received milk HU appear to await handover. Recent windows exclude their start and include their end.

Tick Show hourly outbound queue trend for separate order, carton and HU line panels. Rising means recorded waiting work is building up; falling means it is clearing. Flat counts can still contain arrivals and completions. Use Section 5 for broader order inspection. Forwarder/flow assignments are static extract metadata, not reconstructed assignment history.


In [10]:
# Display labels include the source Excel column; internal names stay unchanged.
EXCEL_LABELS = {
    "Final_forwarder":"Final forwarder [Final Forwarder]",
    "flow": "flow [DN Type]",
    "created": "created [Orderdate Date & Time]",
    "planned": "planned [Planned Post Date]",
    "released": "released [Cargo Release Handover to Forwarder Date & Time]",
    "Visible release time": "Visible release time [Cargo Release Handover to Forwarder Date & Time]",
}

def excel_labels(frame):
    labelled = frame.rename(columns=EXCEL_LABELS)
    return labelled.rename_axis(index=EXCEL_LABELS.get(frame.index.name, frame.index.name))

import ipywidgets as widgets
from IPython.utils.capture import capture_output

# User inputs, without playback or animation.
# Only dates with observed events are offered; counts mark data availability.
outbound_daily = pd.concat([orders.created, orders.released]).dropna().dt.date.value_counts()
receipt_daily = ib["GR Date & Time"].dropna().dt.date.value_counts()
available_days = sorted(set(outbound_daily.index) | set(receipt_daily.index) |
                        set(asns.dock.dropna().dt.date) | {DATA_TODAY})
date_options = [(f"{data_date_label(day)} | outbound events: {outbound_daily.get(day, 0):,} | receipt rows: {receipt_daily.get(day, 0):,}", day)
                for day in available_days]
initial_day = pd.Timestamp(AS_OF).date()
if initial_day not in available_days:
    initial_day = available_days[0]
date_input = widgets.Dropdown(options=date_options, value=initial_day, description="Date:",
    layout=widgets.Layout(width="610px"))
time_input = widgets.Text(value=pd.Timestamp(AS_OF).strftime("%H:%M:%S"), description="Time:",
    placeholder="HH:MM or HH:MM:SS", layout=widgets.Layout(width="220px"))
apply_time = widgets.Button(description="Apply date/time")
status_input = widgets.Dropdown(options=["All open", "Overdue", "Due today", "Due later", "No planned date"],
    value=ORDER_FILTER, description="Orders:")
preview_input = widgets.BoundedIntText(value=PREVIEW_ROWS, min=1, max=100, description="Preview rows:")
time_message = widgets.HTML()
selected_time = pd.Timestamp(f"{date_input.value} {time_input.value}")
result_panels = {}

def replace_output(panel, render):
    with capture_output() as captured:
        render()
    payload = []
    if captured.stdout:
        payload.append(dict(output_type="stream", name="stdout", text=captured.stdout))
    if captured.stderr:
        payload.append(dict(output_type="stream", name="stderr", text=captured.stderr))
    payload.extend(dict(output_type="display_data", data=x.data, metadata=x.metadata)
                   for x in captured.outputs)
    panel.outputs = tuple(payload)

def show_panel(name, render):
    panel = widgets.Output()
    result_panels[name] = (panel, render)
    replace_output(panel, render)

def refresh_results(change=None):
    time_message.value = f"<b>Results as of {selected_time:%Y-%m-%d %H:%M:%S}</b>"
    for panel, render in result_panels.values():
        replace_output(panel, render)

def apply_selected_time(_):
    global selected_time
    try:
        from datetime import datetime
        chosen_time = parse_clock_time(time_input.value)
        candidate = pd.Timestamp.combine(date_input.value, chosen_time)
        if pd.isna(candidate) or candidate.tzinfo is not None:
            raise ValueError("Use a local timestamp without a timezone.")
    except (ValueError, TypeError):
        time_message.value = "Enter a valid time in HH:MM or HH:MM:SS format, for example 12:00. Results have not changed."
        return
    selected_time = candidate
    refresh_results()

apply_time.on_click(apply_selected_time)
date_input.observe(apply_selected_time, names="value")
status_input.observe(refresh_results, names="value")
preview_input.observe(refresh_results, names="value")

refresh_results()


def order_snapshot(at):
    t = pd.Timestamp(at)
    visible = orders.loc[orders.created <= t].copy()
    visible["State"] = "Open"
    visible.loc[visible.released <= t, "State"] = "Released"
    opened = visible.loc[visible.State == "Open"].copy()
    planned_day = opened.planned.dt.normalize()
    opened["Priority"] = "No planned date"
    opened.loc[planned_day < t.normalize(), "Priority"] = "Overdue"
    opened.loc[planned_day == t.normalize(), "Priority"] = "Due today"
    opened.loc[planned_day > t.normalize(), "Priority"] = "Due later"
    return visible, opened


# Additional outbound filters are shown with the existing date/time and results.
outbound_readiness_filter = widgets.Dropdown(options=["All readiness", "All recorded cartons ready", "Not all recorded cartons ready", "No carton IDs in extract"], description="Readiness:", layout=widgets.Layout(width="410px"))
open_flow_filter=filter_widget(kpi_out["DN Type"],"Order DN Type:")
open_forwarder_filter=filter_widget(kpi_out["Final Forwarder"],"Order final forwarder:","420px")
outbound_order_search = widgets.Text(description="Order contains:", continuous_update=False)
outbound_order_page = widgets.BoundedIntText(value=1, min=1, max=1, description="Order page:")
outbound_trend = widgets.Checkbox(value=False, description="Show hourly outbound queue trend (24h)", indent=False)
outbound_queue_view = widgets.Dropdown(options=["Packed cartons awaiting handover", "Milk HUs without observed handover", "Cartons handed over (24h)", "Orders handed over (24h)", "Orders created (24h)"], description="Dispatch view:", layout=widgets.Layout(width="440px"))
outbound_queue_page = widgets.BoundedIntText(value=1, min=1, max=1, description="Records page:")



def render_open_orders():
    global selected_open_order_evidence
    snapshot = kpi_snapshot(selected_time)
    previous = kpi_snapshot(selected_time - pd.Timedelta(hours=24))
    keys = ["Open orders", "Open orders past planned day", "Open orders due today",
            "Open orders with all recorded cartons ready", "Packed non-milk cartons awaiting handover",
            "Milk HUs received, no observed handover", "Orders released in last 24h", "Orders created in last 24h",
            "Cartons handed over in last 24h"]
    counts = pd.DataFrame({"Now": {key: snapshot["values"][key] for key in keys},
                           "24h earlier": {key: previous["values"][key] for key in keys}})
    counts["Change"] = counts.Now - counts["24h earlier"]
    print(f"OUTBOUND WORK AT {selected_time}. Counts cover the full recorded population; filters below narrow the order evidence.")
    display(counts)
    if outbound_trend.value:
        trend = []
        for moment in pd.date_range(selected_time - pd.Timedelta(hours=24), selected_time, periods=25):
            state = kpi_snapshot(moment)["values"]
            trend.append({"Time": moment, "Open orders": state["Open orders"],
                "Packed cartons awaiting handover": state["Packed non-milk cartons awaiting handover"],
                "Milk HUs without observed handover": state["Milk HUs received, no observed handover"]})
        trend_frame = pd.DataFrame(trend).set_index("Time")
        import matplotlib.pyplot as plt
        import matplotlib.dates as mdates
        fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
        labels = [("Open orders", "Orders", "#176b87"),
                  ("Packed cartons awaiting handover", "Cartons", "#db7c26"),
                  ("Milk HUs without observed handover", "HUs", "#279b77")]
        for ax, (column, unit, colour) in zip(axes, labels):
            ax.plot(trend_frame.index, trend_frame[column], color=colour, linewidth=2)
            ax.set_title(column, loc="left", fontsize=10)
            ax.set_ylabel(unit)
            ax.grid(alpha=0.25)
            ax.set_ylim(bottom=0)
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
        axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=4))
        if axes[-1].get_xticklabels():
            axes[-1].get_xticklabels()[-1].set_ha("right")
        axes[-1].set_xlabel("Historical time (hourly observations)")
        fig.suptitle(f"Outbound waiting work: 24 hours ending {selected_time:%d %b %Y %H:%M:%S}", fontsize=12)
        fig.text(0.08, 0.025, "Read this: a rising line means waiting work is building up; a falling line means it is clearing.\nA flat line means a stable count, not necessarily no activity. Milk HU counts can include missing handover coverage.", fontsize=9)
        fig.tight_layout(rect=(0, 0.10, 1, 0.94))
        display(fig)
        plt.close(fig)

    selected = snapshot["readiness"].merge(order_search_attributes,on="Order Key",how="left",validate="one_to_one")
    if open_flow_filter.value!=ALL_FILTER:
        allowed=set(kpi_out.loc[source_filter(kpi_out["DN Type"],open_flow_filter.value),"Order Key"])
        selected=selected.loc[selected["Order Key"].isin(allowed)]
    if open_forwarder_filter.value!=ALL_FILTER:
        allowed=set(kpi_out.loc[source_filter(kpi_out["Final Forwarder"],open_forwarder_filter.value),"Order Key"])
        selected=selected.loc[selected["Order Key"].isin(allowed)]
    if status_input.value != "All open":
        selected = selected.loc[selected.Priority.eq(status_input.value)]
    if outbound_readiness_filter.value != "All readiness":
        selected = selected.loc[selected.Readiness.eq(outbound_readiness_filter.value)]
    if outbound_order_search.value.strip():
        selected = selected.loc[(selected["Order Key"].astype("string").str.contains(outbound_order_search.value.strip(),case=False,regex=False,na=False)|selected.External_orders.astype("string").str.contains(outbound_order_search.value.strip(),case=False,regex=False,na=False))]
    selected_open_order_evidence=selected.copy()
    outbound_order_page.max = max(1, (len(selected) + preview_input.value - 1) // preview_input.value)
    outbound_order_page.value = min(outbound_order_page.value, outbound_order_page.max)
    first = (outbound_order_page.value - 1) * preview_input.value
    print(f"ORDER EVIDENCE: {len(selected):,} matching orders; page {outbound_order_page.value} of {outbound_order_page.max}")
    display(excel_labels(selected[["Order Key", "flow", "Final_forwarder", "created", "planned", "Priority", "Ready / recorded cartons", "Readiness"]].iloc[first:first + preview_input.value]))

show_panel("Open orders", render_open_orders)

def render_order_types():
    visible, _ = order_snapshot(selected_time)
    print(f"Orders created by {selected_time}")
    display(excel_labels(pd.crosstab(visible.flow.fillna("Unknown"), visible.State)))

show_panel("Order types", render_order_types)


def carton_snapshot(at):
    t = pd.Timestamp(at)
    stage = pd.Series("Not yet observed", index=cartons.index)
    stage.loc[cartons.dock <= t] = "Docked / inbound"
    stage.loc[(cartons.receipt <= t) | (cartons.scan <= t)] = "Received / scanned"
    stage.loc[cartons.created <= t] = "Order created"
    stage.loc[cartons.packed <= t] = "Packed"
    stage.loc[cartons.released <= t] = "Released"
    return stage.loc[stage != "Not yet observed"]


def render_carton_stages():
    stages = carton_snapshot(selected_time)
    print(f"Identified cartons observed by {selected_time}: {len(stages):,}")
    display(stages.value_counts().rename_axis("Stage").to_frame("Cartons"))

show_panel("Carton stages", render_carton_stages)


def asn_snapshot(at):
    t = pd.Timestamp(at)
    state = pd.Series("Not yet arrived (known from extract)", index=asns.index)
    state.loc[asns.dock.isna()] = "Unknown dock time"
    state.loc[asns.dock <= t] = "Docked, awaiting receipt"
    state.loc[asns.receipt <= t] = "Received"
    return state


def render_asn_states():
    print(f"ASN states at {selected_time}")
    display(asn_snapshot(selected_time).value_counts().rename_axis("State").to_frame("ASNs"))




def release_evidence(at):
    visible, _ = order_snapshot(at)
    happened = visible.State == "Released"
    visible["Visible release time"] = visible.released.where(happened)
    visible["Outcome"] = "Pending at selected time"
    visible.loc[happened & visible.planned.isna(), "Outcome"] = "Released; no planned date"
    comparable = happened & visible.planned.notna()
    late = visible.released.dt.normalize() > visible.planned.dt.normalize()
    visible.loc[comparable & late, "Outcome"] = "Released late"
    visible.loc[comparable & ~late, "Outcome"] = "Released on/before planned day"
    return visible[["Order Key", "created", "planned", "Visible release time", "Outcome"]]


def render_release_evidence():
    evidence = release_evidence(selected_time)
    print(f"Release evidence available at {selected_time}")
    display(evidence.Outcome.value_counts().rename_axis("Outcome").to_frame("Orders"))
    print(f"First {preview_input.value} evidence records")
    display(excel_labels(evidence.head(preview_input.value)))

show_panel("Release evidence", render_release_evidence)


# Keep the controls and all of their results in the same visible output.
# Outbound period activity belongs beside the outstanding-order analysis.
outbound_activity = ob.copy()
outbound_activity["Qty Shipped"] = pd.to_numeric(outbound_activity["Qty Shipped"], errors="raise")
if outbound_activity["Qty Shipped"].isna().any():
    raise ValueError("Missing shipped quantities need review before totalling.")
outbound_shipping_days = sorted(set(outbound_activity[released_col].dropna().dt.date) | {DATA_TODAY})
initial_ship_day = pd.Timestamp(AS_OF).date()
if initial_ship_day not in outbound_shipping_days:
    initial_ship_day = outbound_shipping_days[0]
outbound_from = widgets.Dropdown(options=[(data_date_label(day), day) for day in outbound_shipping_days], value=initial_ship_day, description="Ship from:")
outbound_to = widgets.Dropdown(options=[(data_date_label(day), day) for day in outbound_shipping_days], value=initial_ship_day, description="Ship to:")
dn_type_options = [("All DN Types", "__ALL_DN__")] + [(str(value), value) for value in sorted(outbound_activity["DN Type"].dropna().unique())]
if outbound_activity["DN Type"].isna().any():
    dn_type_options.append(("Missing DN Type", "__MISSING_DN__"))
outbound_default_forwarder=filter_widget(outbound_activity["Default Forwarder"],"Default forwarder:","400px")
outbound_location=filter_widget(outbound_activity["Loc"],"Location [Loc]:")
outbound_urgent=filter_widget(outbound_activity["Urgent Flag"],"Urgent Flag:")
outbound_dn_type = widgets.Dropdown(options=dn_type_options, value="__ALL_DN__", description="DN Type:",
    layout=widgets.Layout(width="400px"))
outbound_group_by = widgets.Dropdown(options=["DN Type", "Default Forwarder", "Loc"], description="Group by:")
outbound_chart_frequency = widgets.Dropdown(options=["Daily", "Hourly"], description="Frequency:")
outbound_time_basis = widgets.Dropdown(options=["Handover time", "Order creation time"], description="Time basis:", layout=widgets.Layout(width="350px"))
outbound_chart_show = widgets.Checkbox(value=True, description="Show shipping chart", indent=False)
outbound_shipping_view = widgets.Dropdown(options=["Grouped totals", "Shipping trend", "Source shipping rows"], description="Inspect:")
outbound_shipping_page = widgets.BoundedIntText(value=1, min=1, max=1, description="Ship page:")
outbound_shipping_message = widgets.HTML()
outbound_shipping_output = widgets.Output()
outbound_shipping_busy = False

def shipping_activity(start, end, dn_type=None, forwarder=ALL_FILTER, location=ALL_FILTER, urgent=ALL_FILTER):
    first = pd.Timestamp(start).normalize()
    last = pd.Timestamp(end).normalize() + pd.Timedelta(days=1)
    if first >= last:
        raise ValueError("Ship from must be on or before Ship to.")
    shipped = outbound_activity.loc[outbound_activity[released_col].ge(first) & outbound_activity[released_col].lt(last)].copy()
    if dn_type == "__MISSING_DN__":
        shipped = shipped.loc[shipped["DN Type"].isna()]
    elif dn_type is not None and dn_type != "__ALL_DN__":
        shipped = shipped.loc[shipped["DN Type"].eq(dn_type)]
    for column,value in [("Default Forwarder",forwarder),("Loc",location),("Urgent Flag",urgent)]:
        shipped=shipped.loc[source_filter(shipped[column],value)]
    # REVIEW 16 Nov - CHANGED: count Qty Shipped once per order line, not once per carton row.
    # Why: on the full extract, row sums give 1,038,100,865 vs 695,808,651 once per line (about 49% too high).
    # Matches WA-16 and the 99.1% Milkrun reconciliation; DSV still to confirm (Q25). Source rows stay unchanged.
    # [OLD] total = shipped["Qty Shipped"].sum()
    line_once = once_per_order_line(shipped)
    total = line_once["Qty Shipped"].sum()
    # REVIEW 16 Nov - CHANGED: same once-per-order-line grain for the Milkrun share.
    # [OLD] milk = shipped.loc[shipped["DN Type"].astype("string").str.contains("milkrun", case=False, na=False), "Qty Shipped"].sum()
    milk = line_once.loc[line_once["DN Type"].astype("string").str.contains("milkrun", case=False, na=False), "Qty Shipped"].sum()
    return dict(rows=shipped, quantity=total, milk_quantity=milk,
                milk_share=100 * milk / total if total else float("nan"))

def refresh_outbound_shipping(change=None):
    global outbound_shipping_summary, outbound_shipping_busy
    if outbound_shipping_busy:
        return
    outbound_shipping_busy = True
    if change is not None and change.get("owner") is not outbound_shipping_page:
        outbound_shipping_page.value = 1
    try:
        try:
            outbound_shipping_summary = shipping_activity(outbound_from.value,outbound_to.value,outbound_dn_type.value,outbound_default_forwarder.value,outbound_location.value,outbound_urgent.value)
        except ValueError as error:
            outbound_shipping_message.value = str(error)
            outbound_shipping_output.outputs = ()
            return
        outbound_shipping_message.value = ""
        result = outbound_shipping_summary
        if outbound_shipping_view.value == "Grouped totals":
            detail = shipping_group_totals(result["rows"], outbound_group_by.value)
        elif outbound_shipping_view.value == "Shipping trend":
            detail = shipping_time_totals(result["rows"], outbound_chart_frequency.value, outbound_time_basis.value)
        else:
            detail = result["rows"][["Order Key", "DN Type", "SKU", "Qty Shipped", released_col]]
        outbound_shipping_page.max = max(1, (len(detail) + preview_input.value - 1) // preview_input.value)
        outbound_shipping_page.value = min(outbound_shipping_page.value, outbound_shipping_page.max)
        def render():
            print(f"SHIPPED ACTIVITY: {outbound_from.value} to {outbound_to.value} — full calendar days")
            print(f"DN Type: {outbound_dn_type.label}; Default forwarder: {outbound_default_forwarder.label}; Location: {outbound_location.label}; Urgent: {outbound_urgent.label}. Totals and records share these filters.")
            if result["rows"].empty:
                print("No shipping records match the selected date range and DN Type.")
            display(pd.DataFrame({"Value": [f"{result['quantity']:,.0f}", f"{result['milk_quantity']:,.0f}",
                f"{result['milk_share']:.2f}%" if pd.notna(result['milk_share']) else "N/A (zero shipped quantity)"]},
                index=["Shipped quantity, once per order line [Qty Shipped]", "Milk run shipped quantity, once per order line [DN Type, Qty Shipped]", "Milk run share of filtered shipped quantity"]))
            if outbound_chart_show.value:
                if outbound_shipping_view.value == "Shipping trend":
                    if detail.empty:
                        print("No matching timestamps to chart.")
                    else:
                        fig, ax = plt.subplots(figsize=(10, 4))
                        ax.plot(detail.index, detail.iloc[:,0], color="#176b87", marker="o", markersize=3)
                        ax.set_title("Shipped quantity grouped by " + outbound_time_basis.value.lower())
                        ax.set_ylabel("Quantity [Qty Shipped]"); ax.grid(alpha=0.2)
                        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M" if outbound_chart_frequency.value=="Hourly" else "%d %b"))
                        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=7))
                        display_operational_figure(fig)
                    print("The date range always selects cargo handovers. Order creation time groups those shipped records by their earlier creation time; it is not dispatch timing.")
                else:
                    grouped_chart = shipping_group_totals(result["rows"], outbound_group_by.value)
                    plot_group_quantity(grouped_chart, "Shipped quantity [Qty Shipped]", "Shipped quantity by " + outbound_group_by.value, "Quantity [Qty Shipped]")
            print(f"{outbound_shipping_view.value}: {len(detail):,} rows; page {outbound_shipping_page.value} of {outbound_shipping_page.max}")
            first = (outbound_shipping_page.value - 1) * preview_input.value
            display(detail.iloc[first:first + preview_input.value])
        replace_output(outbound_shipping_output, render)
    finally:
        outbound_shipping_busy = False

for control in [outbound_from, outbound_to, outbound_dn_type, outbound_default_forwarder, outbound_location, outbound_urgent, outbound_shipping_view, outbound_shipping_page, outbound_group_by, outbound_chart_frequency, outbound_time_basis, outbound_chart_show, preview_input]:
    control.observe(refresh_outbound_shipping, names="value")
refresh_outbound_shipping()

def render_outbound_queue():
    snapshot = kpi_snapshot(selected_time)
    name = outbound_queue_view.value
    if name == "Packed cartons awaiting handover":
        detail = excel_labels(snapshot["packed"][["Carton ID", "created", "packed"]]).rename(columns={"packed": "Packed [Packing Confirmation Timestamp]"})
    elif name == "Milk HUs without observed handover":
        detail = snapshot["waiting_milk"][["HU / carton ID", "receipt", "ASNs", "Orders", "Cohort"]].rename(columns={
            "HU / carton ID": "HU / carton ID [LPN, Carton ID]", "receipt": "Received [GR Date & Time]", "ASNs": "ASNs [WMS ASN#]", "Orders": "Orders [Order Key]"})
    elif name == "Cartons handed over (24h)":
        detail = excel_labels(snapshot["handovers"][["Carton ID", "created", "released"]])
    else:
        detail = snapshot["released24" if name == "Orders handed over (24h)" else "created24"].copy()
        detail["Visible release time"] = detail.released.where(detail.released <= selected_time)
        detail = excel_labels(detail[["Order Key", "flow", "created", "planned", "Visible release time"]])
    outbound_queue_page.max = max(1, (len(detail) + preview_input.value - 1) // preview_input.value)
    outbound_queue_page.value = min(outbound_queue_page.value, outbound_queue_page.max)
    first = (outbound_queue_page.value - 1) * preview_input.value
    explanations = {
        "Packed cartons awaiting handover": "Cartons packed by this time, with no handover yet recorded. Use this list to check ready-to-dispatch cartons.",
        "Milk HUs without observed handover": "Received milk handling units with no handover observed by this time. Missing outbound coverage can also cause this; it is not proof that every unit is physically waiting.",
        "Cartons handed over (24h)": "Completed carton handovers during the preceding 24 hours. This is dispatch history, not waiting work.",
        "Orders handed over (24h)": "Orders completed by recorded cargo handover during the preceding 24 hours. This is dispatch history, not waiting work.",
        "Orders created (24h)": "Orders created during the preceding 24 hours. They may be open or already handed over; future handover timestamps are hidden.",
    }
    outbound_queue_page.disabled = outbound_queue_page.max == 1
    print(f"DISPATCH RECORDS: {name}")
    print(explanations[name])
    print(f"As of {selected_time}. The Orders/Readiness filters above do not filter this separate record list.")
    if detail.empty:
        print("No records for this dispatch view at the selected time. There is no next page.")
    else:
        last = min(first + preview_input.value, len(detail))
        print(f"Showing records {first+1}–{last} of {len(detail):,}; page {outbound_queue_page.value} of {outbound_queue_page.max}. Change Records page to inspect a different set of records below.")
    display(detail.iloc[first:first + preview_input.value])

show_panel("Handover queues and activity", render_outbound_queue)
for control in [open_flow_filter,open_forwarder_filter,outbound_readiness_filter, outbound_order_search, outbound_order_page, outbound_trend]:
    control.observe(refresh_results, names="value")

dispatch_refresh_busy = False
def refresh_dispatch_records(change=None):
    global dispatch_refresh_busy
    if dispatch_refresh_busy:
        return
    dispatch_refresh_busy = True
    try:
        if change is not None and change.get("owner") is outbound_queue_view:
            outbound_queue_page.value = 1
        replace_output(result_panels["Handover queues and activity"][0], render_outbound_queue)
    finally:
        dispatch_refresh_busy = False

outbound_queue_view.observe(refresh_dispatch_records, names="value")
outbound_queue_page.observe(refresh_dispatch_records, names="value")

sections = [
    widgets.HTML("<h3>Outbound shipped activity — date range</h3>"),
    widgets.HBox([outbound_from, outbound_to]),
    outbound_dn_type,widgets.HBox([outbound_default_forwarder,outbound_location]),outbound_urgent, widgets.HBox([outbound_shipping_view, outbound_shipping_page]),
    outbound_group_by, widgets.HBox([outbound_chart_frequency, outbound_time_basis]), outbound_chart_show,
    outbound_shipping_message, outbound_shipping_output,
    widgets.HTML("<h3>Outstanding work — at the selected date/time</h3>"),
    date_input, widgets.HBox([time_input, apply_time]),
    widgets.HBox([status_input, preview_input]),widgets.HBox([open_flow_filter,open_forwarder_filter]),outbound_readiness_filter,
    widgets.HBox([outbound_order_search, outbound_order_page]), outbound_trend,
    time_message,
]
for name, (panel, _) in result_panels.items():
    if name == "Handover queues and activity":
        sections.extend([widgets.HTML("<h3>Inspect dispatch records</h3><p>Choose the record list, then use Records page to browse it. The selected records are displayed directly below these controls.</p>"),
                         widgets.HBox([outbound_queue_view, outbound_queue_page]), panel])
    else:
        sections.extend([widgets.HTML(f"<h4>{name}</h4>"), panel])
display(widgets.VBox(sections))


### 2.2 Inbound receiving quantities and exception investigation

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Inbound screen and rule IN-03 "Received quantity differs" (0 of 36,533 lines flagged this week).  
> 🟥 **Changed in this version:** Lines are grouped on WMS ASN# + Ref Line# + Functional Packing.  

**Purpose:** Compare expected and received quantities for recorded receipts, identify quantity differences and inspect the ASN lines behind them.

**Use:** Select GR from/to, Shipment type and Putaway zone. Dates use adjusted goods-receipt time and include whole endpoint days. The summary covers all matches. Chart and preview settings change presentation, while the line-status/ASN/SKU filters narrow the exception evidence only. Preview rows controls page or preview size; labels stating 'first N' identify truncated summaries.

**Quantity rules:** ~~Group by WMS ASN# + Ref Line#.~~ **→ Group by WMS ASN# + Ref Line# + Functional Packing (DSV A5).** Count Qty Exp **once per line** because it repeats across split rows; sum Qty Rcvd across those rows. Variance = received − expected. Completion = received / expected × 100; over-receipts may exceed 100%, and a zero expected quantity gives an undefined completion rate. Missing IDs/quantities or conflicting expected values prevent reliable receiving totals and trigger a message rather than a fabricated total.

| Line category | Test |
|---|---|
| Exact match | Received = expected. |
| Shortage | Received < expected. |
| Exactly double | Expected > 0 and received = 2 × expected. |
| Other over-receipt | Received > expected without the exactly-double pattern. The Over-receipt filter includes double cases too. |

The tables include overall counts/quantities, per-ASN totals, shipment-type reconciliation and ASNs ranked by exactly-double line count then excess. Shortage and excess are summed separately so they do not cancel; signed net variance can conceal both. The type table shows where the differences sit, and the ranked ASN table helps choose what to inspect.

Choose Abnormal, a specific status or All lines; narrow by ASN/SKU and page through the line register. Inspect line displays its underlying receipt rows. Open selected ASN-line details shows related lines and event history through the end of the selected receipt period. Do not sum repeated Qty Exp values in the raw rows.

The optional charts compare received quantity by shipment type or expected/received by zone. A line across multiple zones is labelled Multiple zones (unallocated), avoiding allocation of the same expectation to each zone.

**Scope limits:** Expected quantity belongs to lines with receipts **in this selected period**; it is not all quantity scheduled to arrive or still inbound. A line receiving across several days can look different in a short period than over its complete history. Zero shortages do not establish that every scheduled shipment arrived. Exactly double suggests a review pattern, not a proven duplicate posting, and no rows are corrected. ASN counts grouped by type should not be added if an ASN carries several types.


In [11]:
receiving_record_open=widgets.Button(description="Open selected ASN-line details",layout=widgets.Layout(width="250px"))
receiving_record_inspector=RecordInspector()
def open_receiving_record(_):
    if receipt_line.value is None or receiving_inspection_lines.empty:return
    row=receiving_inspection_lines.loc[receipt_line.value]
    cutoff=pd.Timestamp(receipt_end_input.value)+pd.Timedelta(days=1)-pd.Timedelta(seconds=1)
    receiving_record_inspector.open("ASN line",str(row["WMS ASN#"])+"|"+str(row["Ref Line#"]),cutoff,origin="Opened from receiving line evidence; timeline is limited to the end of the selected receipt period.")
receiving_record_open.on_click(open_receiving_record)

inbound_shipment_filter=filter_widget(ib["Shipment Type"],"Shipment type:","420px")
inbound_zone_options = [("All zones", "__ALL_ZONES__")] + [(str(zone), zone) for zone in sorted(inbound_zone_map.dropna().unique())]
inbound_zone_filter = widgets.Dropdown(options=inbound_zone_options, value="__ALL_ZONES__", description="Putaway zone:", layout=widgets.Layout(width="450px"))
inbound_chart_view = widgets.Dropdown(options=["Received by shipment type", "Expected vs received by zone"], description="Chart:", layout=widgets.Layout(width="530px"))
inbound_chart_show = widgets.Checkbox(value=True, description="Show receiving chart", indent=False)
receipt_days = sorted(set(receipt_daily.index) | {DATA_TODAY})
receipt_options = [(f"{data_date_label(day)} | {receipt_daily.get(day, 0):,} receipt rows", day) for day in receipt_days]
start_day = pd.Timestamp(RECEIPT_START).date()
end_day = pd.Timestamp(RECEIPT_END).date()
receipt_start_input = widgets.Dropdown(options=receipt_options,
    value=start_day if start_day in receipt_days else receipt_days[0], description="GR from:",
    layout=widgets.Layout(width="440px"))
receipt_end_input = widgets.Dropdown(options=receipt_options,
    value=end_day if end_day in receipt_days else receipt_days[-1], description="GR to:",
    layout=widgets.Layout(width="440px"))
receiving_refresh_busy = False
receipt_message = widgets.HTML()
receipt_panel = widgets.Output()

# Investigate exceptions at ASN-line level, after summing split receipts.
receipt_status = widgets.Dropdown(options=["Abnormal (short or over)", "Shortage", "Over-receipt", "Exactly double", "Exact match", "All lines"],
    value="Abnormal (short or over)", description="Line status:", layout=widgets.Layout(width="360px"))
receipt_asn = widgets.Dropdown(options=[("All ASNs", "__ALL__")], description="ASN:", layout=widgets.Layout(width="320px"))
receipt_sku = widgets.Text(description="SKU contains:", continuous_update=False)
receipt_page = widgets.BoundedIntText(value=1, min=1, max=1, description="Page:")
receipt_line = widgets.Dropdown(options=[], description="Inspect line:", layout=widgets.Layout(width="900px"))
line_panel = widgets.Output()
line_detail_panel = widgets.Output()
inspection_busy = False
receiving_inspection_lines = pd.DataFrame()



def receipt_records():
    if receipt_start_input.value is None or receipt_end_input.value is None:
        raise ValueError("Select both receipt dates.")
    start = pd.Timestamp(receipt_start_input.value)
    end = pd.Timestamp(receipt_end_input.value)
    if start > end:
        raise ValueError("Receipt start must be on or before receipt end.")
    selected = ib.loc[(ib["GR Date & Time"] >= start) & (ib["GR Date & Time"] < end + pd.Timedelta(days=1))].copy()
    selected=selected.loc[source_filter(selected["Shipment Type"],inbound_shipment_filter.value)]
    selected = selected.assign(_SourceRow=selected.index).merge(inbound_zone_map.reset_index(),
        # REVIEW 16 Nov - CHANGED: zone map is keyed on the 3-part line key (DSV A5).
        # [OLD] on=["WMS ASN#", "Ref Line#"], how="left", validate="many_to_one").set_index("_SourceRow").rename_axis(None)
        on=["WMS ASN#", "Ref Line#", "Functional Packing"], how="left", validate="many_to_one").set_index("_SourceRow").rename_axis(None)
    if inbound_zone_filter.value != "__ALL_ZONES__":
        selected = selected.loc[selected._ZoneGroup.eq(inbound_zone_filter.value)]
    return selected

def receiving_totals(selected):
    # Expected quantity is repeated on split-receipt rows: count it once per ASN line.
    work = selected.copy()
    for column in ["Qty Exp", "Qty Rcvd"]:
        work[column] = pd.to_numeric(work[column], errors="raise")
    # REVIEW 16 Nov - CHANGED: an inbound line is WMS ASN# + Ref Line# + Functional Packing (DSV A5).
    # Why: the 2-part key merges two pack types into one line and shows 109 false "exactly double" lines
    # (84 by 20 Aug 18:00). With the 3-part key all 36,533 lines reconcile. Same key as the prototype (WA-02).
    # [OLD] keys = ["WMS ASN#", "Ref Line#"]
    keys = ["WMS ASN#", "Ref Line#", "Functional Packing"]
    if work[keys].isna().any().any():
        raise ValueError("Missing ASN/line identifiers prevent reliable quantity aggregation.")
    if work[["Qty Exp", "Qty Rcvd"]].isna().any().any():
        raise ValueError("Missing quantities prevent reliable receiving totals.")
    grouped = work.groupby(keys, dropna=False)
    if not grouped["Qty Exp"].nunique().eq(1).all():
        raise ValueError("Conflicting expected quantities exist within an ASN line; review before totalling.")
    lines = grouped.agg(
        expected=("Qty Exp", "first"), received=("Qty Rcvd", "sum"),
        shipment_type=("Shipment Type", "first"), sku=("SKU", "first"),
        receipt_time=("GR Date & Time", "min"),
    ).reset_index()
    lines["variance"] = lines.received - lines.expected
    lines["Line status"] = "Exact match"
    lines.loc[lines.variance.lt(0), "Line status"] = "Shortage"
    lines.loc[lines.variance.gt(0), "Line status"] = "Over-receipt"
    lines.loc[lines.expected.gt(0)&lines.received.eq(2*lines.expected), "Line status"]="Exactly double"
    lines["Completion (%)"] = (100 * lines.received / lines.expected.where(lines.expected.ne(0))).round(2)
    lines["receipt_day"] = lines.receipt_time.dt.date
    lines["short"] = (lines.expected - lines.received).clip(lower=0)
    lines["over"] = (lines.received - lines.expected).clip(lower=0)

    def aggregate(column):
        result = lines.groupby(column, dropna=False).agg(
            ASNs=("WMS ASN#", "nunique"), Lines=("Ref Line#", "size"),
            expected=("expected", "sum"), received=("received", "sum"),
            short=("short", "sum"), over=("over", "sum"),
        )
        result["Completion (%)"] = (100 * result.received / result.expected.where(result.expected.ne(0))).round(2)
        return result.rename(columns={
            "expected": "Expected quantity [Qty Exp]", "received": "Received quantity [Qty Rcvd]",
            "short": "Short quantity", "over": "Over quantity",
        })

    by_asn = aggregate("WMS ASN#").drop(columns="ASNs")
    by_day = aggregate("receipt_day").rename_axis("Receipt date [GR Date & Time]")
    by_type = aggregate("shipment_type").rename_axis("Shipment type [Shipment Type]")
    expected, received = lines.expected.sum(), lines.received.sum()
    rate = 100 * received / expected if expected else float("nan")
    totals = pd.DataFrame({"Value": [
        f"{len(selected):,}", f"{lines['WMS ASN#'].nunique():,}", f"{len(lines):,}",
        f"{expected:,.0f}", f"{received:,.0f}",
        f"{rate:.2f}%" if pd.notna(rate) else "N/A (zero expected quantity)",
        f"{lines['short'].sum():,.0f}", f"{lines['over'].sum():,.0f}",
        f"{int(lines.received.eq(lines.expected).sum()):,}",
    ]}, index=["Source rows", "ASNs", "Distinct ASN lines", "Expected quantity [Qty Exp]",
               "Received quantity [Qty Rcvd]", "Completion: received / expected", "Short quantity (per line)",
               "Over quantity (per line)", "Lines received exactly as expected"])
    totals.loc["Abnormal ASN lines", "Value"]=int(lines.variance.ne(0).sum())
    totals.loc["Over-received ASN lines (including exactly double)", "Value"]=int(lines.variance.gt(0).sum())
    totals.loc["Exactly-double ASN lines", "Value"]=int(lines["Line status"].eq("Exactly double").sum())
    totals.loc["Net receipt variance (received - expected)", "Value"]=received-expected
    for kind,label in [("Exactly double","Exactly-double lines"),("Shortage","Short lines"),("Over-receipt","Other over lines")]:
        by_type[label]=lines.assign(_flag=lines["Line status"].eq(kind)).groupby("shipment_type")._flag.sum().reindex(by_type.index,fill_value=0)
    by_type["Abnormal lines"]=lines.assign(_flag=lines.variance.ne(0)).groupby("shipment_type")._flag.sum().reindex(by_type.index,fill_value=0)
    double_asns=lines.loc[lines["Line status"].eq("Exactly double")].groupby("WMS ASN#").agg(
        Double_lines=("Ref Line#","size"),Excess_quantity=("over","sum"),Received=("received","sum"),Expected=("expected","sum")).sort_values(["Double_lines","Excess_quantity"],ascending=False)
    double_asns=double_asns.rename(columns={"Double_lines":"Exactly-double lines","Excess_quantity":"Excess [Qty Rcvd - Qty Exp]","Received":"Received [Qty Rcvd]","Expected":"Expected once [Qty Exp]"})
    return dict(lines=lines, totals=totals, double_asns=double_asns, by_day=by_day, by_type=by_type, by_asn=by_asn,
                expected=expected, received=received, rate=rate)

def render_receipts():
    global receiving_summary, receiving_chart_table
    selected = receipt_records()
    receiving_summary = receiving_totals(selected)
    print(f"RECEIVING: {receipt_start_input.value} to {receipt_end_input.value}; zone: {inbound_zone_filter.label}")
    display(receiving_summary["totals"])
    if inbound_chart_view.value == "Received by shipment type":
        receiving_chart_table = receiving_summary["by_type"]
        if inbound_chart_show.value:
            plot_group_quantity(receiving_chart_table, "Received quantity [Qty Rcvd]", "Received quantity by shipment type", "Quantity [Qty Rcvd]")
    else:
        receiving_chart_table = inbound_zone_totals(selected, receiving_summary)
        if inbound_chart_show.value and not receiving_chart_table.empty:
            shown = receiving_chart_table
            fig, ax = plt.subplots(figsize=(10,4.5))
            shown.plot.bar(ax=ax, color=["#8eb9c9", "#176b87"], width=0.8)
            ax.set_title("Expected versus received quantity by putaway zone")
            ax.set_ylabel("Reported quantity"); ax.set_xlabel(""); ax.grid(axis="y",alpha=0.2)
            ax.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
            ax.tick_params(axis="x", labelrotation=45, labelsize=8)
            ax.legend(["Expected [Qty Exp]", "Received [Qty Rcvd]"])
            display_operational_figure(fig)
    print(f"{inbound_chart_view.value}: first {preview_input.value} of {len(receiving_chart_table):,} groups")
    display(receiving_chart_table.head(preview_input.value))
    asn_table = receiving_summary["by_asn"].sort_values("Received quantity [Qty Rcvd]",ascending=False)
    print(f"PER-ASN TOTALS: first {preview_input.value} of {len(asn_table):,} ASNs")
    display(asn_table.head(preview_input.value))

    print("SHIPMENT-TYPE RECONCILIATION: same receipt dates/zone/type filters; expected counted once per ASN line.")
    display(receiving_summary["by_type"].head(preview_input.value))
    print("ASNs WITH MOST EXACTLY-DOUBLE LINES — ranked by line count then excess quantity; first "+str(preview_input.value)+" ASNs")
    if receiving_summary["double_asns"].empty:print("No exactly-double lines in this selection.")
    else:display(receiving_summary["double_asns"].head(preview_input.value))
    print("Exactly double is a possible posting pattern, not proof of duplicate receipt. Net variance can offset shortages and excess; use the separate totals.")

def filter_receipt_lines(lines):
    selected = lines.copy()
    if receipt_status.value == "Abnormal (short or over)":
        selected = selected.loc[selected.variance.ne(0)]
    elif receipt_status.value == "Over-receipt":
        selected=selected.loc[selected.variance.gt(0)]
    elif receipt_status.value != "All lines":
        selected = selected.loc[selected["Line status"].eq(receipt_status.value)]
    if receipt_sku.value.strip():
        selected = selected.loc[selected.sku.astype("string").str.contains(receipt_sku.value.strip(), case=False, regex=False, na=False)]
    return selected

def show_line_records(change=None):
    if inspection_busy:
        return
    def render():
        if receipt_line.value is None or receiving_inspection_lines.empty:
            print("No line selected. There are no matching lines for these filters.")
            return
        line = receiving_inspection_lines.loc[receipt_line.value]
        records = receipt_records()
        # REVIEW 16 Nov - CHANGED: also match Functional Packing, so the inspector shows only this 3-part line's rows.
        # [OLD] records = records.loc[records["WMS ASN#"].eq(line["WMS ASN#"]) & records["Ref Line#"].eq(line["Ref Line#"])]
        records = records.loc[records["WMS ASN#"].eq(line["WMS ASN#"]) & records["Ref Line#"].eq(line["Ref Line#"]) & records["Functional Packing"].eq(line["Functional Packing"])]
        print(f"ASN {line['WMS ASN#']} / line {line['Ref Line#']} / SKU {line['sku']}: {line['Line status']}")
        print(f"Expected once: {line.expected:,.0f}; received across rows: {line.received:,.0f}; variance: {line.variance:+,.0f}")
        print(f"Underlying receipt records: first {preview_input.value} of {len(records):,}. Expected quantity repeats across split rows; do not sum it here.")
        columns = ["WMS ASN#", "Ref Line#", "SKU", "Vendor Lot", "Shipment Type", "Shipment Ref", "Qty Exp", "Qty Rcvd", "LPN", "GR Date & Time"]
        display(records[columns].head(preview_input.value))
    replace_output(line_detail_panel, render)

def refresh_inspection(change=None):
    global inspection_busy, receiving_inspection_lines
    if inspection_busy:
        return
    inspection_busy = True
    try:
        lines = filter_receipt_lines(receiving_summary["lines"])
        asn_values = sorted(lines["WMS ASN#"].dropna().unique())
        previous_asn = receipt_asn.value
        receipt_asn.options = [("All ASNs", "__ALL__")] + [(str(value), value) for value in asn_values]
        receipt_asn.value = previous_asn if previous_asn in asn_values else "__ALL__"
        if receipt_asn.value != "__ALL__":
            lines = lines.loc[lines["WMS ASN#"].eq(receipt_asn.value)]
        lines = lines.assign(_severity=lines.variance.abs()).sort_values(
            ["_severity", "WMS ASN#", "Ref Line#"], ascending=[False, True, True]).reset_index(drop=True)
        receiving_inspection_lines = lines
        rows_per_page = preview_input.value
        receipt_page.max = max(1, (len(lines) + rows_per_page - 1) // rows_per_page)
        receipt_page.value = min(receipt_page.value, receipt_page.max)
        start = (receipt_page.value - 1) * rows_per_page
        page = lines.iloc[start:start + rows_per_page]
        previous_line = receipt_line.value
        receipt_line.options = [(f"ASN {row['WMS ASN#']} / line {row['Ref Line#']} / {row['sku']} / {row['Line status']} / variance {row.variance:+,.0f}", index)
                                for index, row in page.iterrows()]
        receipt_line.value = previous_line if previous_line in page.index else (int(page.index[0]) if not page.empty else None)
        def render():
            print(f"{receipt_status.value}: {len(lines):,} matching ASN lines; page {receipt_page.value} of {receipt_page.max}")
            print("Largest absolute variance first. Positive = over-receipt; negative = shortage. Summary totals above still cover the full date range.")
            if page.empty:
                print("No matching lines. Try another status or receipt-date range.")
                return
            view = page[["WMS ASN#", "Ref Line#", "sku", "shipment_type", "expected", "received", "variance", "Completion (%)", "Line status"]].rename(columns={
                "sku": "SKU", "shipment_type": "Shipment type [Shipment Type]",
                "expected": "Expected quantity [Qty Exp]", "received": "Received quantity [Qty Rcvd]",
                "variance": "Variance (received - expected)",
            })
            display(view)
        replace_output(line_panel, render)
    finally:
        inspection_busy = False
    show_line_records()


def refresh_receipts(change=None):
    global receiving_refresh_busy
    if receiving_refresh_busy:
        return
    receiving_refresh_busy = True
    try:
        receiving_totals(receipt_records())
    except ValueError as error:
        receipt_message.value = str(error) + " Previous table cleared."
        receipt_panel.outputs = ()
        line_panel.outputs = ()
        line_detail_panel.outputs = ()
        receiving_refresh_busy = False
        return
    receipt_message.value = ""
    try:
        replace_output(receipt_panel, render_receipts)
        refresh_inspection()
    finally:
        receiving_refresh_busy = False

for control in [inbound_shipment_filter,inbound_zone_filter,inbound_chart_view,inbound_chart_show]:
    control.observe(refresh_receipts,names="value")
receipt_start_input.observe(refresh_receipts, names="value")
receipt_end_input.observe(refresh_receipts, names="value")
preview_input.observe(refresh_receipts, names="value")
receipt_status.observe(refresh_inspection, names="value")
receipt_asn.observe(refresh_inspection, names="value")
receipt_sku.observe(refresh_inspection, names="value")
receipt_page.observe(refresh_inspection, names="value")
receipt_line.observe(show_line_records, names="value")
display(widgets.VBox([
    receipt_start_input, receipt_end_input,inbound_shipment_filter, inbound_zone_filter,
    inbound_chart_view,inbound_chart_show,
    receipt_message, receipt_panel,
    widgets.HTML("<h4>Investigate receiving differences, line by line</h4>"),
    widgets.HBox([receipt_status, receipt_asn]), receipt_sku, receipt_page, line_panel,
    receipt_line, line_detail_panel,receiving_record_open,receiving_record_inspector.widget,
]))
refresh_receipts()


### 2.3 Receiving work: dock states, waiting time and P90

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** IN-01 "Docked, not received in usual time" and DQ-03 "Dock with no receipt (120 h+)". Normal dock-to-receipt P90 = 5.2 h.  
> 🟧 **Overlaps but differs:** The notebook uses a P90 per shipment type from completions seen by the cutoff. Check the prototype uses the same benchmark and the same 120 h data-quality cut-off.  
> 🟨 **Not in the prototype yet:** The hourly receiving queue trend chart. Note: the "ASN lines" count per ASN still counts Ref Line# only (not changed, to do).  

**Purpose:** Identify receiving work still awaiting confirmation at a historical moment, separately from the receipt-period quantities in 2.2.

Choose As-of date/time and Apply time. ASN state selects docked/awaiting receipt, received, known not-yet-arrived, unknown dock time or all states. Shipment type, Duration flag, ASN text and pagination narrow the evidence list. State counts and their 24h comparison retain the full ASN population.

Docked/awaiting receipt means dock-in is observed but the latest ASN receipt is not. Received uses the latest ASN receipt; no partial ASN state is reconstructed. 'Not yet arrived' is a classification of records known from the full extract, not a forecast of incoming shipments. Future receipt timestamps are hidden.

**P90 rule:** Compare completed nonnegative dock-to-receipt durations of the same shipment type, using only completions observed by the cutoff. A completed ASN uses its actual duration; a waiting ASN uses elapsed hours since dock-in. Above-P90 rows show excess hours, completed benchmark count and whether they are still waiting. No completed examples means no benchmark. Evidence filters do not recalculate the benchmark; a small sample makes it unstable. P90 is a descriptive review threshold, not an SLA, cause or prediction.

Tick Show hourly receiving queue trend for a 24-hour line graph. It retains the full dock queue independently of evidence filters. Rising indicates building recorded work; falling indicates clearing; flat totals can hide turnover. Change is net workload movement, not the number received.

Select an ASN and Open ASN details for its summary, all listed lines and event history up to this cutoff. Missing receipt quantities remain unavailable rather than confirmed zero. An old dock timestamp without receipt confirmation could mean a long physical wait or missing system confirmation; the file cannot choose the cause.


In [ ]:
dock_record_pick=widgets.Dropdown(options=[],description="Inspect ASN:",style={"description_width":"initial"},layout=widgets.Layout(width="650px"))
dock_record_open=widgets.Button(description="Open ASN details")
dock_record_inspector=RecordInspector()
def open_dock_record(_):
    if dock_record_pick.value is not None:dock_record_inspector.open("ASN",dock_record_pick.value,dock_selected_time,origin="Opened from receiving dock/P90 evidence.")
dock_record_open.on_click(open_dock_record)

dock_shipment_filter=filter_widget(kpi_asns.shipment_type,"Shipment type:","420px")
dock_day = widgets.Dropdown(options=[(data_date_label(day), day) for day in kpi_available_days], value=DATA_TODAY, description="As-of date:")
dock_time = widgets.Text(value="09:00:00", description="Time:", layout=widgets.Layout(width="220px"))
dock_apply = widgets.Button(description="Apply time")
dock_state_filter = widgets.Dropdown(options=["Docked, awaiting receipt", "Received", "Not yet arrived (known from extract)", "Unknown dock time", "All ASN states"], description="ASN state:", layout=widgets.Layout(width="420px"))
dock_exception_filter = widgets.Dropdown(options=["All records","Longer than P90","Within P90","No completed benchmark"],description="Duration flag:",layout=widgets.Layout(width="360px"))
dock_search = widgets.Text(description="ASN contains:", continuous_update=False)
dock_page = widgets.BoundedIntText(value=1, min=1, max=1, description="ASN page:")
dock_trend = widgets.Checkbox(value=False, description="Show hourly receiving queue trend (24h)", indent=False)
dock_message = widgets.HTML()
dock_output = widgets.Output()
dock_selected_time = pd.Timestamp(AS_OF)
dock_busy = False

def dock_snapshot(at):
    t = pd.Timestamp(at)
    detail = kpi_asns.copy()
    detail["State"] = "Not yet arrived (known from extract)"
    detail.loc[detail.dock.isna(), "State"] = "Unknown dock time"
    detail.loc[detail.dock.le(t), "State"] = "Docked, awaiting receipt"
    detail.loc[detail.receipt.le(t), "State"] = "Received"
    detail["Waiting (hours)"] = ((t - detail.dock).dt.total_seconds() / 3600).where(detail.State.eq("Docked, awaiting receipt"))
    detail["Visible receipt time"] = detail.receipt.where(detail.receipt.le(t))
    # Benchmark each shipment type using completed, nonnegative ASN durations known at t.
    detail["_type"] = detail.shipment_type.astype("string").fillna("Unknown shipment type")
    completed_hours = (detail.receipt-detail.dock).dt.total_seconds()/3600
    completed = detail.loc[detail.State.eq("Received") & detail.dock.le(t) & completed_hours.ge(0)].copy()
    completed["_duration"] = completed_hours.loc[completed.index]
    benchmark = completed.groupby("_type")._duration.agg([("Benchmark samples","count"),("Benchmark P90 (hours)",lambda values:values.quantile(0.9))])
    detail["Benchmark P90 (hours)"] = detail._type.map(benchmark["Benchmark P90 (hours)"])
    detail["Benchmark samples"] = detail._type.map(benchmark["Benchmark samples"]).fillna(0).astype(int)
    detail["Dock elapsed (hours)"] = completed_hours.where(detail.State.eq("Received") & completed_hours.ge(0))
    waiting = detail.State.eq("Docked, awaiting receipt")
    detail.loc[waiting,"Dock elapsed (hours)"] = detail.loc[waiting,"Waiting (hours)"]
    assessable = detail["Dock elapsed (hours)"].notna() & detail["Benchmark P90 (hours)"].notna()
    detail["Beyond P90"] = assessable & detail["Dock elapsed (hours)"].gt(detail["Benchmark P90 (hours)"])
    detail["Hours beyond P90"] = (detail["Dock elapsed (hours)"]-detail["Benchmark P90 (hours)"]).clip(lower=0).where(assessable)
    detail["Receiving flag"] = "Not assessed"
    detail.loc[detail["Dock elapsed (hours)"].notna() & ~assessable,"Receiving flag"] = "No completed benchmark"
    detail.loc[assessable,"Receiving flag"] = "Within P90"
    detail.loc[detail["Beyond P90"] & ~waiting,"Receiving flag"] = "Completed > P90"
    detail.loc[detail["Beyond P90"] & waiting,"Receiving flag"] = "Still waiting > P90"
    return detail

def refresh_dock(change=None):
    global dock_busy, selected_dock_records
    if dock_busy:
        return
    dock_busy = True
    try:
        full = dock_snapshot(dock_selected_time)
        before = dock_snapshot(dock_selected_time - pd.Timedelta(hours=24))
        if dock_state_filter.value == "All ASN states":
            selected = full.copy()
        else:
            selected = full.loc[full.State.eq(dock_state_filter.value)].copy()
        selected=selected.loc[source_filter(selected.shipment_type,dock_shipment_filter.value)]
        if dock_exception_filter.value=="Longer than P90":
            selected=selected.loc[selected["Beyond P90"]]
        elif dock_exception_filter.value=="Within P90":
            selected=selected.loc[selected["Receiving flag"].eq("Within P90")]
        elif dock_exception_filter.value=="No completed benchmark":
            selected=selected.loc[selected["Receiving flag"].eq("No completed benchmark")]
        if dock_search.value.strip():
            selected = selected.loc[selected["WMS ASN#"].astype("string").str.contains(dock_search.value.strip(), regex=False, na=False)]
        selected = selected.sort_values(["Beyond P90","Hours beyond P90","Dock elapsed (hours)"],ascending=[False,False,False],na_position="last")
        selected_dock_records = selected
        dock_page.max = max(1, (len(selected) + preview_input.value - 1) // preview_input.value)
        dock_page.value = min(dock_page.value, dock_page.max)
        first=(dock_page.value-1)*preview_input.value
        evidence_options(dock_record_pick,[("ASN "+str(value),str(value)) for value in selected["WMS ASN#"].iloc[first:first+preview_input.value]])
        def render():
            print(f"RECEIVING WORK AT {dock_selected_time}. This as-of view is independent of the receipt-period filters above.")
            counts = pd.DataFrame({"Now": full.State.value_counts(), "24h earlier": before.State.value_counts()}).fillna(0).astype(int)
            counts["Change"] = counts.Now - counts["24h earlier"]
            display(counts)
            waiting=full.loc[full.State.eq("Docked, awaiting receipt")]
            print(f"WAITING BEYOND SAME-TYPE RECEIPT P90: {int(waiting['Beyond P90'].sum()):,} ASNs. Waiting without a benchmark: {int(waiting['Benchmark samples'].eq(0).sum()):,}.")
            print("P90 uses completed ASN receipts observed by this time, grouped by shipment type. This is a historical review flag, not an SLA breach.")
            if dock_trend.value:
                records = []
                for moment in pd.date_range(dock_selected_time-pd.Timedelta(hours=24), dock_selected_time, periods=25):
                    state = dock_snapshot(moment)
                    records.append({"Time": moment, "ASNs awaiting receipt": int(state.State.eq("Docked, awaiting receipt").sum())})
                trend_frame = pd.DataFrame(records).set_index("Time")
                fig,ax=plt.subplots(figsize=(10,4.5))
                ax.plot(trend_frame.index,trend_frame["ASNs awaiting receipt"],color="#176b87",linewidth=2,marker="o",markersize=3)
                ax.set_title(f"Receiving queue: 24 hours ending {dock_selected_time:%d %b %Y %H:%M:%S}")
                ax.set_ylabel("ASNs docked, awaiting receipt")
                ax.set_xlabel("Historical time (hourly observations)")
                ax.set_ylim(bottom=0);ax.grid(alpha=0.25)
                ax.yaxis.set_major_locator(plt.matplotlib.ticker.MaxNLocator(integer=True))
                ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
                fig.text(0.10,0.025,"Rising line: receiving work is building up. Falling line: the dock queue is clearing.\nA flat count can still contain new arrivals and completed receipts. The graph covers the full recorded ASN queue.",fontsize=9)
                fig.tight_layout(rect=(0,0.13,1,1))
                display(fig);plt.close(fig)
            print(f"{dock_state_filter.value}: {len(selected):,} matching ASNs; page {dock_page.value} of {dock_page.max}")
            first = (dock_page.value-1)*preview_input.value
            view = selected[["WMS ASN#", "shipment_type", "dock", "Visible receipt time", "Dock elapsed (hours)", "Benchmark P90 (hours)", "Hours beyond P90", "Benchmark samples", "Receiving flag", "lines", "State"]].copy()
            view["dock"] = view.dock.where(view.dock <= dock_selected_time)
            display(view.rename(columns={"shipment_type":"Shipment type [Shipment Type]", "dock":"Docked [Dock in Date & Time]",
                "Visible receipt time":"Visible receipt [GR Date & Time]", "lines":"ASN lines [Ref Line#]"}).iloc[first:first+preview_input.value].round(3))
        replace_output(dock_output, render)
    finally:
        dock_busy = False

def apply_dock_time(change=None):
    global dock_selected_time
    try:
        candidate = pd.Timestamp.combine(dock_day.value, parse_clock_time(dock_time.value))
    except (ValueError, TypeError):
        dock_message.value = "Enter time as HH:MM or HH:MM:SS. Results have not changed."
        return
    dock_selected_time = candidate
    dock_message.value = ""
    refresh_dock()

dock_day.observe(apply_dock_time, names="value")
dock_apply.on_click(apply_dock_time)
for control in [dock_shipment_filter,dock_state_filter, dock_exception_filter, dock_search, dock_page, dock_trend, preview_input]:
    control.observe(refresh_dock, names="value")
display(widgets.VBox([widgets.HBox([dock_day, dock_time, dock_apply]), dock_message,
    widgets.HBox([dock_state_filter, dock_page]),dock_shipment_filter,dock_exception_filter, dock_search, dock_trend, dock_output,widgets.HBox([dock_record_pick,dock_record_open]),dock_record_inspector.widget]))
refresh_dock()


32 ASNs changed | extra lines: 109
          old  new
WMS ASN#          
31254     500  502
31368      44   45
31370      60   71
31372      49   52
31374      47   50
31375      50   52
31377      59   81
31378      50   54
31388     235  239
31422      49   57


## 3. Milk run search and recorded journey

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Trace screen: search a carton ID, ASN or order and see its journey. Same counts: 7,869 cartons, 5,564 linked.  

**Purpose:** Follow recorded inbound-to-outbound associations for inbound **Milkrun to ABC** and outbound **Milkrun from ABC**. This view is restricted to that direction, not all warehouse flows.

Choose the independent date/time, Link status and optional search text. Search accepts partial carton/LPN, ASN, order or SKU. Page through the compact results and select a carton/LPN; its associated ASNs/orders, shipment references, SKU/lot, forwarder and observed event times appear directly below.

| Link status | Meaning by the selected time |
|---|---|
| Exactly linked | An observed inbound LPN equals an outbound Carton ID on an order created by the cutoff. |
| Inbound only | Receipt is observed, but no matching outbound order is observed. |
| Outbound only | An outbound order is observed, but no matching inbound receipt is observed. |

The counts describe distinct recorded IDs across these sides. Linked does not mean handed over, quantities reconciled or stock provenance independently verified. No counterpart can reflect timing, extract coverage or a different recorded identifier; it is not proof of missing stock. Multiple recorded associations are retained and identified rather than resolved arbitrarily.

The event list masks future events and labels adjusted inbound times. Receipt and Cyclone scan remain distinct system observations for evidence, but their near-coincident timestamps are not automatically treated as two physical processing steps. Select another carton here or use Section 5 for broader search. Quantity-content agreement is tested separately in Section 7.1.


In [13]:
# Milk-run-only search. Universal record lookup remains in Section 5.
trace_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="Date:")
trace_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"))
trace_apply=widgets.Button(description="Apply time")
trace_cohort=widgets.Dropdown(options=["All cohorts","Exactly linked","Inbound only","Outbound only"],description="Link status:",style={"description_width":"initial"})
trace_search=widgets.Text(description="Search:",continuous_update=False,layout=widgets.Layout(width="400px"))
trace_example=widgets.HTML("Example: carton/LPN, ASN, order key or SKU; partial IDs are accepted.")
trace_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Page:")
trace_record_pick=widgets.Dropdown(options=[],description="Select carton/LPN:",style={"description_width":"initial"},layout=widgets.Layout(width="650px"))
trace_message=widgets.HTML();trace_output=widgets.Output();trace_detail_output=widgets.Output()
trace_selected_time=pd.Timestamp(AS_OF);trace_busy=False

# Restrict both the register and detail events to the documented milk-run direction.
trace_in_mask=kpi_in.is_milkrun.to_numpy()
trace_out_mask=trace_outbound.to_numpy()
milk_record_in=record_ib.loc[trace_in_mask].copy()
milk_record_out=record_ob.loc[trace_out_mask].copy()

def milk_journey(unit,at):
    key=canonical_record_id(unit);t=pd.Timestamp(at)
    incoming=milk_record_in.loc[milk_record_in._HU.eq(key)&milk_record_in["GR Date & Time"].le(t)].copy()
    outgoing=milk_record_out.loc[milk_record_out._HU.eq(key)&milk_record_out[created_col].le(t)].copy()
    events=[]
    for frame,fields,id_col in [(incoming,[("Dock in Date & Time","Dock-in (+8h)"),("GR Date & Time","Goods receipt (+8h)")],"_ASN"),
        (outgoing,[(scan_col,"Cyclone scan"),(created_col,"Order created"),(packed_col,"Packed"),(released_col,"Handover to forwarder")],"_Order")]:
        for col,label in fields:
            for _,row in frame.loc[frame[col].notna()&frame[col].le(t)].iterrows():
                events.append({"Event time":row[col],"Event":label,"ASN [WMS ASN#]":row[id_col] if id_col=="_ASN" else "",
                    "Order [Order Key]":row[id_col] if id_col=="_Order" else "","Excel timestamp":col})
    timeline=pd.DataFrame(events,columns=["Event time","Event","ASN [WMS ASN#]","Order [Order Key]","Excel timestamp"]).drop_duplicates()
    if not timeline.empty:timeline=timeline.sort_values(["Event time","Event"]).reset_index(drop=True)
    return incoming,outgoing,timeline

def show_milk_journey(change=None):
    global selected_milk_timeline
    if trace_busy:return
    def render():
        global selected_milk_timeline
        if trace_record_pick.value is None:
            selected_milk_timeline=pd.DataFrame();print("No milk run records match these filters.");return
        unit=trace_record_pick.value
        incoming,outgoing,timeline=milk_journey(unit,trace_selected_time)
        selected_milk_timeline=timeline
        print(f"MILK RUN CARTON / LPN {unit} — as of {trace_selected_time}")
        if incoming.empty:print("No matching milk run receipt observed by this time; this is a traceability gap, not proof of missing stock.")
        if outgoing.empty:print("No matching milk run outbound order observed by this time.")
        if incoming._ASN.nunique()>1 or outgoing._Order.nunique()>1:print("Multiple recorded associations: all are retained below.")
        facts=pd.DataFrame({"Detail":["ASNs [WMS ASN#]","Orders [Order Key]","Shipment references [Shipment Ref]","SKUs [SKU]","Vendor lots [Vendor Lot]","Forwarders [Final Forwarder]"],
            "Value":[join_ids(incoming._ASN),join_ids(outgoing._Order),join_ids(incoming["Shipment Ref"]),
                join_ids(pd.concat([incoming.SKU,outgoing.SKU])),join_ids(pd.concat([incoming["Vendor Lot"],outgoing["Vendor Lot"]])),join_ids(outgoing["Final Forwarder"])]})
        display(facts)
        print("EVENT TIMES — inbound times include the project +8h adjustment; future events are hidden.")
        if timeline.empty:print("No observed event timestamps.")
        else:display(timeline)
    replace_output(trace_detail_output,render)

def refresh_trace(change=None):
    global trace_busy,selected_trace_records
    if trace_busy:return
    trace_busy=True
    try:
        full=kpi_cohorts(trace_selected_time)
        attrs,_,_=observed_trace_attributes(trace_selected_time)
        full=full.merge(attrs[["HU / carton ID","SKUs"]],on="HU / carton ID",how="left",validate="one_to_one")
        selected=full.copy() if trace_cohort.value=="All cohorts" else full.loc[full.Cohort.eq(trace_cohort.value)].copy()
        query=trace_search.value.strip()
        if query:selected=selected.loc[selected[["HU / carton ID","ASNs","Orders","SKUs"]].astype("string").apply(lambda col:col.str.contains(query,case=False,regex=False,na=False)).any(axis=1)]
        selected_trace_records=selected
        trace_page.max=max(1,(len(selected)+preview_input.value-1)//preview_input.value)
        trace_page.value=min(trace_page.value,trace_page.max)
        start=(trace_page.value-1)*preview_input.value;shown=selected.iloc[start:start+preview_input.value]
        evidence_options(trace_record_pick,[(str(v),str(v)) for v in shown["HU / carton ID"]])
        def render():
            print(f"MILK RUN ONLY — as of {trace_selected_time}")
            print("Linked: {:,} | Inbound-only: {:,} | Outbound-only: {:,}".format(*(int(full.Cohort.eq(v).sum()) for v in ["Exactly linked","Inbound only","Outbound only"])))
            print(f"{len(selected):,} matching cartons/LPNs; page {trace_page.value}/{trace_page.max}. Select a carton below to see details and event times.")
            view=shown[["HU / carton ID","ASNs","Orders","Cohort","receipt","created","Visible handover"]].rename(columns={
                "HU / carton ID":"Carton / LPN [LPN, Carton ID]","ASNs":"ASN [WMS ASN#]","Orders":"Order [Order Key]","Cohort":"Link status",
                "receipt":"Received (+8h) [GR Date & Time]","created":"Order created [Orderdate Date & Time]",
                "Visible handover":"Handover [Cargo Release Handover to Forwarder Date & Time]"})
            display(view)
        replace_output(trace_output,render)
    finally:trace_busy=False
    show_milk_journey()

def apply_trace_time(change=None):
    global trace_selected_time
    try:candidate=pd.Timestamp.combine(trace_day.value,parse_clock_time(trace_time.value))
    except (ValueError,TypeError):trace_message.value="Enter a valid time as HH:MM or HH:MM:SS. Results have not changed.";return
    trace_selected_time=candidate;trace_message.value="";trace_page.value=1;refresh_trace()

def reset_milk_page(change=None):
    trace_page.value=1;refresh_trace()
trace_day.observe(apply_trace_time,names="value");trace_apply.on_click(apply_trace_time)
for control in [trace_cohort,trace_search]:control.observe(reset_milk_page,names="value")
for control in [trace_page,preview_input]:control.observe(refresh_trace,names="value")
trace_record_pick.observe(show_milk_journey,names="value")
display(widgets.VBox([widgets.HBox([trace_day,trace_time,trace_apply]),trace_message,
    widgets.HBox([trace_search,trace_cohort]),trace_example,trace_page,trace_output,trace_record_pick,trace_detail_output]))
refresh_trace()


### 3.1 Elapsed process timing for linked milk run units

> **Review 16 Nov 2026 (prototype handover)**  
> 🟨 **Not in the prototype yet:** Stage timing with median and P90 per stage (dock → receipt → order → handover). The prototype shows the event timeline but not stage benchmarks.  

**Purpose:** Find linked units with unusually long recorded intervals, and inspect which stage accounts for the time.

This section has its **own As-of date, Time and Apply time**, independent of Section 3. Choose Stage, Records, partial ID search and page. The chart can be hidden without hiding statistics or evidence. Select a carton/LPN to see its ASN/order, start/end times, duration or current wait, P90 comparison, excess and event timeline.

| Stage | Recorded endpoints |
|---|---|
| Dock to goods receipt | Adjusted Dock in Date & Time → adjusted GR Date & Time. |
| Goods receipt to order creation | Adjusted GR Date & Time → Orderdate Date & Time. |
| Order creation to handover | Orderdate Date & Time → final recorded cargo handover. |
| Dock to handover | Adjusted dock-in → final recorded cargo handover. |

These are elapsed **calendar hours**, including waiting and non-working time, not measured labour time. There is no separate release-to-floor timestamp; creation is named explicitly rather than presented as a verified work-release event.

**Population and benchmarks:** Exact milk run ID links must have receipt and order creation observed by the cutoff. Ambiguous ASN/order associations, missing/unobserved endpoints and negative timestamp sequences are excluded for the affected stage. Median describes the middle completed duration; P90 is the 90th percentile of valid completed durations observed by this time. Pending waits do not enter those percentiles.

Completed > P90 compares actual duration with the stage benchmark. Still waiting > P90 compares elapsed time from an observed stage start with that benchmark. The result table shows flags, threshold and excess, ordered by excess hours. An empty waiting list for dock-to-receipt is expected when this linked population already requires a receipt; use 2.3 to inspect ASNs that have not yet received.

Thresholds use unrounded values; displays are rounded. By construction some completed observations exceed P90; sample size and ties affect the count. This is not a service deadline, forecast or diagnosis. These linked-unit stage benchmarks differ from 2.3's shipment-type ASN benchmarks. Creation/readiness constraints and capacity cannot be inferred from duration alone.


In [14]:
timing_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="As-of date:",style={"description_width":"initial"})
timing_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"),style={"description_width":"initial"})
timing_apply=widgets.Button(description="Apply time")
timing_message=widgets.HTML()
timing_selected_time=pd.Timestamp(AS_OF)

timing_record_pick=widgets.Dropdown(options=[],description="Select carton/LPN:",style={"description_width":"initial"},layout=widgets.Layout(width="650px"))
timing_detail_output=widgets.Output()

def show_timing_journey(change=None):
    global selected_timing_detail,selected_timing_timeline
    if timing_busy:return
    selected_timing_detail=pd.DataFrame();selected_timing_timeline=pd.DataFrame()
    def render():
        global selected_timing_detail,selected_timing_timeline
        unit=timing_record_pick.value
        if unit is None:
            print("No units match the selected stage and filters.");return
        matches=selected_timing_records.loc[selected_timing_records["HU / carton ID"].astype(str).eq(str(unit))]
        if matches.empty:
            print("Choose a carton from the current results page.");return
        row=matches.iloc[0]
        print(f"MILK RUN TIMING — carton/LPN {unit} — as of {timing_selected_time}")
        print("Stage: "+timing_stage.value)
        link="ID matched" if row.ASN_count==1 and row.Order_count==1 else "ID matched; multiple ASN/order associations — timing excluded"
        start,end=TIMING_STAGES[timing_stage.value]
        names={"dock":"Dock-in (+8h) [Dock in Date & Time]","receipt":"Receipt (+8h) [GR Date & Time]","created":"Order created ["+created_col+"]","handover":"Handover ["+released_col+"]"}
        facts=[("Link status",link),("ASNs [WMS ASN#]",row.ASNs),("Orders [Order Key]",row.Orders),
            ("Stage start — "+names[start],row[start]),("Stage end — "+names[end],row[end]),
            ("Completed duration (hours)",row["Duration (hours)"]),("Still waiting (hours)",row["Current wait (hours)"]),
            ("P90 benchmark (hours)",row["Benchmark P90 (hours)"]),("Hours beyond P90",row["Hours beyond P90"]),
            ("Result",row.Flag),("Measurement status",row.Status)]
        selected_timing_detail=pd.DataFrame(facts,columns=["Detail","Value"])
        display(selected_timing_detail)
        print("Missing duration/end time means it is unavailable or excluded, not zero. Pending waits are compared with P90 but do not enter the benchmark.")
        print("ID equality connects the records; it does not verify SKU/lot/quantity reconciliation.")
        _,_,timeline=milk_journey(unit,timing_selected_time)
        selected_timing_timeline=timeline
        print("EVENT TIMES — inbound +8h adjustment applied; future events hidden.")
        if timeline.empty:print("No observed events.")
        else:display(timeline)
    replace_output(timing_detail_output,render)

timing_stage = widgets.Dropdown(options=list(TIMING_STAGES),description="Stage:",layout=widgets.Layout(width="440px"))
timing_status = widgets.Dropdown(options=["Longer than P90","Completed > P90","Still waiting > P90","Included durations","Excluded / not yet observed","All linked records"],description="Records:")
timing_search = widgets.Text(description="ID contains:",continuous_update=False)
timing_page = widgets.BoundedIntText(value=1,min=1,max=1,description="Page:")
timing_chart_show = widgets.Checkbox(value=True,description="Show process-timing chart",indent=False)
timing_output = widgets.Output()
timing_busy=False

def refresh_process_timing(change=None):
    global timing_busy,selected_timing_summary,selected_timing_records
    if timing_busy:return
    timing_busy=True
    try:
        selected_timing_summary,stages=process_timing(timing_selected_time)
        table=stages[timing_stage.value]
        if timing_status.value=="Longer than P90":table=table.loc[table["Beyond P90"]]
        elif timing_status.value=="Completed > P90":table=table.loc[table.Flag.eq("Completed > P90")]
        elif timing_status.value=="Still waiting > P90":table=table.loc[table.Flag.eq("Still waiting > P90")]
        elif timing_status.value=="Included durations":table=table.loc[table.Status.eq("Included")]
        elif timing_status.value=="Excluded / not yet observed":table=table.loc[table.Status.ne("Included")]
        if timing_search.value.strip():
            table=table.loc[table[["HU / carton ID","ASNs","Orders"]].astype("string").apply(lambda col:col.str.contains(timing_search.value.strip(),case=False,regex=False,na=False)).any(axis=1)]
        table=table.sort_values("Hours beyond P90",ascending=False,na_position="last")
        selected_timing_records=table
        timing_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value)
        timing_page.value=min(timing_page.value,timing_page.max)
        first=(timing_page.value-1)*preview_input.value
        evidence_options(timing_record_pick,[(str(value),str(value)) for value in table["HU / carton ID"].iloc[first:first+preview_input.value]])
        def render():
            print(f"ELAPSED TIME FOR EXACTLY LINKED MILK RUN UNITS, AS OF {timing_selected_time}")
            display(selected_timing_summary.round(2))
            valid_summary=selected_timing_summary.loc[selected_timing_summary["Included HUs"].gt(0)]
            if timing_chart_show.value and not valid_summary.empty:
                fig,ax=plt.subplots(figsize=(10,4.5))
                valid_summary[["Median (hours)","P90 (hours)"]].plot.barh(ax=ax,color=["#176b87","#db7c26"])
                ax.set_title("Elapsed time by stage — linked, valid observations only")
                ax.set_xlabel("Hours");ax.set_ylabel("");ax.grid(axis="x",alpha=0.2)
                display_operational_figure(fig)
            print(f"{timing_stage.value}; {timing_status.value}: {len(table):,} HUs; page {timing_page.value}/{timing_page.max}")
            first=(timing_page.value-1)*preview_input.value
            start_column,end_column=TIMING_STAGES[timing_stage.value]
            source_names={"dock":"Dock in Date & Time","receipt":"GR Date & Time","created":created_col,"handover":released_col}
            print("P90 is calculated from completed valid observations, regardless of the evidence filter. Pending work is compared using current wait and is not included in duration percentiles.")
            view=table[["HU / carton ID","ASNs","Orders",start_column,end_column,"Duration (hours)","Current wait (hours)",
                "Benchmark P90 (hours)","Hours beyond P90","Flag","Status"]].rename(columns={
                    "HU / carton ID":"HU / carton ID [LPN, Carton ID]","ASNs":"ASNs [WMS ASN#]","Orders":"Orders [Order Key]",
                    start_column:"Stage start ["+source_names[start_column]+"]",end_column:"Stage end ["+source_names[end_column]+"]"})
            display(view.iloc[first:first+preview_input.value].round(3))
        replace_output(timing_output,render)
    finally:timing_busy=False
    show_timing_journey()

for control in [timing_stage,timing_status,timing_search,timing_page,timing_chart_show,preview_input]:control.observe(refresh_process_timing,names="value")
def apply_timing_time(change=None):
    global timing_selected_time,timing_busy
    try:candidate=pd.Timestamp.combine(timing_day.value,parse_clock_time(timing_time.value))
    except (ValueError,TypeError):
        timing_message.value="Enter time as HH:MM or HH:MM:SS. Results have not changed.";return
    timing_selected_time=candidate;timing_message.value=""
    timing_busy=True;timing_page.value=1;timing_busy=False
    refresh_process_timing()
timing_day.observe(apply_timing_time,names="value")
timing_apply.on_click(apply_timing_time)
display(widgets.VBox([widgets.HBox([timing_day,timing_time,timing_apply]),timing_message,
    widgets.HBox([timing_stage,timing_status]),widgets.HBox([timing_search,timing_page]),timing_chart_show,timing_output,timing_record_pick,timing_detail_output]))
refresh_process_timing()

timing_record_pick.observe(show_timing_journey,names="value")


## 4. Event activity and recent history

> **Review 16 Nov 2026 (prototype handover)**  
> 🟨 **Not in the prototype yet:** Hourly event activity charts and the 3-hour recent-event log.  

**Purpose:** See when recorded activity occurred and identify the records behind recent events across all flows.

Choose the independent date/time and event types. Show event activity charts controls optional hourly charts over 6, 24 or 48 hours. The recent-event log always covers the preceding **three hours**, newest first. Search event ID accepts a full or partial ASN, order or carton reference and filters **the log only**; chart counts retain all selected event types. Use Log page for the matching records.

| Event | Counting unit / timestamp |
|---|---|
| Dock-in / goods receipt | Distinct ASN; earliest dock and latest receipt, adjusted +8h. |
| Order creation | Distinct Order Key; earliest recorded creation. |
| Packing / handover | Distinct identified carton; latest recorded stage timestamp. |
| Cyclone scan | Distinct identified carton; earliest recorded scan in this event overview. |

Different lane units cannot be added as shipments or treated as directly comparable physical throughput. Missing IDs/times are not counted. Windows exclude their start and include their end; the final hour can be partial. Zero recorded activity does not establish complete coverage or no physical work.

**Record history:** Select an event and Show record history. Order events open the order; ASN events open the ASN. Carton events open the owning order when exactly one observed owner is available; multiple owners retain a shared-carton view. The summary, related line/ID tables and event history are read-only. History extends before the three-hour log and stops at the selected cutoff. Use its table/history pages for longer records, or search a related ID in Section 5.

Creation is not a separate verified release-to-floor event. Inbound alignment and static-assignment limitations from Section 1 apply; no unseen events or causes are inferred.


In [15]:
event_record_pick=widgets.Dropdown(options=[],description="Select event:",style={"description_width":"initial"},layout=widgets.Layout(width="750px"))
event_record_open=widgets.Button(description="Show record history")
event_record_inspector=RecordInspector()
def event_history_target(row,at):
    kind={"ASN":"ASN","Order":"Order","Carton":"Handling unit / carton"}[row.Entity]
    key=canonical_record_id(row["Record ID"])
    if kind=="Handling unit / carton":
        rows=record_ob.loc[record_ob._HU.eq(key)&record_ob[[created_col,scan_col,packed_col,released_col]].le(pd.Timestamp(at)).any(axis=1)]
        owners=sorted(set(rows._Order)-{""})
        if len(owners)==1:return "Order",owners[0]
        # Retain a shared-carton view when several orders exist; never pick an arbitrary owner.
    return kind,key

def open_event_record(_=None):
    if event_record_pick.value is None:return
    row=selected_event_log.reset_index(drop=True).loc[event_record_pick.value]
    kind,key=event_history_target(row,event_selected_time)
    event_record_inspector.open(kind,key,event_selected_time,origin="Recent event: "+row.Event)
event_record_open.on_click(open_event_record)

# One event per entity/stage, never one event per duplicated item row.
carton_event_data=kpi_out.dropna(subset=["Carton ID"]).groupby("Carton ID").agg(
    packed=(packed_col,"max"),scan=(scan_col,"min"),handover=(released_col,"max"))
def event_frame(times,ids,event,entity,source):
    return pd.DataFrame({"Time":times.to_numpy(),"Record ID":ids.astype("string").to_numpy(),
        "Event":event,"Entity":entity,"Source timestamp":source}).dropna(subset=["Time","Record ID"])
warehouse_events=pd.concat([
    event_frame(kpi_asns.dock,kpi_asns["WMS ASN#"],"Dock-in","ASN","Dock in Date & Time (+8h)"),
    event_frame(kpi_asns.receipt,kpi_asns["WMS ASN#"],"Goods receipt","ASN","GR Date & Time (+8h)"),
    event_frame(orders.created,orders["Order Key"],"Order creation / release-to-work proxy","Order",created_col),
    event_frame(carton_event_data.packed,pd.Series(carton_event_data.index),"Packing","Carton",packed_col),
    event_frame(carton_event_data.scan,pd.Series(carton_event_data.index),"Cyclone scan","Carton",scan_col),
    event_frame(carton_event_data.handover,pd.Series(carton_event_data.index),"Handover","Carton",released_col),
],ignore_index=True).sort_values("Time")
EVENT_TYPES=["Dock-in","Goods receipt","Order creation / release-to-work proxy","Packing","Cyclone scan","Handover"]
event_days=sorted(set(warehouse_events.Time.dt.date)|{DATA_TODAY})
event_day=widgets.Dropdown(options=[(data_date_label(day),day) for day in event_days],value=DATA_TODAY,description="Date:")
event_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"))
event_apply=widgets.Button(description="Apply time")
event_types=widgets.SelectMultiple(options=EVENT_TYPES,value=tuple(EVENT_TYPES),description="Events:",rows=6,layout=widgets.Layout(width="570px"))
event_chart_show=widgets.Checkbox(value=False,description="Show event activity charts",indent=False)
event_window=widgets.Dropdown(options=[("Last 6 hours",6),("Last 24 hours",24),("Last 48 hours",48)],value=24,description="Chart window:")
event_search=widgets.Text(description="Log ID contains:",continuous_update=False)
event_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Log page:")
event_message=widgets.HTML();event_output=widgets.Output()
event_selected_time=pd.Timestamp(AS_OF);event_busy=False

def event_selection(at,hours,types,query=""):
    t=pd.Timestamp(at)
    selected=warehouse_events.loc[warehouse_events.Event.isin(types)]
    chart=selected.loc[selected.Time.gt(t-pd.Timedelta(hours=hours))&selected.Time.le(t)]
    log=selected.loc[selected.Time.gt(t-pd.Timedelta(hours=3))&selected.Time.le(t)].sort_values("Time",ascending=False)
    if query.strip():log=log.loc[log["Record ID"].str.contains(query.strip(),case=False,regex=False,na=False)]
    hours_index=pd.date_range((t-pd.Timedelta(hours=hours)).floor("h"),t.floor("h"),freq="h")
    hourly=pd.DataFrame(index=hours_index)
    for event in types:
        times=chart.loc[chart.Event.eq(event),"Time"]
        hourly[event]=times.dt.floor("h").value_counts().reindex(hours_index,fill_value=0)
    return chart,log,hourly

def refresh_events(change=None):
    global event_busy,selected_event_chart,selected_event_log,selected_event_hourly
    if event_busy:return
    event_busy=True
    try:
        if change is not None and change.get("owner") is not event_page:event_page.value=1
        selected_event_chart,selected_event_log,selected_event_hourly=event_selection(event_selected_time,event_window.value,event_types.value,event_search.value)
        event_page.max=max(1,(len(selected_event_log)+preview_input.value-1)//preview_input.value)
        event_page.value=min(event_page.value,event_page.max)
        first=(event_page.value-1)*preview_input.value
        log_rows=selected_event_log.reset_index(drop=True).iloc[first:first+preview_input.value]
        evidence_options(event_record_pick,[(str(row["Record ID"])+" — "+row.Event,int(index)) for index,row in log_rows.iterrows()])
        def render():
            print(f"EVENT ACTIVITY ENDING {event_selected_time}; chart: {event_window.value}h, log: 3h")
            if not event_types.value:
                print("Select at least one event type. Use Ctrl/Cmd-click to select multiple event types.")
            elif event_chart_show.value:
                fig,axes=plt.subplots(len(event_types.value),1,figsize=(10,max(3,1.5*len(event_types.value))),sharex=True,squeeze=False)
                units={"Dock-in":"ASN","Goods receipt":"ASN","Order creation / release-to-work proxy":"Order","Packing":"Carton","Cyclone scan":"Carton","Handover":"Carton"}
                for ax,event in zip(axes[:,0],event_types.value):
                    ax.bar(selected_event_hourly.index,selected_event_hourly[event],width=0.03,color="#176b87")
                    ax.set_ylabel(units[event]+"s",fontsize=8)
                    ax.set_title(event,loc="left",fontsize=9);ax.grid(axis="y",alpha=0.2)
                axes[-1,0].xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=7))
                axes[-1,0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
                axes[-1,0].set_xlabel("Historical hour; the final hour may be partial")
                fig.suptitle("Recorded events per hour — ASNs, orders and cartons are different units",fontsize=11)
                display_operational_figure(fig)
            print(f"RECENT EVENT LOG: {len(selected_event_log):,} matches during the preceding 3h; page {event_page.value}/{event_page.max}")
            print("ID search filters the log only. Chart counts retain all selected event types.")
            first=(event_page.value-1)*preview_input.value
            display(selected_event_log.rename(columns={"Time":"Event time [see Source timestamp]","Record ID":"Record ID [WMS ASN#, Order Key, Carton ID]"}).iloc[first:first+preview_input.value])
        replace_output(event_output,render)
        event_record_inspector.clear()
    finally:event_busy=False

def apply_event_time(change=None):
    global event_selected_time
    try:candidate=pd.Timestamp.combine(event_day.value,parse_clock_time(event_time.value))
    except (ValueError,TypeError):
        event_message.value="Enter time as HH:MM or HH:MM:SS. Results have not changed.";return
    event_selected_time=candidate;event_message.value="";refresh_events()

event_day.observe(apply_event_time,names="value");event_apply.on_click(apply_event_time)
for control in [event_chart_show,event_types,event_window,event_search,event_page,preview_input]:control.observe(refresh_events,names="value")
for control in [event_day,event_time,event_types,event_window,event_search,event_page]:
    control.style.description_width="initial"
    control.layout.max_width="100%"
event_search.description="Search event ID:"
event_page.description="Log page:"
event_window.description="Chart period:"
display(widgets.VBox([widgets.HBox([event_day,event_time,event_apply]),event_message,event_types,widgets.HBox([event_chart_show,event_window]),
    widgets.HBox([event_search,event_page]),event_output,widgets.HBox([event_record_pick,event_record_open]),event_record_inspector.widget]))
refresh_events()


## 5. Universal record search

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Trace screen search (carton, ASN, order).  
> 🟧 **Overlaps but differs:** The notebook also searches Shipment Ref, LP, Loading List#, Wave Key and SKU; the prototype searches fewer fields.  
> 🟥 **Changed in this version:** The order inspector now counts shipped quantity once per order line.  

**Purpose:** Find a record using whichever reference you have, then inspect its facts, lines, recorded associations and history across all flows.

1. Enter a reference and select Search reports. The example beside the input is a format hint, not a prefilled search.
2. Optionally open Search options for Contains, Starts with or Exact identifier. Matching ignores case; exact numeric identifiers ignore leading-zero padding. Prefix/contains searches support partial SKUs and carton IDs.
3. Choose a typed result (ASN, Order or Handling unit/carton). Use Record type and Results page to narrow/browse matches.
4. Read the summary, related-record/line tables and event timeline. Larger tables have pages. To inspect a related entity, search its listed ID.

Inbound search fields are WMS ASN#, Shipment Ref, LPN and SKU. Outbound fields are Order Key, External Order Key, Carton ID, LP, Loading List#, Wave Key and SKU. A matching SKU/carton can return its owning ASN or order as well; Matched fields explains why. Matching source rows are report-row counts, not unique shipments.

**Scope:** Search uses the full imported reports independently of other date/filter controls. Its selected result opens **full-report history**. In contrast, local inspectors opened from a snapshot or receipt period use that view's cutoff. Tables use original Excel names or bracketed source labels and adjusted inbound timestamps.

An exact shared ID establishes a recorded association, not physical provenance or reconciled contents. Missing counterparts and multiple associations remain explicit. Expected quantity belongs to an ASN line, not each carton. General order-line shipped quantities sum reported source rows, which may repeat per-line totals across cartons; Section 7.1 uses a different, explicit reconciliation grain. No data is edited and no experimental link is inserted into these records.


In [16]:
# One search across original identifiers. The input stays empty; its example is outside.
LOOKUP_FIELDS={
    "Inbound":["WMS ASN#","Shipment Ref","LPN","SKU"],
    "Outbound":["Order Key","External Order Key","Carton ID","LP","Loading List#","Wave Key","SKU"],
}
LOOKUP_FRAMES={"Inbound":ib,"Outbound":ob}

def lookup_text(values):
    return values.astype("string").str.strip().str.replace(r"\.0$","",regex=True).replace("",pd.NA)

lookup_cache={source:{field:lookup_text(frame[field]) for field in LOOKUP_FIELDS[source]}
              for source,frame in LOOKUP_FRAMES.items()}

def identifier_match(values,query,mode):
    key=str(query).strip()
    if mode=="Exact identifier":
        clean=values.copy()
        numeric=clean.str.fullmatch(r"[0-9]+",na=False)
        clean.loc[numeric]=clean.loc[numeric].str.lstrip("0").replace("","0")
        normal=key.lstrip("0") or "0" if key.isdigit() else key
        return clean.str.casefold().eq(normal.casefold()).fillna(False)
    if mode=="Starts with":return values.str.casefold().str.startswith(key.casefold(),na=False)
    return values.str.contains(key,case=False,regex=False,na=False)

def unified_lookup(query,mode):
    if not str(query).strip():return pd.DataFrame(columns=["Result type","Record ID","Reports","Matched fields","Matching source rows"])
    matches=[]
    for source,frame in LOOKUP_FRAMES.items():
        fields=pd.Series("",index=frame.index,dtype="string")
        any_match=pd.Series(False,index=frame.index)
        for field,values in lookup_cache[source].items():
            mask=identifier_match(values,query,mode)
            any_match|=mask
            fields.loc[mask]=fields.loc[mask]+field+"; "
        entities=[("ASN","WMS ASN#"),("Handling unit / carton","LPN")] if source=="Inbound" else [("Order","Order Key"),("Handling unit / carton","Carton ID")]
        for entity,id_column in entities:
            ids=lookup_text(frame.loc[any_match,id_column])
            hit=pd.DataFrame({"Result type":entity,"Record ID":ids,"Reports":source,"Matched fields":fields.loc[any_match]})
            matches.append(hit.loc[hit["Record ID"].notna()])
    combined=pd.concat(matches,ignore_index=True)
    if combined.empty:return pd.DataFrame(columns=["Result type","Record ID","Reports","Matched fields","Matching source rows"])
    def matched_fields(values):
        return ", ".join(sorted(set(token.strip() for value in values for token in value.split(";") if token.strip())))
    return combined.groupby(["Result type","Record ID"],as_index=False).agg(
        Reports=("Reports",join_ids),**{"Matched fields":("Matched fields",matched_fields),"Matching source rows":("Reports","size")})

lookup_query=widgets.Text(value="",description="Search:",placeholder="",layout=widgets.Layout(width="430px"))
lookup_example=str(ib["WMS ASN#"].dropna().iloc[0])
lookup_hint=widgets.HTML(value=f"<span style='color:#888'>Example: {lookup_example} (ASN)</span>")
lookup_mode=widgets.Dropdown(options=["Contains","Starts with","Exact identifier"],description="Match:")
lookup_button=widgets.Button(description="Search reports")
lookup_type=widgets.Dropdown(options=["All result types","ASN","Order","Handling unit / carton"],description="Results:")
lookup_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Result page:")
lookup_pick=widgets.Dropdown(options=[],description="Inspect result:",style={"description_width":"initial"},layout=widgets.Layout(width="800px"))
lookup_record_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Record page:")
lookup_output=widgets.Output();lookup_inspector=RecordInspector(scope="Full reports")
lookup_results=pd.DataFrame();lookup_visible_results=pd.DataFrame();lookup_busy=False;lookup_last_query=""

def lookup_entity_records(kind,id_):
    if kind=="ASN":return ib.loc[identifier_match(lookup_cache["Inbound"]["WMS ASN#"],id_,"Exact identifier")],ob.iloc[:0]
    if kind=="Order":return ib.iloc[:0],ob.loc[identifier_match(lookup_cache["Outbound"]["Order Key"],id_,"Exact identifier")]
    return (ib.loc[identifier_match(lookup_cache["Inbound"]["LPN"],id_,"Exact identifier")],
            ob.loc[identifier_match(lookup_cache["Outbound"]["Carton ID"],id_,"Exact identifier")])

def show_lookup_entity(change=None):
    if lookup_busy:return
    if lookup_pick.value is None or lookup_visible_results.empty:
        lookup_inspector.clear();return
    chosen=lookup_visible_results.loc[lookup_pick.value]
    lookup_inspector.open(chosen["Result type"],chosen["Record ID"],full=True,origin="Opened from unified lookup; full-report reference scope.")

def refresh_lookup_results(change=None):
    global lookup_busy,lookup_visible_results
    if lookup_busy:return
    lookup_busy=True
    try:
        table=lookup_results.copy()
        if not table.empty and lookup_type.value!="All result types":table=table.loc[table["Result type"].eq(lookup_type.value)]
        lookup_visible_results=table.reset_index(drop=True)
        lookup_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value)
        lookup_page.value=min(lookup_page.value,lookup_page.max)
        first=(lookup_page.value-1)*preview_input.value
        shown=lookup_visible_results.iloc[first:first+preview_input.value]
        lookup_pick.options=[(f"{row['Result type']} {row['Record ID']} — {row['Reports']}",index) for index,row in shown.iterrows()]
        lookup_pick.value=int(shown.index[0]) if not shown.empty else None
        lookup_record_page.value=1
        def render():
            if not lookup_last_query:print("Enter an identifier or partial SKU/carton prefix, then Search reports.")
            else:
                print(f"Results for {lookup_last_query!r}: {len(table):,} matching typed results; page {lookup_page.value}/{lookup_page.max}. Matches can identify the owning ASN/order as well as a carton.")
                display(shown)
        replace_output(lookup_output,render)
    finally:lookup_busy=False
    show_lookup_entity()

def run_lookup(change=None):
    global lookup_results,lookup_last_query
    lookup_last_query=lookup_query.value.strip()
    lookup_results=unified_lookup(lookup_query.value,lookup_mode.value)
    lookup_page.value=1;refresh_lookup_results()

lookup_button.on_click(run_lookup)
lookup_mode.observe(run_lookup,names="value")
for control in [lookup_type,lookup_page,preview_input]:control.observe(refresh_lookup_results,names="value")
lookup_pick.observe(show_lookup_entity,names="value")
def search_ui_row(*children):return widgets.HBox(list(children),layout=widgets.Layout(flex_flow="row wrap",align_items="center",gap="8px"))
for control in [lookup_query,lookup_mode,lookup_type,lookup_page,lookup_pick]:
    control.style.description_width="initial";control.layout.max_width="100%"
lookup_pick.description="Choose a result:"
lookup_type.description="Record type:"
lookup_page.description="Results page:"
lookup_mode.description="Match method:"
search_options=widgets.Accordion(children=[lookup_mode],selected_index=None)
search_options.set_title(0,"Search options — contains, prefix or exact ID")
display(widgets.VBox([widgets.HTML("<b>1. Find a record</b>"),search_ui_row(lookup_query,lookup_button),lookup_hint,search_options,
    widgets.HTML("<b>2. Choose a result</b>"),search_ui_row(lookup_type,lookup_page),lookup_output,lookup_pick,
    widgets.HTML("<b>3. Record summary and history</b>"),lookup_inspector.widget]))
refresh_lookup_results()


### 5.1 Optional experiment: suggesting an ASN from SKU, lot and receipt time

> **Review 16 Nov 2026 (prototype handover)**  
> 🟨 **Not in the prototype yet:** The SKU + lot ASN suggestion back-test. To decide at the handover: keep as a notebook experiment, or add it as a clearly labelled "possible link" (not an exact trace).  

**Purpose:** Evaluate a fallback linking rule when a direct LPN/Carton ID connection is unavailable. This is a method-validation experiment rather than an operational trace or a repair.

**Rule:** For an outbound row's SKU and vendor lot, find the latest adjusted inbound receipt at or before order creation. One ASN gives a unique candidate; several ASNs at the latest timestamp give an ambiguous candidate set; no eligible receipt gives no candidate.

Click Run inference back-test. The experiment uses milk run rows with a known unambiguous reference ASN from exact IDs, ignores that reference while predicting, then compares candidate and reference. Cases filters All reference rows, Unique correct, Unique incorrect, Ambiguous inference or No candidate. Page browses results; Show back-test results hides the output.

| Metric | Denominator / interpretation |
|---|---|
| Reference outbound source rows | Eligible report rows, not unique cartons/orders. Ambiguous exact reference associations are excluded and counted. |
| Correct / all reference rows | Unique correct predictions divided by all reference rows, including no-candidate/tied cases. |
| Correct / unique candidate rows | Accuracy among rows with exactly one proposed ASN. |
| Candidate coverage | Rows with any prior candidate divided by reference rows. Coverage is not correctness. |

The detail table shows source row, carton, order, SKU/lot, reference and candidate ASN, result/reason, candidate receipt and creation cutoff. The reference is a recorded ID match, not independently observed physical allocation.

**Limits:** A result on these milk run rows does not establish accuracy for Normal flow, new data or an individual prediction. No quantity allocation, source-file repair or confirmed link is produced. Operational traceability and scenario readiness do not use these inferred candidates.


In [17]:
backtest_button=widgets.Button(description="Run inference back-test",layout=widgets.Layout(width="220px"))
backtest_show=widgets.Checkbox(value=True,description="Show back-test results",indent=False)
backtest_case=widgets.Dropdown(options=["All reference rows","Unique correct","Unique incorrect","Ambiguous inference","No candidate"],description="Cases:")
backtest_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Page:")
backtest_output=widgets.Output();inference_backtest_rows=pd.DataFrame();inference_backtest_metrics=pd.DataFrame()

def run_inference_backtest():
    truth=record_ib.loc[record_ib._HU.ne("")].groupby("_HU")._ASN.agg(lambda values:tuple(sorted(set(values)-{""})))
    exact=record_ob.loc[record_ob["DN Type"].astype("string").str.strip().str.casefold().eq("milkrun from abc")&record_ob._HU.isin(truth.index)].copy()
    ambiguous_truth=exact._HU.map(truth).map(len).ne(1)
    reference=exact.loc[~ambiguous_truth].copy()
    cache={};rows=[]
    for index,row in reference.iterrows():
        cache_key=(row._SKU,row._Lot,row[created_col])
        if cache_key not in cache:cache[cache_key]=infer_receipt_candidate(row._SKU,row._Lot,row[created_col])
        candidate=cache[cache_key];true_asn=truth.loc[row._HU][0]
        unique=len(candidate["Candidates"])==1
        correct=unique and candidate["Candidates"][0]==true_asn
        category="Unique correct" if correct else "Unique incorrect" if unique else "Ambiguous inference" if len(candidate["Candidates"])>1 else "No candidate"
        rows.append({"Source row":index,"Carton ID":row._HU,"Order Key":row._Order,"SKU":row._SKU,"Vendor Lot":row._Lot,
            "True ASN (exact LPN link)":true_asn,"Candidate ASNs":", ".join(candidate["Candidates"]),"Result":category,
            "Reason":candidate["Reason"],"Candidate receipt (+8h) [GR Date & Time]":candidate["Receipt time"],
            "Order created [Orderdate Date & Time]":row[created_col]})
    results=pd.DataFrame(rows)
    n=len(results);unique=int(results.Result.isin(["Unique correct","Unique incorrect"]).sum()) if n else 0
    correct=int(results.Result.eq("Unique correct").sum()) if n else 0
    ties=int(results.Result.eq("Ambiguous inference").sum()) if n else 0
    covered=int(results.Result.ne("No candidate").sum()) if n else 0
    metrics=pd.DataFrame({"Value":[n,int(ambiguous_truth.sum()),unique,correct,ties,covered,
        f"{100*correct/n:.2f}%" if n else "N/A",f"{100*correct/unique:.2f}%" if unique else "N/A",f"{100*covered/n:.2f}%" if n else "N/A"]},
        index=["Reference outbound source rows","Rows excluded: ambiguous exact ASN truth","Unique candidate rows","Unique correct rows","Tied candidate rows","Rows with any prior candidate",
               "Unique correct / all reference rows","Correct / unique candidate rows","Candidate coverage / reference rows"])
    return metrics,results

def show_inference_backtest(change=None):
    if not backtest_show.value:backtest_output.outputs=();return
    def render():
        if inference_backtest_rows.empty:
            print("Back-test has not run. Click Run inference back-test to evaluate the candidate rule on known milk-run references.");return
        print("OFFLINE FULL-EXTRACT BACK-TEST — not validated accuracy for Normal orders or new data")
        display(inference_backtest_metrics)
        table=inference_backtest_rows if backtest_case.value=="All reference rows" else inference_backtest_rows.loc[inference_backtest_rows.Result.eq(backtest_case.value)]
        backtest_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value)
        backtest_page.value=min(backtest_page.value,backtest_page.max)
        first=(backtest_page.value-1)*preview_input.value
        print(f"{backtest_case.value}: {len(table):,} rows; page {backtest_page.value}/{backtest_page.max}. Each reference row is an outbound report row, not a unique carton.")
        display(table.iloc[first:first+preview_input.value])
    replace_output(backtest_output,render)

def execute_inference_backtest(_):
    global inference_backtest_metrics,inference_backtest_rows
    inference_backtest_metrics,inference_backtest_rows=run_inference_backtest()
    backtest_show.value=True;show_inference_backtest()

backtest_button.on_click(execute_inference_backtest)
for control in [backtest_show,backtest_case,backtest_page,preview_input]:control.observe(show_inference_backtest,names="value")
display(widgets.VBox([backtest_button,backtest_show,widgets.HBox([backtest_case,backtest_page]),backtest_output]))
show_inference_backtest()


## 6. Handover performance and group analysis

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Manager view and MGR-01 "On-time below target", MGR-02 "Forwarder with a high late share". Uses definition A (25.5% released after plan for orders planned 17–23 Aug).  
> 🟧 **Overlaps but differs:** The prototype still says "Late = handed over on a later calendar day than planned" and has a 70% target. DSV said the planned date is a review signal, not proof of lateness.  
> 🟨 **Not in the prototype yet:** Definitions B, C and D (4.1%, 58.5%, 21.5%) and the forwarder × country matrix.  
> 🟥 **Changed in this version:** Visible widget labels say "released after plan". Table column names still say "Late" (to do).  

**Purpose:** Compare planned dates with recorded handover to the forwarder, then inspect which days/groups contain lateness. This does not measure customer delivery or establish why a group is late.

**Cohort controls:** Choose inclusive Planned from/to, DN type and Final/Default Forwarder. One order counts once using its latest handover. This is **full-extract completed-outcome analysis**, not a 09:00 snapshot. Conflicting plans and absent handovers are excluded from measurable outcomes. Missing-plan orders cannot enter a planned-day cohort and are investigated in 6.1. Planned-before-creation records remain kept and flagged. Multiple metadata values are labelled rather than assigned arbitrarily.

~~| Definition | Late when actual handover is… |~~ **→ | Definition | ~~Late~~ **Released after plan** when actual handover is… |**
|---|---|
| A — default | On a later calendar day than planned. |
| B | Two or more calendar days after planned. |
| C | Strictly after 18:00 on the planned day. |
| D | Strictly after noon the following day. |

These are working alternatives from the project documentation, not confirmed service rules. The comparison table applies all four to the same measurable cohort. The selected rule controls headline/group late rates and late-order evidence. On-time/not-late includes early handover.

**Read the tables:** Selected planned orders = measurable + excluded; measurable = late + not late. The daily table reconciles these counts by planned day. Group by DN type, country, forwarder, planned day or handover day. The group table shows sample size, exclusions, late/on-time shares, median/average calendar-day difference among orders classified late, 2+ day share and share of all late orders. Under C an order can be late on the same day and therefore have zero calendar days late. The 1/2/3+ day bands and 2+ day share always use calendar differences, independently of the chosen definition.

Minimum group N labels small groups; Hide groups can suppress them. It is not statistical confidence. Selecting a group, matrix cell, day band or order/customer-order text narrows the evidence only, not headline totals. Use group/day/order pages and select a late order to open its full actual history.

**Optional charts:** All start hidden. Calendar-day distribution includes early and same-day orders. Group severity bars split the displayed group page by calendar-day bands. The paged forwarder × country matrix shows late % with n; cells below the chosen matrix minimum show only n. Blank cells have no orders for that pair on the displayed page. The matrix-cell selector narrows late-order evidence.

Extract edges and absent still-open orders can bias completed-outcome rates. Static forwarder/country codes do not establish transport mode, collection schedule, cause or responsibility. Source definitions: Outbound_Twin_Documentation.pdf, rules/definitions and planned-date reconciliation sections; percentages are calculated from the files rather than copied from the document.


In [18]:
# REVIEW 16 Nov - CHANGED (wording only): visible widget labels say "released after plan" instead of "late".
# Why: DSV (11 Nov, A4 follow-up) said the Planned Post Date alone does not prove lateness (holds, full flights),
# so the deck, logs and prototype treat it as a signal to review. Old labels: "Late definition:" and "Select late order:".
# TODO (data team): rename the "Late"/"Late (%)" table columns the same way; internal names are left as they are.
# Completed-outcome analysis, using a planned-date cohort from the full report.
performance_base=orders.merge(order_search_attributes,on="Order Key",how="left",validate="one_to_one")
performance_base["Plan variants"]=performance_base["Order Key"].map(kpi_out.groupby("Order Key")["Planned Post Date"].nunique())
performance_base["Days late"]=(performance_base.released.dt.normalize()-performance_base.planned.dt.normalize()).dt.days
performance_base["Planned before created"]=performance_base.planned.dt.normalize().lt(performance_base.created.dt.normalize())
performance_base["Planned day"]=performance_base.planned.dt.strftime("%Y-%m-%d")
performance_base["Handover day"]=performance_base.released.dt.strftime("%Y-%m-%d")
PERFORMANCE_GROUPS={"DN Type":"DN_types","Country Code":"Country","Final Forwarder":"Final_forwarder", "Default Forwarder":"Default_forwarder","Planned day":"Planned day","Handover day":"Handover day"}

# Four explicit definitions evaluated on one consistent full-report outcome cohort.
LATE_DEFINITIONS={"A":"A — later calendar day","B":"B — 2+ calendar days","C":"C — after 18:00 on planned day","D":"D — after noon next day"}

def late_mask(frame,definition):
    plan=frame.planned.dt.normalize();actual=frame.released
    valid=plan.notna()&actual.notna()
    if definition=="A":flag=actual.dt.normalize().gt(plan)
    elif definition=="B":flag=(actual.dt.normalize()-plan).dt.days.ge(2)
    elif definition=="C":flag=actual.gt(plan+pd.Timedelta(hours=18))
    elif definition=="D":flag=actual.gt(plan+pd.Timedelta(days=1,hours=12))
    else:raise ValueError("Unknown late definition")
    return (valid&flag).fillna(False)

def lateness_population(start,end,flow=ALL_FILTER,forwarder=ALL_FILTER,forwarder_field="Final Forwarder"):
    first=pd.Timestamp(start).normalize();last=pd.Timestamp(end).normalize()+pd.Timedelta(days=1)
    if first>=last:raise ValueError("Planned from must be on or before Planned to.")
    table=performance_base.loc[performance_base.planned.ge(first)&performance_base.planned.lt(last)].copy()
    if flow!=ALL_FILTER:table=table.loc[table["Order Key"].isin(kpi_out.loc[source_filter(kpi_out["DN Type"],flow),"Order Key"])]
    if forwarder!=ALL_FILTER:table=table.loc[table["Order Key"].isin(kpi_out.loc[source_filter(kpi_out[forwarder_field],forwarder),"Order Key"])]
    measured=table.loc[table["Plan variants"].eq(1)&table.released.notna()].copy()
    return table,measured

def definition_comparison(measured):
    n=len(measured);records=[]
    for key,label in LATE_DEFINITIONS.items():
        flag=late_mask(measured,key)
        records.append({"Definition":label,"Measurable orders":n,"Late":int(flag.sum()),"Not late":int((~flag).sum()),
            "Late (%)":100*flag.sum()/n if n else float("nan"),"Median calendar days (late only)":measured.loc[flag,"Days late"].median()})
    return pd.DataFrame(records)

def performance_daily(all_rows,measured,definition):
    selected=all_rows.groupby("Planned day").size().rename("Selected planned orders")
    work=measured.assign(_late=late_mask(measured,definition))
    daily=work.groupby("Planned day").agg(Measurable=("Order Key","size"),Late=("_late","sum"))
    result=selected.to_frame().join(daily).fillna({"Measurable":0,"Late":0})
    result["Excluded"]=result["Selected planned orders"]-result.Measurable
    result["Not late"]=result.Measurable-result.Late
    result["Late (%)"]=100*result.Late/result.Measurable.where(result.Measurable.ne(0))
    return result.reset_index()

def lateness_groups(all_rows,measured,group_field,minimum,definition):
    key=PERFORMANCE_GROUPS[group_field];work=measured.assign(_late=late_mask(measured,definition))
    records=[];total_late=int(work._late.sum())
    for label,selected in all_rows.groupby(key,dropna=False):
        rows=work.loc[work[key].fillna("Missing / blank").eq(label if pd.notna(label) else "Missing / blank")]
        n=len(rows);late=rows.loc[rows._late,"Days late"];days=rows["Days late"]
        records.append({"Group":str(label),"Selected":len(selected),"Orders":n,"Excluded":len(selected)-n,"Late orders":len(late),
            "Late (%)":100*len(late)/n if n else float("nan"),"On-time/not-late (%)":100*(n-len(late))/n if n else float("nan"),
            "Median days (late only)":late.median(),"Average days (late only)":late.mean(),"2+ calendar days (%)":100*days.ge(2).sum()/n if n else float("nan"),
            "Share of all late (%)":100*len(late)/total_late if total_late else float("nan"),
            "Early/same calendar day":int(days.le(0).sum()),"1 day":int(days.eq(1).sum()),"2 days":int(days.eq(2).sum()),"3+ days":int(days.ge(3).sum()),
            "Sample note":"Below selected minimum" if n<minimum else "At/above selected minimum"})
    cols=["Group","Selected","Orders","Excluded","Late orders","Late (%)","On-time/not-late (%)","Median days (late only)","Average days (late only)","2+ calendar days (%)","Share of all late (%)","Early/same calendar day","1 day","2 days","3+ days","Sample note"]
    return pd.DataFrame(records,columns=cols).sort_values(["Late orders","Orders"],ascending=False)

def forwarder_country_table(measured,definition,field):
    key="Final_forwarder" if field=="Final Forwarder" else "Default_forwarder"
    work=measured.assign(_late=late_mask(measured,definition))
    result=work.groupby([key,"Country"],dropna=False).agg(Orders=("Order Key","size"),Late=("_late","sum")).reset_index()
    result["Late (%)"]=100*result.Late/result.Orders
    return result.rename(columns={key:"Forwarder","Country":"Country"})

planned_days=sorted(set(performance_base.planned.dropna().dt.date.loc[performance_base.planned.dropna().dt.date<=DATA_TODAY])|{DATA_TODAY})
performance_from=widgets.Dropdown(options=[(data_date_label(d),d) for d in planned_days],value=DATA_TODAY,description="Planned from:")
performance_to=widgets.Dropdown(options=[(data_date_label(d),d) for d in planned_days],value=DATA_TODAY,description="Planned to:")
performance_flow=filter_widget(kpi_out["DN Type"],"DN type:")
performance_forwarder_field=widgets.Dropdown(options=["Final Forwarder","Default Forwarder"],description="Forwarder field:")
performance_forwarder=filter_widget(kpi_out["Final Forwarder"],"Forwarder:")
performance_definition=widgets.Dropdown(options=[(label,key) for key,label in LATE_DEFINITIONS.items()],value="A",description="Released-after-plan rule:",layout=widgets.Layout(width="490px"))
performance_group_by=widgets.Dropdown(options=list(PERFORMANCE_GROUPS),value="Final Forwarder",description="Group by:")
performance_min_size=widgets.BoundedIntText(value=100,min=1,max=100000,description="Minimum group N:")
performance_hide_small=widgets.Checkbox(value=False,description="Hide groups below minimum N",indent=False)
performance_group=widgets.Dropdown(options=[("All groups",ALL_FILTER)],description="Inspect group:")
performance_band=widgets.Dropdown(options=["All late orders","Same-day late (C)","1 day late","2 days late","3+ days late"],description="Day band:")
performance_search=widgets.Text(description="Order contains:",continuous_update=False)
performance_group_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Group page:")
performance_order_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Order page:")
performance_day_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Day-table page:")
performance_chart_show=widgets.Checkbox(value=False,description="Show performance chart",indent=False)
performance_chart_kind=widgets.Dropdown(options=["Calendar-day distribution","Group severity bars","Forwarder × country matrix"],description="Chart:")
performance_matrix_min=widgets.BoundedIntText(value=20,min=1,max=100000,description="Matrix minimum n:")
performance_matrix_pick=widgets.Dropdown(options=[("All cells",ALL_FILTER)],value=ALL_FILTER,description="Inspect matrix cell:",layout=widgets.Layout(width="700px"))
performance_message=widgets.HTML();performance_output=widgets.Output();performance_busy=False
late_record_pick=widgets.Dropdown(options=[],description="Select order released after plan:",layout=widgets.Layout(width="650px"))
late_record_open=widgets.Button(description="Show order history");late_record_inspector=RecordInspector(scope="Full reports")

def performance_row(*items):return widgets.HBox(list(items),layout=widgets.Layout(flex_flow="row wrap",gap="8px"))
for control in [performance_from,performance_to,performance_forwarder_field,performance_definition,performance_group_by,performance_min_size,performance_group,
    performance_band,performance_search,performance_group_page,performance_order_page,performance_day_page,performance_chart_kind,performance_matrix_min,
    performance_matrix_pick,late_record_pick]:
    control.style.description_width="initial";control.layout.max_width="100%"

def render_performance_chart(measured,groups,matrix):
    if not performance_chart_show.value:return
    kind=performance_chart_kind.value
    if measured.empty:print("No measurable orders to chart.");return
    if kind=="Calendar-day distribution":
        counts=measured["Days late"].value_counts().sort_index()
        fig,ax=plt.subplots(figsize=(10,4));ax.bar(counts.index,counts.values,color="#176b87")
        ax.set_xlabel("Handover calendar day minus planned day (negative = early)");ax.set_ylabel("Orders");ax.set_title("Calendar-day difference — all measurable orders")
    elif kind=="Group severity bars":
        first=(performance_group_page.value-1)*preview_input.value;shown=groups.iloc[first:first+preview_input.value]
        if shown.empty:print("No groups on this page.");return
        fig,ax=plt.subplots(figsize=(10,max(3,len(shown)*.4)))
        shown.set_index("Group")[["Early/same calendar day","1 day","2 days","3+ days"]].plot.barh(stacked=True,ax=ax,color=["#9fc5ba","#80abc4","#d59c47","#b85650"])
        ax.set_xlabel("Orders, calendar-day bands (independent of late definition)");ax.set_ylabel("");ax.set_title("Severity — displayed group page")
    else:
        # Page matrix forwarders/countries to keep a large matrix readable.
        forwarders=sorted(matrix.Forwarder.astype(str).unique());countries=sorted(matrix.Country.astype(str).unique())
        fs=(performance_matrix_row_page.value-1)*10;cs=(performance_matrix_col_page.value-1)*8
        forwarders=forwarders[fs:fs+10];countries=countries[cs:cs+8]
        sub=matrix.loc[matrix.Forwarder.isin(forwarders)&matrix.Country.isin(countries)]
        if sub.empty:print("No matrix cells on this page.");return
        rates=sub.pivot(index="Forwarder",columns="Country",values="Late (%)").reindex(index=forwarders,columns=countries)
        ns=sub.pivot(index="Forwarder",columns="Country",values="Orders").reindex(index=forwarders,columns=countries)
        values=rates.where(ns.ge(performance_matrix_min.value))
        fig,ax=plt.subplots(figsize=(max(7,len(countries)*1.2),max(3,len(forwarders)*.6)))
        im=ax.imshow(values.to_numpy(dtype=float),vmin=0,vmax=100,cmap="YlOrRd",aspect="auto")
        ax.set_xticks(range(len(countries)),countries,rotation=35,ha="right");ax.set_yticks(range(len(forwarders)),forwarders)
        for r in range(len(forwarders)):
            for c in range(len(countries)):
                n=ns.iloc[r,c]
                if pd.notna(n):ax.text(c,r,(f"{rates.iloc[r,c]:.1f}%\nn={int(n)}" if n>=performance_matrix_min.value else f"n={int(n)}"),ha="center",va="center",fontsize=8)
        fig.colorbar(im,ax=ax,label="Late (%)");ax.set_title("Recorded forwarder × country performance; small cells show n only")
    ax.grid(False);display_operational_figure(fig)

performance_matrix_row_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Forwarder page:",style={"description_width":"initial"})
performance_matrix_col_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Country page:",style={"description_width":"initial"})

def refresh_performance(change=None):
    global performance_busy,selected_performance_orders,selected_performance_groups,selected_performance_population,selected_definition_comparison,selected_performance_daily,selected_performance_matrix
    if performance_busy:return
    performance_busy=True
    try:
        try:all_rows,measured=lateness_population(performance_from.value,performance_to.value,performance_flow.value,performance_forwarder.value,performance_forwarder_field.value)
        except ValueError as error:
            performance_message.value=str(error);performance_output.outputs=();evidence_options(late_record_pick,[]);late_record_inspector.clear();return
        performance_message.value="";definition=performance_definition.value
        selected_performance_population=measured
        selected_definition_comparison=definition_comparison(measured)
        selected_performance_daily=performance_daily(all_rows,measured,definition)
        groups=lateness_groups(all_rows,measured,performance_group_by.value,performance_min_size.value,definition)
        if performance_hide_small.value:groups=groups.loc[groups.Orders.ge(performance_min_size.value)]
        selected_performance_groups=groups
        previous=performance_group.value;labels=groups.Group.tolist()
        performance_group.options=[("All groups",ALL_FILTER)]+[(s,s) for s in labels];performance_group.value=previous if previous in labels else ALL_FILTER
        matrix=forwarder_country_table(measured,definition,performance_forwarder_field.value);selected_performance_matrix=matrix
        performance_matrix_row_page.max=max(1,(matrix.Forwarder.nunique()+9)//10);performance_matrix_col_page.max=max(1,(matrix.Country.nunique()+7)//8)
        fs=(performance_matrix_row_page.value-1)*10;cs=(performance_matrix_col_page.value-1)*8
        fpage=sorted(matrix.Forwarder.astype(str).unique())[fs:fs+10];cpage=sorted(matrix.Country.astype(str).unique())[cs:cs+8]
        previous=performance_matrix_pick.value
        cell_rows=matrix.loc[matrix.Forwarder.isin(fpage)&matrix.Country.isin(cpage)]
        performance_matrix_pick.options=[("All cells",ALL_FILTER)]+[(f"{r.Forwarder} / {r.Country} — n={r.Orders}",(str(r.Forwarder),str(r.Country))) for _,r in cell_rows.iterrows()]
        performance_matrix_pick.value=previous if previous in performance_matrix_pick._options_values else ALL_FILTER
        matrix_visible=performance_chart_show.value and performance_chart_kind.value=="Forwarder × country matrix"
        for control in [performance_matrix_pick,performance_matrix_min,performance_matrix_row_page,performance_matrix_col_page]:control.layout.display="" if matrix_visible else "none"
        if not matrix_visible:performance_matrix_pick.value=ALL_FILTER
        detail=measured.loc[late_mask(measured,definition)].copy()
        if performance_group.value!=ALL_FILTER:detail=detail.loc[detail[PERFORMANCE_GROUPS[performance_group_by.value]].astype(str).eq(performance_group.value)]
        if performance_matrix_pick.value!=ALL_FILTER:
            forwarder,country=performance_matrix_pick.value;fkey="Final_forwarder" if performance_forwarder_field.value=="Final Forwarder" else "Default_forwarder"
            detail=detail.loc[detail[fkey].eq(forwarder)&detail.Country.eq(country)]
        bands={"Same-day late (C)":detail["Days late"].eq(0),"1 day late":detail["Days late"].eq(1),"2 days late":detail["Days late"].eq(2),"3+ days late":detail["Days late"].ge(3)}
        if performance_band.value in bands:detail=detail.loc[bands[performance_band.value]]
        if performance_search.value.strip():
            query=performance_search.value.strip();detail=detail.loc[detail["Order Key"].astype("string").str.contains(query,case=False,regex=False,na=False)|detail.External_orders.astype("string").str.contains(query,case=False,regex=False,na=False)]
        detail=detail.sort_values(["Days late","Order Key"],ascending=[False,True]);selected_performance_orders=detail
        for control,table in [(performance_group_page,groups),(performance_order_page,detail),(performance_day_page,selected_performance_daily)]:
            control.max=max(1,(len(table)+preview_input.value-1)//preview_input.value);control.value=min(control.value,control.max)
        first=(performance_order_page.value-1)*preview_input.value;evidence_options(late_record_pick,[(str(k),str(k)) for k in detail["Order Key"].iloc[first:first+preview_input.value]])
        def render():
            flag=late_mask(measured,definition);n=len(measured);late=int(flag.sum());days=measured["Days late"]
            print(f"HANDOVER PERFORMANCE — planned {performance_from.value} to {performance_to.value}, inclusive; full-report outcomes; definition {definition}")
            display(pd.DataFrame({"Metric":["Selected planned orders","Excluded: conflicting plan / missing handover","Measurable completed orders","Late orders","Not-late orders (including early)","Late (%)","On-time/not-late (%)","Median calendar days (late only)","Early handover orders","Same planned-day handover orders","Planned before creation (kept, flagged)"],
                "Value":[len(all_rows),len(all_rows)-n,n,late,n-late,100*late/n if n else float("nan"),100*(n-late)/n if n else float("nan"),days.loc[flag].median(),int(days.lt(0).sum()),int(days.eq(0).sum()),int(measured["Planned before created"].sum())]}).round(2))
            print("ALL FOUR DEFINITIONS — identical measurable cohort; these are working alternatives, not confirmed service rules.");display(selected_definition_comparison.round(2))
            print("BY PLANNED DAY — selected = measurable + excluded; measurable = late + not late")
            first=(performance_day_page.value-1)*preview_input.value;display(selected_performance_daily.iloc[first:first+preview_input.value].round(2))
            print(f"GROUPS BY {performance_group_by.value}: {len(groups)} displayed; page {performance_group_page.value}/{performance_group_page.max}")
            first=(performance_group_page.value-1)*preview_input.value;display(groups.iloc[first:first+preview_input.value].round(2))
            render_performance_chart(measured,groups,matrix)
            print(f"LATE-ORDER EVIDENCE: {len(detail)} matches; page {performance_order_page.value}/{performance_order_page.max}. Evidence filters do not change headline totals.")
            first=(performance_order_page.value-1)*preview_input.value
            view=detail[["Order Key","External_orders","DN_types","Country","Final_forwarder","Default_forwarder","planned","released","Days late","Planned before created"]].rename(columns={"External_orders":"Customer orders [External Order Key]","DN_types":"Flow [DN Type]","Country":"Destination [Country Code]","Final_forwarder":"Final [Final Forwarder]","Default_forwarder":"Default [Default Forwarder]"})
            display(excel_labels(view.iloc[first:first+preview_input.value]))
            print("Recorded handover to forwarder, not customer delivery. Missing open orders and extract boundaries can bias these results.")
        replace_output(performance_output,render);late_record_inspector.clear()
    finally:performance_busy=False

def open_late_record(_):
    if late_record_pick.value is not None:late_record_inspector.open("Order",late_record_pick.value,full=True,origin="Full-report handover outcome evidence.")
def change_performance_forwarder(change=None):
    performance_forwarder.options=filter_options(kpi_out[performance_forwarder_field.value]);performance_forwarder.value=ALL_FILTER;refresh_performance()
late_record_open.on_click(open_late_record);performance_forwarder_field.observe(change_performance_forwarder,names="value")
for control in [performance_from,performance_to,performance_flow,performance_forwarder,performance_definition,performance_group_by,performance_min_size,performance_hide_small,
    performance_group,performance_band,performance_search,performance_group_page,performance_order_page,performance_day_page,performance_chart_show,performance_chart_kind,
    performance_matrix_min,performance_matrix_pick,performance_matrix_row_page,performance_matrix_col_page,preview_input]:
    control.observe(refresh_performance,names="value")
display(widgets.VBox([performance_row(performance_from,performance_to),performance_definition,performance_flow,performance_row(performance_forwarder_field,performance_forwarder),
    performance_group_by,performance_row(performance_min_size,performance_hide_small),performance_row(performance_group,performance_band,performance_search),
    performance_row(performance_day_page,performance_group_page,performance_order_page),performance_row(performance_chart_show,performance_chart_kind),
    performance_row(performance_matrix_min,performance_matrix_row_page,performance_matrix_col_page),performance_matrix_pick,
    performance_message,performance_output,performance_row(late_record_pick,late_record_open),late_record_inspector.widget]))
refresh_performance()

### 6.1 Exclusions and date-consistency investigation

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** DQ-01 "Planned before created" (181 orders) and DQ-02 "No planned date" (513 orders: 501 Milkrun to ABC + 12 Scrap).  

**Purpose:** Explain why some records cannot enter a reliable completed-order comparison, and show dates needing review.

Use this section's **own** As-of date/time, planned-date range, DN type and forwarder controls. These do not change Section 6 or other snapshot views. Only orders created by this cutoff are reviewed. Select Reason, browse pages and open an order for its line tables and history up to the same cutoff.

| Reason | Treatment |
|---|---|
| Missing planned date | Cannot be assigned to a planned window; shown from the filtered observed source with an unassigned scope label. |
| Conflicting planned dates | Affected orders with any source plan in the selected window are excluded from measurable outcomes. |
| No handover observed by selected time | No handover is known yet, even if a later one exists in the full extract. It may be open work or missing confirmation, not automatically a data error. |
| Planned before creation | Planned calendar day precedes creation day. Kept if otherwise measurable; available as a separate warning category. |

All excluded records does not include the kept-only warning. One order can have several reasons, so reason rows and unique-order counts differ. Missing-plan cases do not belong to the selected planned period. Missing orders cannot be enumerated. Static plan/flow/forwarder values reflect the extract, not historical changes; this review does not edit them.


In [19]:
# Independent planned cohort and historical cutoff for Section 6.1.
exclusion_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="As-of date:",style={"description_width":"initial"})
exclusion_time=widgets.Text(value="09:00:00",description="Time:",style={"description_width":"initial"},layout=widgets.Layout(width="220px"))
exclusion_apply=widgets.Button(description="Apply time")
exclusion_message=widgets.HTML();exclusion_selected_time=pd.Timestamp(AS_OF)
exclusion_from=widgets.Dropdown(options=[(data_date_label(d),d) for d in planned_days],value=DATA_TODAY,description="Planned from:",style={"description_width":"initial"})
exclusion_to=widgets.Dropdown(options=[(data_date_label(d),d) for d in planned_days],value=DATA_TODAY,description="Planned to:",style={"description_width":"initial"})
exclusion_flow=filter_widget(kpi_out["DN Type"],"DN type:")
exclusion_forwarder_field=widgets.Dropdown(options=["Final Forwarder","Default Forwarder"],value="Final Forwarder",description="Forwarder field:",style={"description_width":"initial"})
exclusion_forwarder=widgets.Dropdown(options=list(performance_forwarder.options),value=ALL_FILTER,description="Forwarder:",style={"description_width":"initial"})
exclusion_reason=widgets.Dropdown(options=["All excluded records","Missing planned date (unassigned window)","Conflicting planned dates","No handover observed by selected time","Planned before creation (kept, flagged)"],description="Reason:",style={"description_width":"initial"},layout=widgets.Layout(width="520px"))
exclusion_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Page:")
exclusion_pick=widgets.Dropdown(options=[],description="Inspect order:",style={"description_width":"initial"},layout=widgets.Layout(width="700px"))
exclusion_open=widgets.Button(description="Open selected evidence",layout=widgets.Layout(width="200px"))
exclusion_output=widgets.Output();exclusion_inspector=RecordInspector();exclusion_busy=False

def performance_exclusions(start,end,flow=ALL_FILTER,forwarder=ALL_FILTER,field="Final Forwarder",at=None):
    t=pd.Timestamp(exclusion_selected_time if at is None else at)
    first=pd.Timestamp(start).normalize();last=pd.Timestamp(end).normalize()+pd.Timedelta(days=1)
    if first>=last:raise ValueError("Planned from must be on or before Planned to.")
    window_ids=set(kpi_out.loc[kpi_out["Planned Post Date"].ge(first)&kpi_out["Planned Post Date"].lt(last),"Order Key"])
    # Only orders with creation observed by the selected cutoff enter the review.
    base=performance_base.loc[performance_base.created.le(t)].copy()
    if flow!=ALL_FILTER:base=base.loc[base["Order Key"].isin(set(kpi_out.loc[source_filter(kpi_out["DN Type"],flow),"Order Key"]))]
    if forwarder!=ALL_FILTER:base=base.loc[base["Order Key"].isin(set(kpi_out.loc[source_filter(kpi_out[field],forwarder),"Order Key"]))]
    records=[]
    for _,row in base.iterrows():
        reasons=[]
        if pd.isna(row.planned):reasons.append(("Missing planned date (unassigned window)","No due date recorded."))
        if row["Order Key"] in window_ids:
            if row["Plan variants"]>1:reasons.append(("Conflicting planned dates","Different planned dates exist on the order's source rows."))
            if pd.isna(row.released) or row.released>t:
                reasons.append(("No handover observed by selected time","No handover observed by this cutoff; the order may still be open or confirmation may be missing."))
        for reason,note in reasons:records.append({"Order Key":row["Order Key"],"Reason":reason,"Scope":"Unassigned date: full filtered observed source" if reason.startswith("Missing planned") else "Selected planned-date window","Policy":"Excluded from measurable outcomes at cutoff","Explanation":note})
        if row["Order Key"] in window_ids and row["Planned before created"]:
            records.append({"Order Key":row["Order Key"],"Reason":"Planned before creation (kept, flagged)","Scope":"Selected planned-date window","Policy":"Kept if otherwise measurable at cutoff","Explanation":"Planned day precedes creation; investigate without automatically excluding."})
    return pd.DataFrame(records,columns=["Order Key","Reason","Scope","Policy","Explanation"])

def refresh_exclusions(change=None):
    global exclusion_busy,selected_exclusions
    if exclusion_busy:return
    exclusion_busy=True
    try:
        try:table=performance_exclusions(exclusion_from.value,exclusion_to.value,exclusion_flow.value,exclusion_forwarder.value,exclusion_forwarder_field.value,exclusion_selected_time)
        except ValueError as error:
            selected_exclusions=pd.DataFrame();evidence_options(exclusion_pick,[]);exclusion_inspector.clear()
            replace_output(exclusion_output,lambda:print(str(error)));return
        if exclusion_reason.value=="All excluded records":table=table.loc[table.Policy.eq("Excluded from measurable outcomes at cutoff")]
        else:table=table.loc[table.Reason.eq(exclusion_reason.value)]
        selected_exclusions=table
        exclusion_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value);exclusion_page.value=min(exclusion_page.value,exclusion_page.max)
        first=(exclusion_page.value-1)*preview_input.value;shown=table.iloc[first:first+preview_input.value]
        evidence_options(exclusion_pick,[(f"Order {row['Order Key']} — {row.Reason}",row["Order Key"]) for _,row in shown.iterrows()])
        def render():
            print(f"EXCLUSIONS / FLAGS AS OF {exclusion_selected_time}; planned {exclusion_from.value} to {exclusion_to.value}, inclusive")
            print(f"{len(table):,} reason rows for {table['Order Key'].nunique():,} orders; page {exclusion_page.value}/{exclusion_page.max}")
            print("Independent of Section 6. Only orders created by the cutoff are reviewed. Missing-plan cases span the filtered observed source, not the selected planned period.")
            print("Planned dates and classifications reflect the extract; historical changes to those fields cannot be reconstructed. No handover observed does not prove a missing confirmation.")
            display(shown)
        replace_output(exclusion_output,render)
        exclusion_inspector.clear()
    finally:exclusion_busy=False

def apply_exclusion_time(change=None):
    global exclusion_selected_time,exclusion_busy
    try:candidate=pd.Timestamp.combine(exclusion_day.value,parse_clock_time(exclusion_time.value))
    except (ValueError,TypeError):
        exclusion_message.value="Enter time as HH:MM or HH:MM:SS. Results have not changed.";return
    exclusion_selected_time=candidate;exclusion_message.value=""
    exclusion_busy=True;exclusion_page.value=1;exclusion_busy=False
    refresh_exclusions()

def open_exclusion_evidence(_):
    if exclusion_pick.value is not None:exclusion_inspector.open("Order",exclusion_pick.value,exclusion_selected_time,origin="Section 6.1 independent historical review.")

def update_exclusion_forwarder(change=None):
    options=filter_widget(kpi_out[exclusion_forwarder_field.value],"Forwarder:").options
    exclusion_forwarder.options=options;exclusion_forwarder.value=ALL_FILTER
    refresh_exclusions()

exclusion_open.on_click(open_exclusion_evidence)
exclusion_day.observe(apply_exclusion_time,names="value");exclusion_apply.on_click(apply_exclusion_time)
exclusion_forwarder_field.observe(update_exclusion_forwarder,names="value")
for control in [exclusion_reason,exclusion_page,exclusion_from,exclusion_to,exclusion_flow,exclusion_forwarder,preview_input]:control.observe(refresh_exclusions,names="value")
display(widgets.VBox([widgets.HBox([exclusion_day,exclusion_time,exclusion_apply]),exclusion_message,
    widgets.HBox([exclusion_from,exclusion_to]),exclusion_flow,widgets.HBox([exclusion_forwarder_field,exclusion_forwarder]),
    widgets.HBox([exclusion_reason,exclusion_page]),exclusion_output,exclusion_pick,exclusion_open,exclusion_inspector.widget]))
refresh_exclusions()


## 7. Data quality and record-pattern checks

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** Data checks screen: DQ-06 identical rows (label, never delete) and DQ-07 missing fields. Forwarder change is information only.  
> 🟥 **Changed in this version:** Expected quantity is counted once per 3-part inbound line.  

**Purpose:** Find missing fields, repetition patterns and code differences that merit investigation. Select a check to see summary counts and paged source-row evidence. Nothing is automatically deduplicated or repaired.

**Time scope:** Observed by selected time requires a dock/receipt event for inbound rows, or creation/scan/packing/handover for outbound rows, by this section's independent cutoff. Future times and received/shipped quantities are masked in evidence. Rows without any usable event timestamp are reported as undated; Full extract includes them and ignores the cutoff. Static metadata remains the extract's values.

| Check | Interpretation |
|---|---|
| Repeated order keys | Additional rows beyond the first per order. Multiple item/carton rows are normally legitimate; this is not an error count. |
| Exact duplicate rows | All retained source fields match. Participating rows and additional copies are different counts. Equality suggests review, not proof of a duplicate physical movement. |
| Repeated order/line/carton/SKU/lot | A business-key repetition pattern that can reflect splits or repeated postings/exports. All rows remain visible. |
| Source-row receipt variance | A row differs from the line's repeated expected quantity. Splits can be legitimate; Section 2.2 aggregates the line before judging shortage/excess. |
| Blank location / LPN | Missing or whitespace values. Missing LPNs with receipt observed are separated from records without an observed receipt. |
| Inbound by shipment type | Known expected/received totals, blank-LPN rows/rates and unresolved expected lines. ~~Expectations count once per identified ASN line;~~ **→ Expectations count once per identified ASN + Ref Line# + Functional Packing line;** missing IDs are counted separately. Multiple types on a line are not arbitrarily assigned to one type. |
| Final/default forwarder | Compare trimmed, case-normalized codes only when both exist. Different-code rows, their comparable-row rate and affected orders are reported; missing codes are separate. A difference can be legitimate routing or code mapping. |

Use the ID/SKU search on source-row views, then Page. The shipment-type view is grouped and does not use that search. Summary counts retain the full chosen time scope before evidence search/view selection. Forwarder percentage here is **source-row-level**, not a percentage of unique orders. Known quantity totals can exclude unresolved lines; unavailable values must not be read as zero. These checks identify patterns, not their cause or a verified repair.


In [20]:
# Shared data-quality definitions. Prepared copies keep original source fields.
import numpy as np

def dq_blank(values):
    return values.isna()|values.astype("string").str.strip().eq("").fillna(True)

def dq_rows(at,scope):
    if scope=="Full extract":return record_ib.copy(),record_ob.copy()
    t=pd.Timestamp(at)
    incoming=record_ib.loc[record_ib[["Dock in Date & Time","GR Date & Time"]].le(t).any(axis=1)].copy()
    outgoing=record_ob.loc[record_ob[[created_col,scan_col,packed_col,released_col]].le(t).any(axis=1)].copy()
    return incoming,outgoing

def dq_expected_lines(incoming):
    identified=incoming.loc[~dq_blank(incoming._ASN)&~dq_blank(incoming._Line)].copy()
    # REVIEW 16 Nov - CHANGED: output carries Packing (Functional Packing) as part of the line key (DSV A5).
    # [OLD] if identified.empty:return pd.DataFrame(columns=["ASN","Line","Shipment type","Expected","Received","Expected unresolved","Source rows","Blank LPN rows"])
    if identified.empty:return pd.DataFrame(columns=["ASN","Line","Packing","Shipment type","Expected","Received","Expected unresolved","Source rows","Blank LPN rows"])
    identified["_exp"]=pd.to_numeric(identified["Qty Exp"],errors="coerce")
    identified["_rcv"]=pd.to_numeric(identified["Qty Rcvd"],errors="coerce")
    identified["_bad_exp"]=~np.isfinite(identified._exp)
    identified["_bad_rcv"]=~np.isfinite(identified._rcv)
    identified["_blank_lpn"]=dq_blank(identified.LPN)
    identified["_type"]=identified["Shipment Type"].astype("string").str.strip().replace("",pd.NA)
    # REVIEW 16 Nov - CHANGED: group by the 3-part line key. Why: the 2-part key counts a second pack type's expected quantity as missing.
    # [OLD] grouped=identified.groupby(["_ASN","_Line"],dropna=False)
    grouped=identified.groupby(["_ASN","_Line","_FP"],dropna=False)
    result=grouped.agg(Expected=("_exp","first"),Received=("_rcv","sum"),exp_variants=("_exp","nunique"),bad_exp=("_bad_exp","max"),
        bad_rcv=("_bad_rcv","max"),type_first=("_type","first"),type_variants=("_type","nunique"),source_rows=("_Line","size"),blank_lpn=("_blank_lpn","sum")).reset_index()
    result["Expected unresolved"]=result.bad_exp|result.exp_variants.ne(1)
    result["Expected"]=result.Expected.where(~result["Expected unresolved"])
    result["Received"]=result.Received.where(~result.bad_rcv)
    result["Shipment type"]=result.type_first.where(result.type_variants.eq(1),"Multiple types").where(result.type_variants.gt(0),"Missing type")
    # REVIEW 16 Nov - CHANGED: keep Packing in the returned table.
    # [OLD] return result.rename(columns={"_ASN":"ASN","_Line":"Line","source_rows":"Source rows","blank_lpn":"Blank LPN rows"})[["ASN","Line","Shipment type","Expected","Received","Expected unresolved","Source rows","Blank LPN rows"]]
    return result.rename(columns={"_ASN":"ASN","_Line":"Line","_FP":"Packing","source_rows":"Source rows","blank_lpn":"Blank LPN rows"})[["ASN","Line","Packing","Shipment type","Expected","Received","Expected unresolved","Source rows","Blank LPN rows"]]

def dq_type_summary(incoming):
    lines=dq_expected_lines(incoming)
    if lines.empty:return pd.DataFrame()
    grouped=lines.groupby("Shipment type",dropna=False)
    result=grouped.agg(ASNs=("ASN","nunique"),Lines=("Line","size"),Source_rows=("Source rows","sum"),Blank_LPN_rows=("Blank LPN rows","sum"),
        Expected=("Expected",lambda v:v.sum(min_count=1)),Received=("Received",lambda v:v.sum(min_count=1)),Unresolved_expected_lines=("Expected unresolved","sum"))
    result["Blank LPN rate (%)"]=100*result.Blank_LPN_rows/result.Source_rows
    return result.rename(columns={"Source_rows":"Identified source rows","Blank_LPN_rows":"Blank LPN rows","Expected":"Known expected [Qty Exp]","Received":"Known received [Qty Rcvd]","Unresolved_expected_lines":"Missing/conflicting expected lines"})

def dq_snapshot(at,scope):
    incoming,outgoing=dq_rows(at,scope)
    incols=[c for c in ib.columns if c in incoming];outcols=[c for c in ob.columns if c in outgoing]
    indup=incoming.duplicated(subset=incols,keep=False);outdup=outgoing.duplicated(subset=outcols,keep=False)
    inextra=int(incoming.duplicated(subset=incols).sum());outextra=int(outgoing.duplicated(subset=outcols).sum())
    repeat_order=outgoing._Order.ne("")&outgoing.duplicated("_Order",keep=False)
    key=["_Order","_Line","_HU","_SKU","_Lot"]
    key_known=outgoing[key].ne("").all(axis=1)
    repeat_key=key_known&outgoing.duplicated(key,keep=False)
    blankloc=dq_blank(outgoing["Loc"]);blanklpn=dq_blank(incoming["LPN"])
    receipt_observed=incoming["GR Date & Time"].notna() if scope=="Full extract" else incoming["GR Date & Time"].le(at)
    raw_expected=pd.to_numeric(incoming["Qty Exp"],errors="coerce");raw_received=pd.to_numeric(incoming["Qty Rcvd"],errors="coerce")
    raw_variance=receipt_observed&np.isfinite(raw_expected)&np.isfinite(raw_received)&raw_expected.ne(raw_received)
    final=outgoing["Final Forwarder"].astype("string").str.strip().str.upper()
    default=outgoing["Default Forwarder"].astype("string").str.strip().str.upper()
    missing=dq_blank(final)|dq_blank(default);both=~missing;different=both&final.ne(default).fillna(False)
    rate=100*int(different.sum())/int(both.sum()) if both.sum() else np.nan
    extra_keys=int(outgoing.loc[outgoing._Order.ne("")].duplicated("_Order").sum())
    summary=pd.DataFrame({"Metric":["Inbound rows reviewed","Outbound rows reviewed","Additional rows with repeated order key (normally legitimate)",
        "Inbound rows in exact duplicate groups","Inbound additional exact copies","Outbound rows in exact duplicate groups","Outbound additional exact copies",
        "Outbound rows in repeated business-key groups","Blank outbound-location rows","Blank inbound-LPN rows","Blank LPN with receipt observed",
        "Blank LPN without receipt observed","Comparable forwarder-code rows","Different final/default rows","Different / comparable forwarder rows (%)",
        "Orders with a forwarder difference","Rows with either forwarder missing","Inbound rows missing ASN/line ID"],
        "Value":[len(incoming),len(outgoing),extra_keys,int(indup.sum()),inextra,int(outdup.sum()),outextra,int(repeat_key.sum()),int(blankloc.sum()),
            int(blanklpn.sum()),int((blanklpn&receipt_observed).sum()),int((blanklpn&~receipt_observed).sum()),int(both.sum()),int(different.sum()),rate,
            outgoing.loc[different,"_Order"].replace("",pd.NA).nunique(),int(missing.sum()),int((dq_blank(incoming._ASN)|dq_blank(incoming._Line)).sum())]})
    extra_rates=pd.DataFrame({"Metric":["Blank outbound-location rows (%)","Blank inbound-LPN rows (%)","Inbound source rows in duplicate groups (%)","Outbound source rows in duplicate groups (%)"],
        "Value":[100*blankloc.sum()/len(outgoing) if len(outgoing) else np.nan,100*blanklpn.sum()/len(incoming) if len(incoming) else np.nan,
            100*indup.sum()/len(incoming) if len(incoming) else np.nan,100*outdup.sum()/len(outgoing) if len(outgoing) else np.nan]})
    summary=pd.concat([summary,extra_rates],ignore_index=True)
    summary.loc[len(summary)]=["Inbound source-row quantity differences (split receipts may be legitimate)",int(raw_variance.sum())]
    checks={"Repeated order-key rows (structural)":("Outbound",outgoing.loc[repeat_order]),
        "Exact duplicate inbound rows":("Inbound",incoming.loc[indup]),"Exact duplicate outbound rows":("Outbound",outgoing.loc[outdup]),
        "Repeated order/line/carton/SKU/lot":("Outbound",outgoing.loc[repeat_key]),"Blank outbound location":("Outbound",outgoing.loc[blankloc]),
        "Blank LPN — receipt observed":("Inbound",incoming.loc[blanklpn&receipt_observed]),
        "Blank LPN — receipt not observed":("Inbound",incoming.loc[blanklpn&~receipt_observed]),
        "Source-row receipt variance (structural)":("Inbound",incoming.loc[raw_variance]),
        "Different final/default forwarder":("Outbound",outgoing.loc[different]),"Missing forwarder code":("Outbound",outgoing.loc[missing])}
    return summary,checks,incoming,outgoing

dq_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="As-of date:",style={"description_width":"initial"})
dq_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"))
dq_apply=widgets.Button(description="Apply time");dq_message=widgets.HTML();dq_selected_time=pd.Timestamp(AS_OF)
dq_scope=widgets.Dropdown(options=["Observed by selected time","Full extract"],description="Scope:",layout=widgets.Layout(width="390px"))
dq_view=widgets.Dropdown(options=["Inbound by shipment type","Repeated order-key rows (structural)","Exact duplicate inbound rows","Exact duplicate outbound rows","Repeated order/line/carton/SKU/lot",
    "Blank outbound location","Blank LPN — receipt observed","Blank LPN — receipt not observed","Source-row receipt variance (structural)","Different final/default forwarder","Missing forwarder code"],description="Inspect:",layout=widgets.Layout(width="500px"))
dq_search=widgets.Text(description="ID / SKU contains:",style={"description_width":"initial"},continuous_update=False)
dq_hint=widgets.HTML("Example: an ASN, order key, carton/LPN or SKU. Leave blank for all records.")
dq_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Page:");dq_output=widgets.Output();dq_busy=False;dq_cached=None;dq_cache_key=None

def refresh_dq(change=None):
    global dq_busy,dq_cached,dq_cache_key,selected_dq_rows,selected_dq_summary
    if dq_busy:return
    dq_busy=True
    try:
        key=(dq_selected_time,dq_scope.value)
        if dq_cache_key!=key:dq_cached=dq_snapshot(*key);dq_cache_key=key
        summary,checks,incoming,outgoing=dq_cached;selected_dq_summary=summary
        if change is not None and change.get("owner") is not dq_page:dq_page.value=1
        side=None
        if dq_view.value=="Inbound by shipment type":
            type_rows=incoming.copy()
            if dq_scope.value!="Full extract":type_rows["Qty Rcvd"]=type_rows["Qty Rcvd"].where(type_rows["GR Date & Time"].le(dq_selected_time))
            table=dq_type_summary(type_rows).reset_index()
        else:
            side,table=checks[dq_view.value];table=table.copy()
            if dq_search.value.strip():
                cols=[c for c in ["_ASN","_Order","_HU","_SKU"] if c in table]
                table=table.loc[table[cols].astype("string").apply(lambda s:s.str.contains(dq_search.value.strip(),case=False,regex=False,na=False)).any(axis=1)]
        selected_dq_rows=table
        dq_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value);dq_page.value=min(dq_page.value,dq_page.max)
        def render():
            print("DATA QUALITY — "+("full extract" if dq_scope.value=="Full extract" else "as of "+str(dq_selected_time)))
            display(summary.round(2))
            if dq_scope.value!="Full extract":
                undated_in=~record_ib[["Dock in Date & Time","GR Date & Time"]].notna().any(axis=1)
                undated_out=~record_ob[[created_col,scan_col,packed_col,released_col]].notna().any(axis=1)
                print(f"Undated rows outside this snapshot: inbound {int(undated_in.sum()):,}; outbound {int(undated_out.sum()):,}. Use Full extract to review them.")
            print(f"{dq_view.value}: {len(table):,} rows/groups; page {dq_page.value}/{dq_page.max}")
            if side is None:print("Known quantity totals exclude unresolved lines; missing ASN/line rows are counted separately above. ID search applies to source-row views only.")
            elif dq_view.value=="Repeated order-key rows (structural)":print("Multiple rows per order are usually legitimate lines/cartons. Participating rows below differ from additional-row count above.")
            elif dq_view.value=="Source-row receipt variance (structural)":print("A split receipt row may differ from the ASN line's repeated expected quantity. Use Section 2's aggregated line status to judge a real receipt exception.")
            first=(dq_page.value-1)*preview_input.value;shown=table.iloc[first:first+preview_input.value]
            if shown.empty:print("No matching rows.")
            elif side is None:display(shown.round(2))
            else:
                view=source_records_for_display(shown,side,None if dq_scope.value=="Full extract" else dq_selected_time)
                cols=(["WMS ASN#","Ref Line#","Shipment Type","SKU","Vendor Lot","Qty Exp","Qty Rcvd","LPN","Docked (+8h) [Dock in Date & Time]","Received (+8h) [GR Date & Time]"] if side=="Inbound" else
                    ["Order Key","Order Line Number","DN Type","SKU","Vendor Lot","Carton ID","Qty Shipped","Loc","Default Forwarder","Final Forwarder",created_col,released_col])
                view=view[[c for c in cols if c in view]].copy();view.insert(0,"Source row",shown.index)
                display(view)
        replace_output(dq_output,render)
    finally:dq_busy=False

def apply_dq_time(change=None):
    global dq_selected_time
    try:candidate=pd.Timestamp.combine(dq_day.value,parse_clock_time(dq_time.value))
    except (ValueError,TypeError):dq_message.value="Enter time as HH:MM or HH:MM:SS. Results unchanged.";return
    dq_selected_time=candidate;dq_message.value="";refresh_dq()
dq_day.observe(apply_dq_time,names="value");dq_apply.on_click(apply_dq_time)
for control in [dq_scope,dq_view,dq_search,dq_page,preview_input]:control.observe(refresh_dq,names="value")
display(widgets.VBox([widgets.HBox([dq_day,dq_time,dq_apply]),dq_message,dq_scope,dq_view,dq_search,dq_hint,dq_page,dq_output]))
refresh_dq()


### 7.1 Milk run quantity reconciliation by SKU and lot (Drop?)

> **Review 16 Nov 2026 (prototype handover)**  
> 🟨 **Not in the prototype yet:** Prototype rule DQ-04 "Contents mismatch" is marked "Not built yet". This section is the logic it would use: 99.1% of fully linked Milkrun orders reconcile (3,214 of 3,242). Decide at the handover: build DQ-04 from this, or drop 7.1.  

**Purpose:** Test whether linked records agree on contents, beyond sharing an ID. Scope is inbound Milkrun to ABC and outbound Milkrun from ABC.

Use the independent as-of date/time and inclusive order-creation range. Defaults select Dataset Today at 09:00:00; widen Order from/to for historical work. The cutoff limits observed receipts and handovers. Filter Result or order/ASN text, page through orders and select one to inspect its SKU/lot comparison and paged supporting rows.

**Calculation:** Identify each order's recorded cartons, then exact observed inbound LPN counterparts. Count each inbound source row once and sum Qty Rcvd by SKU + vendor lot. For outbound, treat Qty Shipped as a repeated order-line total: count once per order/line only if SKU, lot and quantity are consistent. Conflicting line facts are unassessable. This is an explicit grain assumption, not independently verified physical allocation. No row-to-row many-to-many join multiplies quantities, and original evidence remains intact.

Compare the union of SKU/lot keys across both complete sides. A key absent from a complete side contributes zero there; an existing row with missing quantity is unavailable. Differences beyond floating-point noise of 1e-9 create a mismatch. Equal grand totals cannot hide different contents.

| Result | Meaning |
|---|---|
| Reconciled | Every assessable SKU/lot total agrees. |
| Mismatch | At least one assessable SKU/lot differs. |
| Partially linked | Missing carton ID or no observed inbound counterpart for every carton. Not automatically a quantity mismatch. |
| Cannot assess | Links, chronology or required SKU/lot/quantities do not support a reliable complete comparison. This includes shared order/ASN associations, incomplete receipts/handovers and missing, negative or non-finite quantities. |

Reconciled percentage = reconciled / (reconciled + mismatched). Partial and unassessable orders are counted separately and excluded from that denominator. Exactly-double inbound means inbound = 2 × outbound with outbound > 0 for a SKU/lot; it is a possible pattern, not proof of duplicate posting. This differs from receiving's comparison of received to expected.

Duplicate source patterns are retained and counted; inbound reported totals still include those rows, while consistent outbound lines count once. Investigate before interpreting a difference as physical shortage/excess. ~~Section 2.1 and general inspectors use reported-row shipping sums, so their quantities must not be directly compared with this line-grain measure.~~ **→ Section 2.1 and the record inspector now use the same once-per-order-line grain, so they can be compared.** Static assignments and extract omissions also limit physical traceability; no stock is allocated or repaired.


In [21]:
def reconcile_order_contents(incoming,outgoing):
    """Compare source-row totals without a row-to-row join. Caller checks link eligibility."""
    keys=["_SKU","_Lot"]
    for frame,col in [(incoming,"Qty Rcvd"),(outgoing,"Qty Shipped")]:
        qty=pd.to_numeric(frame[col],errors="coerce")
        if frame[keys].eq("").any().any() or qty.isna().any() or not np.isfinite(qty).all() or qty.lt(0).any():
            return pd.DataFrame(),"Missing SKU/lot or missing, negative or invalid quantity"
    left=incoming.assign(_qty=pd.to_numeric(incoming["Qty Rcvd"])).groupby(keys)._qty.sum().rename("Inbound received [Qty Rcvd]")
    if outgoing._Line.eq("").any():return pd.DataFrame(),"Missing outbound order-line identifier"
    line_groups=outgoing.groupby(["_Order","_Line"],dropna=False)
    if (line_groups._SKU.nunique().ne(1).any() or line_groups._Lot.nunique().ne(1).any()
        or line_groups["Qty Shipped"].nunique().ne(1).any()):
        return pd.DataFrame(),"Conflicting SKU, lot or shipped quantity within an outbound order line"
    line_quantities=outgoing.drop_duplicates(["_Order","_Line"])
    right=line_quantities.assign(_qty=pd.to_numeric(line_quantities["Qty Shipped"])).groupby(keys)._qty.sum().rename("Outbound shipped [Qty Shipped]")
    comparison=pd.concat([left,right],axis=1).fillna(0).reset_index().rename(columns={"_SKU":"SKU","_Lot":"Vendor Lot"})
    a=comparison["Inbound received [Qty Rcvd]"];b=comparison["Outbound shipped [Qty Shipped]"]
    comparison["Difference (inbound - outbound)"]=a-b
    comparison["Result"]=np.where(np.isclose(a,b,rtol=0,atol=1e-9),"Matches","Different")
    comparison["Exactly double inbound"]=b.gt(0)&np.isclose(a,2*b,rtol=0,atol=1e-9)
    return comparison,""

recon_all_in=record_ib.loc[kpi_in.is_milkrun.to_numpy()].copy()
recon_all_out=record_ob.loc[trace_outbound.to_numpy()].copy()
recon_in_groups={key:group for key,group in recon_all_in.loc[recon_all_in._HU.ne("")].groupby("_HU")}
recon_out_groups={key:group for key,group in recon_all_out.loc[recon_all_out._Order.ne("")].groupby("_Order")}
recon_created=recon_all_out.loc[recon_all_out._Order.ne("")].groupby("_Order")[created_col].min()
recon_in_duplicate=recon_all_in.duplicated(subset=list(ib.columns),keep=False)
recon_out_duplicate=recon_all_out.duplicated(subset=list(ob.columns),keep=False)

def milk_reconciliation(at,start,end):
    t=pd.Timestamp(at);first=pd.Timestamp(start);last=pd.Timestamp(end)+pd.Timedelta(days=1)
    if first>=last:raise ValueError("Order from must be on or before Order to.")
    selected_ids=recon_created.loc[recon_created.ge(first)&recon_created.lt(last)&recon_created.le(t)].index
    # Ownership is checked across all observed flows/orders, not just the selected cohort.
    owners=record_ob.loc[record_ob[created_col].le(t)&record_ob._HU.ne("")].groupby("_HU")._Order.agg(lambda v:set(v)-{""})
    records=[];details={}
    for order in selected_ids:
        outgoing=recon_out_groups[order];units=set(outgoing._HU)-{""}
        found=[];missing=[];incomplete_receipt=False
        for unit in sorted(units):
            rows=recon_in_groups.get(unit)
            observed=None if rows is None else rows.loc[rows["GR Date & Time"].le(t)]
            if observed is None or observed.empty:missing.append(unit)
            else:
                found.append(observed)
                if len(observed)!=len(rows):incomplete_receipt=True
        incoming=pd.concat(found) if found else recon_all_in.iloc[:0].copy()
        # A source row can belong to only one original LPN, so concat above cannot multiply it.
        assert incoming.index.is_unique
        reasons=[];partial=bool(missing) or outgoing._HU.eq("").any() or not units
        if partial:reasons.append("Missing carton ID or no observed inbound counterpart for every carton")
        if any(len(owners.get(unit,set()))!=1 for unit in units):reasons.append("Carton associated with multiple/unknown observed orders")
        if not incoming.empty and (incoming._ASN.eq("").any() or incoming.groupby("_HU")._ASN.nunique().ne(1).any()):reasons.append("LPN associated with multiple/unknown ASNs")
        if incomplete_receipt:reasons.append("Receipt not observed for all matching inbound source rows")
        if outgoing[created_col].isna().any() or outgoing[created_col].gt(t).any():reasons.append("Some order rows not yet created or lack a creation time")
        if outgoing[released_col].isna().any() or outgoing[released_col].gt(t).any():reasons.append("Handover not observed for all outbound rows")
        comparison=pd.DataFrame()
        if not reasons:
            comparison,error=reconcile_order_contents(incoming,outgoing)
            if error:reasons.append(error)
        status="Partially linked" if partial else "Cannot assess" if reasons else "Reconciled" if comparison.Result.eq("Matches").all() else "Mismatch"
        duplicate_in=int(recon_in_duplicate.reindex(incoming.index).fillna(False).sum());duplicate_out=int(recon_out_duplicate.reindex(outgoing.index).fillna(False).sum())
        records.append({"Order [Order Key]":order,"Created ["+created_col+"]":recon_created.loc[order],"Status":status,
            "Cartons":len(units),"Observed linked cartons":len(units)-len(missing),"ASNs [WMS ASN#]":join_ids(incoming._ASN),
            "Different SKU/lot pairs":int(comparison.Result.eq("Different").sum()) if not comparison.empty else pd.NA,
            "Exactly-double SKU/lot pairs":int(comparison["Exactly double inbound"].sum()) if not comparison.empty else pd.NA,
            "Duplicate-pattern inbound rows":duplicate_in,"Duplicate-pattern outbound rows":duplicate_out,
            "Reason":"; ".join(reasons) if reasons else "Reported quantities agree by SKU/lot" if status=="Reconciled" else "Reported quantities differ by SKU/lot"})
        details[order]=(incoming,outgoing,comparison)
    columns=["Order [Order Key]","Created ["+created_col+"]","Status","Cartons","Observed linked cartons","ASNs [WMS ASN#]","Different SKU/lot pairs","Exactly-double SKU/lot pairs","Duplicate-pattern inbound rows","Duplicate-pattern outbound rows","Reason"]
    return pd.DataFrame(records,columns=columns),details

recon_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="As-of date:",style={"description_width":"initial"})
recon_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"))
recon_apply=widgets.Button(description="Apply time");recon_message=widgets.HTML();recon_selected_time=pd.Timestamp(AS_OF)
recon_days=sorted(set(recon_created.dropna().dt.date)|{DATA_TODAY})
recon_from=widgets.Dropdown(options=[(data_date_label(d),d) for d in recon_days],value=DATA_TODAY,description="Order from:",style={"description_width":"initial"})
recon_to=widgets.Dropdown(options=[(data_date_label(d),d) for d in recon_days],value=DATA_TODAY,description="Order to:",style={"description_width":"initial"})
recon_status=widgets.Dropdown(options=["All orders","Mismatch","Reconciled","Partially linked","Cannot assess","Exactly-double pattern"],description="Result:",layout=widgets.Layout(width="350px"))
recon_search=widgets.Text(description="Order / ASN contains:",style={"description_width":"initial"},continuous_update=False)
recon_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Order page:",style={"description_width":"initial"})
recon_pick=widgets.Dropdown(options=[],description="Select order:",style={"description_width":"initial"},layout=widgets.Layout(width="700px"))
recon_detail_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Detail page:",style={"description_width":"initial"})
recon_output=widgets.Output();recon_detail_output=widgets.Output();recon_busy=False;recon_cache_key=None;recon_cache=None

def show_recon_detail(change=None):
    if recon_busy:return
    def render():
        if recon_pick.value is None:print("No order selected. Widen the order-date range or change Result.");return
        order=recon_pick.value;incoming,outgoing,comparison=recon_cache[1][order]
        row=recon_cache[0].loc[recon_cache[0]["Order [Order Key]"].eq(order)].iloc[0]
        print(f"ORDER {order}: {row.Status}. {row.Reason}")
        print("Inbound receipts are summed once per source row; outbound quantity is counted once per consistent order line. All raw evidence is retained.")
        if row["Duplicate-pattern inbound rows"] or row["Duplicate-pattern outbound rows"]:print("Source duplicate patterns present. Inbound reported totals retain them; outbound line totals count a consistent line once. Investigate before inferring a physical shortage/excess.")
        recon_detail_page.max=max(1,(max(len(incoming),len(outgoing),len(comparison))+preview_input.value-1)//preview_input.value)
        if recon_detail_page.value>recon_detail_page.max:recon_detail_page.value=recon_detail_page.max
        first=(recon_detail_page.value-1)*preview_input.value
        if comparison.empty:print("SKU/lot quantity assessment withheld: "+row.Reason)
        else:
            print(f"SKU/LOT COMPARISON: {len(comparison):,} pairs")
            display(comparison.iloc[first:first+preview_input.value])
        for label,frame,side,cols in [("INBOUND EVIDENCE",incoming,"Inbound",["WMS ASN#","Ref Line#","LPN","SKU","Vendor Lot","Qty Rcvd","Received (+8h) [GR Date & Time]"]),
            ("OUTBOUND EVIDENCE",outgoing,"Outbound",["Order Key","Order Line Number","Carton ID","SKU","Vendor Lot","Qty Shipped",created_col,released_col])]:
            print(f"{label}: {len(frame):,} source rows; page {recon_detail_page.value}/{recon_detail_page.max}")
            shown=source_records_for_display(frame.iloc[first:first+preview_input.value],side,recon_selected_time)
            if shown.empty:print("No rows on this page.")
            else:
                shown=shown[[c for c in cols if c in shown]].copy();shown.insert(0,"Source row",frame.iloc[first:first+preview_input.value].index);display(shown)
    replace_output(recon_detail_output,render)

def refresh_recon(change=None):
    global recon_busy,recon_cache,recon_cache_key,selected_recon_orders
    if recon_busy:return
    recon_busy=True
    try:
        key=(recon_selected_time,recon_from.value,recon_to.value)
        if key!=recon_cache_key:
            try:recon_cache=milk_reconciliation(*key);recon_cache_key=key
            except ValueError as error:
                selected_recon_orders=pd.DataFrame();evidence_options(recon_pick,[]);recon_detail_output.outputs=()
                replace_output(recon_output,lambda:print(str(error)));return
        full,_=recon_cache
        if change is not None and change.get("owner") is not recon_page:recon_page.value=1
        table=full.copy()
        if recon_status.value=="Exactly-double pattern":table=table.loc[pd.to_numeric(table["Exactly-double SKU/lot pairs"],errors="coerce").gt(0)]
        elif recon_status.value!="All orders":table=table.loc[table.Status.eq(recon_status.value)]
        if recon_search.value.strip():table=table.loc[table[["Order [Order Key]","ASNs [WMS ASN#]"]].astype("string").apply(lambda s:s.str.contains(recon_search.value.strip(),case=False,regex=False,na=False)).any(axis=1)]
        selected_recon_orders=table
        recon_page.max=max(1,(len(table)+preview_input.value-1)//preview_input.value);recon_page.value=min(recon_page.value,recon_page.max)
        first=(recon_page.value-1)*preview_input.value;shown=table.iloc[first:first+preview_input.value]
        evidence_options(recon_pick,[(str(row["Order [Order Key]"])+" — "+row.Status,str(row["Order [Order Key]"])) for _,row in shown.iterrows()])
        recon_detail_page.value=1
        def render():
            n=len(full);matched=int(full.Status.eq("Reconciled").sum());mismatch=int(full.Status.eq("Mismatch").sum());assessable=matched+mismatch
            print(f"MILK RUN QUANTITY RECONCILIATION AS OF {recon_selected_time}; orders created {recon_from.value} to {recon_to.value}, inclusive")
            display(pd.DataFrame({"Metric":["Selected orders","Reconciled","Mismatched","Partially linked","Cannot assess","Assessable fully linked orders","Reconciled / assessable (%)","Orders with an exactly-double SKU/lot pattern"],
                "Value":[n,matched,mismatch,int(full.Status.eq("Partially linked").sum()),int(full.Status.eq("Cannot assess").sum()),assessable,100*matched/assessable if assessable else np.nan,int(pd.to_numeric(full["Exactly-double SKU/lot pairs"],errors="coerce").gt(0).sum())]}).round(2))
            print("The percentage excludes partial/unassessable orders. Summary covers the entire selected cohort, before Result/search filters.")
            print(f"{len(table):,} matching orders; page {recon_page.value}/{recon_page.max}")
            display(shown)
        replace_output(recon_output,render)
    finally:recon_busy=False
    show_recon_detail()

def apply_recon_time(change=None):
    global recon_selected_time
    try:candidate=pd.Timestamp.combine(recon_day.value,parse_clock_time(recon_time.value))
    except (ValueError,TypeError):recon_message.value="Enter time as HH:MM or HH:MM:SS. Results unchanged.";return
    recon_selected_time=candidate;recon_message.value="";refresh_recon()
recon_day.observe(apply_recon_time,names="value");recon_apply.on_click(apply_recon_time)
for control in [recon_from,recon_to,recon_status,recon_search,recon_page,preview_input]:control.observe(refresh_recon,names="value")
recon_pick.observe(show_recon_detail,names="value");recon_detail_page.observe(show_recon_detail,names="value")
display(widgets.VBox([widgets.HBox([recon_day,recon_time,recon_apply]),recon_message,widgets.HBox([recon_from,recon_to]),
    widgets.HBox([recon_status,recon_search]),recon_page,recon_output,recon_pick,recon_detail_page,recon_detail_output]))
refresh_recon()


## 8. Needs attention and suggested checks

> **Review 16 Nov 2026 (prototype handover)**  
> 🟦 **In the prototype:** All four checks exist in the prototype: IN-01, OUT-01, IN-03, DQ-01.  
> 🟧 **Overlaps but differs:** The prototype groups OUT-01 by country × forwarder and adds rules the notebook lacks (IN-02, IN-04, IN-05, OUT-02 to OUT-06, DQ-02 to DQ-07, MGR-01 to MGR-04).  
> 🟥 **Changed in this version:** Rule IDs renamed to the prototype names.  

**Purpose:** Bring existing exceptions into one list with supporting facts and a practical follow-up. These are explicit rule tests, not predictions, causes or automatic actions.

Choose this section's independent as-of date/time. Check and partial ID/SKU text narrow the list; the summary counts remain before those filters. Search can also match an order, carton/LPN, final forwarder or shipment reference. Select a result for its rule facts, suggested check, line tables and recorded history at this cutoff. Larger results use pages.

| Rule | Trigger | Suggested investigation |
|---|---|---|
| ~~ASN-01~~ **IN-01**: Long receipt wait | Docked ASN, no receipt observed and current wait above completed same-type P90 observed by cutoff. | Confirm receipt status and missing confirmation before assuming congestion. Benchmark n is shown; n < 20 is a small-sample caution, not an SLA failure. |
| ~~ORD-01~~ **OUT-01**: Open past planned day | Created order, no final handover observed, and one consistent planned day earlier than the selected day. | Review recorded readiness/unfinished work and confirm the handover plan. No wave capacity or collection schedule is assumed. |
| ~~QTY-01~~ **IN-03**: Receipt discrepancy | All line receipt rows observed, valid consistent expected quantity counted once, and summed received quantity differs. | Review receipt rows/posting history before adjustment. Exactly double is a pattern, not proof. |
| ~~DATE-01~~ **DQ-01**: Planned before creation | An observed order has one planned calendar day earlier than creation day. | Confirm the plan with its owner. The order need not still be open. |

Counts use ASNs, orders or ASN lines and must not be added into a total-problems score. One order can appear under several checks. Open-work checks describe the reconstructed current state; quantity/date checks describe observed records whose investigation/resolution status is not supplied.

Not assessed lists missing benchmarks/plans and incomplete or invalid quantity evidence separately; these are neither passes nor healthy zeros. Expected quantity is counted once per ASN/reference line, and receipt checks wait until all source receipt rows for that line are observed. Static fields and incomplete source coverage remain limitations. Suggestions do not send notifications, assign resources, alter records or establish the reason for an exception.


In [22]:
ATTENTION_CHECKS=["Long receipt wait","Open past planned day","Receipt discrepancy","Planned before creation"]
ATTENTION_COLUMNS=["Rule","Check","Kind","Record ID","Unit","Scope","Observed","Baseline / threshold","Suggested check"]

# One reusable line model; original source rows and every split receipt are retained.
attention_receipt_lines=dq_expected_lines(record_ib)
# REVIEW 16 Nov - CHANGED: 3-part inbound line key (DSV A5), matching dq_expected_lines.
# [OLD] attention_line_meta=record_ib.loc[record_ib._ASN.ne("")&record_ib._Line.ne("")].groupby(["_ASN","_Line"]).agg(
attention_line_meta=record_ib.loc[record_ib._ASN.ne("")&record_ib._Line.ne("")].groupby(["_ASN","_Line","_FP"]).agg(
    last_receipt=("GR Date & Time","max"),receipt_count=("GR Date & Time","count"),source_count=("_Line","size"),
    min_received=("Qty Rcvd",lambda v:pd.to_numeric(v,errors="coerce").min()),skus=("SKU",join_ids),lots=("Vendor Lot",join_ids),
    lpns=("_HU",join_ids),references=("Shipment Ref",join_ids)).reset_index().rename(columns={"_ASN":"ASN","_Line":"Line","_FP":"Packing"})
# REVIEW 16 Nov - CHANGED: merge on ASN + Line + Packing.
# [OLD] attention_receipt_lines=attention_receipt_lines.merge(attention_line_meta,on=["ASN","Line"],how="left",validate="one_to_one")
attention_receipt_lines=attention_receipt_lines.merge(attention_line_meta,on=["ASN","Line","Packing"],how="left",validate="one_to_one")

def attention_readiness(at,opened):
    pairs=kpi_order_cartons.loc[kpi_order_cartons.created.le(at)&kpi_order_cartons["Order Key"].isin(opened["Order Key"])].copy()
    pairs["_ready"]=pairs.ready_time.le(at)
    counts=pairs.groupby("Order Key").agg(recorded_cartons=("Carton ID","nunique"),ready_cartons=("_ready","sum"))
    out=opened.merge(order_search_attributes,on="Order Key",how="left",validate="one_to_one").merge(counts,on="Order Key",how="left",validate="one_to_one")
    for col in ["recorded_cartons","ready_cartons"]:out[col]=out[col].fillna(0).astype(int)
    observed=record_ob.loc[record_ob[created_col].le(at)]
    missing_units=set(observed.loc[observed._HU.eq(""),"_Order"])
    out["Recorded readiness"]=out.ready_cartons.astype(str)+" / "+out.recorded_cartons.astype(str)+" recorded cartons ready"
    out.loc[out.recorded_cartons.eq(0)|out["Order Key"].map(canonical_record_id).isin(missing_units),"Recorded readiness"]="Unknown/incomplete: missing carton IDs"
    out["Age (hours)"]=(pd.Timestamp(at)-out.created).dt.total_seconds()/3600
    return out

def needs_attention(at):
    t=pd.Timestamp(at);records=[];details={};unassessed=[]
    def add(rule,check,kind,key,unit,scope,observed,threshold,action,facts,search=""):
        key=str(key);index=len(records)
        records.append({"Rule":rule,"Check":check,"Kind":kind,"Record ID":key,"Unit":unit,"Scope":scope,"Observed":observed,
            "Baseline / threshold":threshold,"Suggested check":action,"_search":str(search)})
        details[index]={"kind":kind,"key":key,"facts":pd.DataFrame(facts,columns=["Fact","Value"])}
    dock=dock_snapshot(t)
    waiting=dock.loc[dock.State.eq("Docked, awaiting receipt")]
    slow=waiting.loc[waiting["Beyond P90"]]
    for _,row in slow.iterrows():
        asn=canonical_record_id(row["WMS ASN#"]);n=int(row["Benchmark samples"])
        caution=f"; small comparison sample (n={n})" if n<20 else f"; n={n} completed ASNs"
        # REVIEW 16 Nov - CHANGED: rule ID ASN-01 -> IN-01, the prototype's name for the same rule, "Docked, not received in usual time".
        # Why: one naming scheme across notebook, deck and prototype (the deck currently mixes both).
        # [OLD] add("ASN-01","Long receipt wait","ASN",asn,"ASNs","Open work at cutoff",f"{row['Waiting (hours)']:.2f}h waiting",
        add("IN-01","Long receipt wait","ASN",asn,"ASNs","Open work at cutoff",f"{row['Waiting (hours)']:.2f}h waiting",
            f"Same-type P90 {row['Benchmark P90 (hours)']:.2f}h"+caution,
            "Confirm receipt status and missing confirmation; review dock records before assuming congestion.",
            [("Shipment type [Shipment Type]",row.shipment_type),("Docked (+8h) [Dock in Date & Time]",row.dock),
             ("Elapsed wait (hours)",row["Waiting (hours)"]),("Same-type P90 (hours)",row["Benchmark P90 (hours)"]),
             ("Completed benchmark ASNs",n),("Hours beyond P90",row["Hours beyond P90"]),("Comparison caution",caution.lstrip("; "))],row.shipment_type)
    unassessed.append({"Check":"Long receipt wait","Not assessed":int(waiting["Benchmark samples"].eq(0).sum()),"Unit":"Waiting ASNs","Reason":"No completed same-type receipt benchmark"})
    _,opened=order_snapshot(t);opened=attention_readiness(t,opened)
    plan_variants=performance_base.set_index("Order Key")["Plan variants"]
    opened["Plan variants"]=opened["Order Key"].map(plan_variants)
    valid_plan=opened.planned.notna()&opened["Plan variants"].eq(1)
    overdue=opened.loc[valid_plan&opened.planned.dt.normalize().lt(t.normalize())]
    for _,row in overdue.iterrows():
        days=(t.normalize()-row.planned.normalize()).days
        # REVIEW 16 Nov - CHANGED: rule ID ORD-01 -> OUT-01, the prototype's name for the same rule, "Past planned day" (the prototype then groups these by country x forwarder).
        # Why: one naming scheme across notebook, deck and prototype (the deck currently mixes both).
        # [OLD] add("ORD-01","Open past planned day","Order",canonical_record_id(row["Order Key"]),"Orders","Open work at cutoff",
        add("OUT-01","Open past planned day","Order",canonical_record_id(row["Order Key"]),"Orders","Open work at cutoff",
            f"{days} calendar day(s) past planned; {row['Recorded readiness']}","Planned day < selected day; no handover observed",
            "Review recorded readiness and unfinished work; confirm the handover plan with the responsible team.",
            [("Planned [Planned Post Date]",row.planned),("Created ["+created_col+"]",row.created),("Days past planned",days),
             ("Recorded readiness",row["Recorded readiness"]),("Final forwarder [Final Forwarder]",row.Final_forwarder),
             ("Order age (hours)",row["Age (hours)"])],str(row.Final_forwarder)+" "+str(row.External_orders))
    unassessed.append({"Check":"Open past planned day","Not assessed":int((~valid_plan).sum()),"Unit":"Open orders","Reason":"Missing or conflicting planned dates"})
    known_asns=set(dock.loc[dock.State.isin(["Docked, awaiting receipt","Received"]),"WMS ASN#"].map(canonical_record_id))
    lines=attention_receipt_lines.loc[attention_receipt_lines.ASN.isin(known_asns)].copy()
    complete=lines.receipt_count.eq(lines.source_count)&lines.last_receipt.le(t)
    valid=~lines["Expected unresolved"]&lines.Expected.notna()&lines.Received.notna()&np.isfinite(lines.Expected)&np.isfinite(lines.Received)&lines.Expected.ge(0)&lines.min_received.ge(0)
    checked=lines.loc[complete&valid]
    for _,row in checked.loc[~np.isclose(checked.Received,checked.Expected,rtol=0,atol=1e-9)].iterrows():
        delta=row.Received-row.Expected
        double=row.Expected>0 and np.isclose(row.Received,2*row.Expected,rtol=0,atol=1e-9)
        pattern="Exactly double — possible posting pattern" if double else "Short receipt" if delta<0 else "Over receipt"
        # REVIEW 16 Nov - CHANGED: rule ID QTY-01 -> IN-03, the prototype's name for the same rule, "Received quantity differs".
        # Why: one naming scheme across notebook, deck and prototype (the deck currently mixes both).
        # [OLD] add("QTY-01","Receipt discrepancy","ASN line",str(row.ASN)+"|"+str(row.Line),"ASN lines","Recorded data check",
        add("IN-03","Receipt discrepancy","ASN line",str(row.ASN)+"|"+str(row.Line),"ASN lines","Recorded data check",
            f"Expected {row.Expected:,.0f}; received {row.Received:,.0f}; difference {delta:+,.0f}","Received differs from expected once per ASN line",
            "Review the receipt rows and posting history; verify the quantity before any adjustment.",
            [("Pattern",pattern),("Expected once [Qty Exp]",row.Expected),("Received across source rows [Qty Rcvd]",row.Received),
             ("Difference (received - expected)",delta),("Receipt observed (+8h)",row.last_receipt),("SKUs [SKU]",row.skus),
             ("Lots [Vendor Lot]",row.lots),("Resolution status","Not recorded in these files")],row.skus+" "+row.lots+" "+row.lpns+" "+row.references)
    unassessed.extend([
        {"Check":"Receipt discrepancy","Not assessed":int((~complete).sum()),"Unit":"Lines on observed ASNs","Reason":"Not all receipt rows observed by cutoff"},
        {"Check":"Receipt discrepancy","Not assessed":int((complete&~valid).sum()),"Unit":"Completed receipt lines","Reason":"Missing/conflicting/non-finite/negative quantities"}])
    observed_orders=performance_base.loc[performance_base.created.le(t)]
    date_bad=observed_orders.loc[observed_orders["Plan variants"].eq(1)&observed_orders["Planned before created"]]
    for _,row in date_bad.iterrows():
        # REVIEW 16 Nov - CHANGED: rule ID DATE-01 -> DQ-01, the prototype's name for the same rule, "Planned before created".
        # Why: one naming scheme across notebook, deck and prototype (the deck currently mixes both).
        # [OLD] add("DATE-01","Planned before creation","Order",canonical_record_id(row["Order Key"]),"Orders","Recorded data check",
        add("DQ-01","Planned before creation","Order",canonical_record_id(row["Order Key"]),"Orders","Recorded data check",
            f"Planned {row.planned:%Y-%m-%d}; created {row.created:%Y-%m-%d %H:%M}","Planned calendar day < creation calendar day",
            "Confirm the planned date with its owner; retain the source value until verified.",
            [("Planned [Planned Post Date]",row.planned),("Created ["+created_col+"]",row.created),("Final forwarder [Final Forwarder]",row.Final_forwarder),
             ("Resolution status","Not recorded in these files")],str(row.Final_forwarder)+" "+str(row.External_orders))
    unassessed.append({"Check":"Planned before creation","Not assessed":int((~observed_orders["Plan variants"].eq(1)).sum()),"Unit":"Observed orders","Reason":"Missing/conflicting planned date"})
    unassessed.append({"Check":"Order visibility","Not assessed":int(performance_base.created.isna().sum()),"Unit":"Orders in full extract","Reason":"Creation time unavailable; cannot establish presence at a historical cutoff"})
    table=pd.DataFrame(records,columns=ATTENTION_COLUMNS+["_search"])
    table.index.name="Attention row"
    # Add observed SKU/carton/shipment references for consistent partial search.
    observed_in=record_ib.loc[record_ib[["Dock in Date & Time","GR Date & Time"]].le(t).any(axis=1)]
    observed_out=record_ob.loc[record_ob[created_col].le(t)]
    ins=observed_in.assign(_s=observed_in._SKU+" "+observed_in._HU.where(observed_in["GR Date & Time"].le(t),"")+" "+observed_in["Shipment Ref"].fillna("").astype(str)).groupby("_ASN")._s.agg(join_ids)
    outs=observed_out.assign(_s=observed_out._SKU+" "+observed_out._HU).groupby("_Order")._s.agg(join_ids)
    for index,meta in details.items():
        extra=ins.get(meta["key"],"") if meta["kind"]=="ASN" else outs.get(meta["key"],"") if meta["kind"]=="Order" else ""
        table.loc[index,"_search"]+=" "+str(extra)
    counts=[]
    units=["ASNs","Orders","ASN lines","Orders"]
    for check,unit in zip(ATTENTION_CHECKS,units):
        counts.append({"Check":check,"Count":int(table.Check.eq(check).sum()),"Unit":unit})
    return table,pd.DataFrame(counts),pd.DataFrame(unassessed),details

attention_day=widgets.Dropdown(options=[(data_date_label(d),d) for d in kpi_available_days],value=DATA_TODAY,description="As-of date:")
attention_time=widgets.Text(value="09:00:00",description="Time:",layout=widgets.Layout(width="220px"))
attention_apply=widgets.Button(description="Apply time");attention_message=widgets.HTML();attention_selected_time=pd.Timestamp(AS_OF)
attention_type=widgets.Dropdown(options=["All checks"]+ATTENTION_CHECKS,description="Check:",layout=widgets.Layout(width="400px"))
attention_search=widgets.Text(value="",description="ID / SKU contains:",continuous_update=False,layout=widgets.Layout(width="450px"))
attention_hint=widgets.HTML("Example: ASN, order key, SKU, carton/LPN, shipment reference or final forwarder. Partial text is accepted.")
attention_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Results page:")
attention_pick=widgets.Dropdown(options=[],description="Select result:",layout=widgets.Layout(width="800px"))
attention_output=widgets.Output();attention_detail=widgets.Output();attention_inspector=RecordInspector()
attention_busy=False;attention_detail_busy=False;attention_cache_key=None;attention_cache=None
for control in [attention_day,attention_time,attention_type,attention_search,attention_page,attention_pick]:
    control.style.description_width="initial";control.layout.max_width="100%"

def show_attention_detail(change=None):
    global attention_detail_busy
    if attention_busy or attention_detail_busy:return
    attention_detail_busy=True
    try:
        attention_inspector.clear()
        if attention_pick.value is None:
            replace_output(attention_detail,lambda:print("No matching attention records. This does not establish that all operations are healthy; review unassessable cases above."));return
        index=attention_pick.value;meta=attention_cache[3][index];row=attention_cache[0].loc[index]
        def render():
            print(f"{row.Rule}: {row.Check} — {meta['kind']} {meta['key'].replace('|',' / ')} — as of {attention_selected_time}")
            display(meta["facts"])
            print("Suggested check: "+row["Suggested check"])
        replace_output(attention_detail,render)
        attention_inspector.open(meta["kind"],meta["key"],attention_selected_time,origin="Needs-attention evidence: "+row.Rule)
    finally:attention_detail_busy=False

def refresh_attention(change=None):
    global attention_busy,attention_cache_key,attention_cache,selected_attention_rows
    if attention_busy:return
    attention_busy=True
    try:
        key=(attention_selected_time,)
        if attention_cache_key!=key:attention_cache=needs_attention(*key);attention_cache_key=key
        table,counts,unassessed,_=attention_cache
        selected=table.copy()
        if attention_type.value!="All checks":selected=selected.loc[selected.Check.eq(attention_type.value)]
        if attention_search.value.strip():
            query=attention_search.value.strip();selected=selected.loc[selected[["Record ID","Observed","_search"]].astype("string").apply(lambda s:s.str.contains(query,case=False,regex=False,na=False)).any(axis=1)]
        selected=selected.assign(_category=pd.Categorical(selected.Check,categories=ATTENTION_CHECKS,ordered=True)).sort_values(["_category","Record ID"]).drop(columns="_category")
        selected_attention_rows=selected
        if change is not None and change.get("owner") is not attention_page:attention_page.value=1
        attention_page.max=max(1,(len(selected)+preview_input.value-1)//preview_input.value);attention_page.value=min(attention_page.value,attention_page.max)
        start=(attention_page.value-1)*preview_input.value;shown=selected.iloc[start:start+preview_input.value]
        evidence_options(attention_pick,[(f"{r.Rule} — {r.Kind} {str(r['Record ID']).replace('|',' / ')}",int(i)) for i,r in shown.iterrows()])
        def render():
            print(f"NEEDS ATTENTION AS OF {attention_selected_time}")
            print("Counts are before Check/ID filters. Units differ; rows can overlap. There is no summed problem count.")
            display(counts)
            review=unassessed.loc[unassessed["Not assessed"].gt(0)]
            if not review.empty:print("NOT ASSESSED — these cases are not healthy zeros");display(review)
            print(f"{len(selected)} matching attention rows; page {attention_page.value}/{attention_page.max}")
            view=shown[ATTENTION_COLUMNS].drop(columns="Kind").copy()
            view["Record ID"]=view["Record ID"].astype(str).str.replace("|"," / ",regex=False)
            display(view.style.set_properties(**{"white-space":"normal","overflow-wrap":"anywhere","max-width":"230px"}))
        replace_output(attention_output,render)
    finally:attention_busy=False
    show_attention_detail()

def apply_attention_time(change=None):
    global attention_selected_time
    try:candidate=pd.Timestamp.combine(attention_day.value,parse_clock_time(attention_time.value))
    except (ValueError,TypeError):attention_message.value="Enter time as HH:MM or HH:MM:SS. Results unchanged.";return
    attention_selected_time=candidate;attention_message.value="";refresh_attention()
attention_day.observe(apply_attention_time,names="value");attention_apply.on_click(apply_attention_time)
for control in [attention_type,attention_search,attention_page,preview_input]:control.observe(refresh_attention,names="value")
attention_pick.observe(show_attention_detail,names="value")

def attention_row(*items):return widgets.HBox(list(items),layout=widgets.Layout(flex_flow="row wrap",gap="8px"))
display(widgets.VBox([attention_row(attention_day,attention_time,attention_apply),attention_message,
    attention_row(attention_type,attention_search),attention_hint,attention_page,attention_output,
    attention_pick,attention_detail,attention_inspector.widget]))
refresh_attention()

AttributeError: The '.style' accessor requires jinja2

## 9. Historical what-if: earlier handover and readiness

> **Review 16 Nov 2026 (prototype handover)**  
> 🟨 **Not in the prototype yet:** This earlier-handover what-if. The prototype what-if is about **staffing** (volume +X% → people needed, about 505 lines per person per day). Decide: keep both, or move this one into the planner screen.  

**Purpose:** Ask how recorded lateness and waiting-carton counts would change under an earlier-handover rule. This is a historical counterfactual, **not a forecast, capacity plan or proof that the schedule could be achieved**. Actual source events remain unchanged.

**Choose the cohort:** Set the independent planned-date range, DN type, All/one recorded final-forwarder group and late definition A–D. This view uses **full-extract actual outcomes**, not a 09:00 snapshot. The initial period is the preceding seven planned days before Dataset Today where available, reducing the obvious final-day bias without proving cohort completeness. Selected, excluded and measurable counts are displayed. Missing/conflicting plans and incomplete actual handovers cannot support measurable outcomes.

### Scenario controls and bounds

| Mode | Meaning of X and calculation |
|---|---|
| X hours earlier | Request actual final handover − X hours. Clip to the earliest allowed time; never delay actual handover. X = 0 leaves history unchanged. |
| Within X hours of readiness | Request earliest allowed time + X hours; use it only if earlier than actual. X is allowed waiting after the bound, not a subtraction from actual. X = 0 requests immediate handover at the bound. |

Earliest allowed = later of creation and derived order readiness. Non-milk cartons use latest recorded packing; milk cartons use latest Cyclone scan, with latest adjusted receipt only as an unambiguous exact-ID fallback. SKU/lot inferred links are not used. Order readiness is the latest effective readiness among required recorded carton pairs. Cartons must have known readiness/handover and unambiguous recorded ownership/flow. A missing ID/time, ambiguous assignment or invalid actual chronology leaves the order unchanged with a reason; it stays in the before/after denominator when its actual outcome is measurable. Creation is a proxy lower bound; a distinct floor-release event and physical capacity are unavailable.

### Reading the results

Actual and scenario use the **same measurable cohort**. Orders moved earlier can include already-on-time orders; only Late → on time counts deadline crossings. Still late means not fixed by this setting, not necessarily impossible. Requested-move limitation counts for readiness and creation can overlap. Grouped DN-type impact uses the same cohort. Inspect/search narrows evidence only, not totals; pages control its displayed slice.

| Evidence field | Interpretation |
|---|---|
| Actual / Scenario handover | Recorded outcome versus computed counterfactual time. |
| Recorded order readiness / Earliest allowed | System-event proxy versus the binding creation/readiness lower bound. An early scan is not proof of physically idle stock. |
| Hours earlier | Achieved shift after clipping; it can be smaller than requested. |
| Hours earlier needed | Theoretical actual-to-on-time-deadline shift for an actually-late order; not a promise it is achievable. |
| Actual late / Scenario late / Late to on time | Classification using the selected definition, and whether it changes from late to not late. |
| Known bound after deadline | Even the recorded earliest allowed time is too late. A larger handover-only shift cannot fix it within these bounds. |
| Reason | How the move was limited or why the order remained unchanged. |

Deadline A is the last second before next calendar day; B is the last second before two calendar days later; C is 18:00 on planned day; D is noon next day. The one-second convention is an explicit calculation assumption matching source timestamp resolution. For example, an order planned on 17 August with readiness on 18 August remains late under A even with immediate handover. If the requested shift is merely too small, a larger X may instead cross the deadline.

### Charts, backlog and source evidence

All scenario charts start hidden. Sensitivity repeats the calculation over X; in readiness mode its x-axis is **allowed waiting time**, so its interpretation differs from hours earlier. The required-hours histogram shows theoretical shifts. Hourly backlog compares actual and scenario waiting counts for the selected cohort's comparable identifiable cartons, not the whole warehouse.

Waiting intervals are [effective readiness/creation, handover). Each eligible order's shift is applied to its cartons and clipped to each carton's bounds. IDs count once, not once per SKU row. Peaks are calculated at exact event times; hourly curves can miss intra-hour peaks. Included and identifiable-excluded counts are shown; blank IDs prevent an exhaustive excluded-carton count. Missing coverage is unavailable, not zero. Queue history can extend outside the planned-date window because those selected orders may have become ready earlier or handed over later.

Start with X hours earlier at zero, then try a nonzero shift. Inspect Late to on time, Still late or constraint-held orders to understand the effect. Select an order and Show actual source history for its real records; hypothetical scenario timestamps are not inserted into that history. No labour, equipment, dock capacity or collection availability is modelled, so a numerical improvement is not an operational commitment.


In [ ]:
# Strict, recorded-readiness model for historical scenarios; no inferred SKU/lot links.
def prepare_scenario_readiness():
    source=record_ob.loc[record_ob._Order.ne("")].copy()
    source["_milk"]=source["DN Type"].astype("string").str.contains("milkrun",case=False,na=False)
    pairs=source.groupby(["_Order","_HU"],dropna=False).agg(pair_created=(created_col,"max"),actual_carton_handover=(released_col,"max"),
        handover_count=(released_col,"count"),source_rows=("_Order","size"),packed=(packed_col,"max"),scan=(scan_col,"max"),
        flow_variants=("DN Type","nunique"),is_milk=("_milk","max")).reset_index()
    owners=source.loc[source._HU.ne("")].groupby("_HU")._Order.nunique()
    unknown_owners=set(record_ob.loc[record_ob._Order.eq("")&record_ob._HU.ne(""),"_HU"])
    inbound_link=record_ib.loc[record_ib._HU.ne("")].groupby("_HU").agg(receipt=("GR Date & Time","max"),asn_count=("_ASN","nunique"),
        receipt_count=("GR Date & Time","count"),source_rows=("_HU","size"),blank_asn=("_ASN",lambda s:s.eq("").any()))
    good=inbound_link.asn_count.eq(1)&inbound_link.receipt_count.eq(inbound_link.source_rows)&~inbound_link.blank_asn
    receipt=inbound_link.loc[good,"receipt"]
    pairs["ready_time"]=pairs.packed
    pairs["Readiness source"]="Packing [Packing Confirmation Timestamp]"
    milk=pairs.is_milk
    pairs.loc[milk,"ready_time"]=pairs.loc[milk,"scan"]
    pairs.loc[milk,"Readiness source"]="Cyclone scan ["+scan_col+"]"
    fallback=milk&pairs.ready_time.isna()
    pairs.loc[fallback,"ready_time"]=pairs.loc[fallback,"_HU"].map(receipt)
    pairs.loc[fallback,"Readiness source"]="Exact LPN receipt fallback (+8h) [GR Date & Time]"
    pairs["Constraint issue"]=""
    pairs.loc[pairs.ready_time.isna(),"Constraint issue"]="Readiness timestamp unavailable"
    pairs.loc[pairs.pair_created.isna(),"Constraint issue"]="Carton/order creation timestamp unavailable"
    pairs.loc[pairs.handover_count.ne(pairs.source_rows),"Constraint issue"]="Some actual handover timestamps unavailable"
    pairs.loc[pairs.flow_variants.ne(1),"Constraint issue"]="Missing/multiple flow labels on carton"
    pairs.loc[pairs._HU.map(owners).gt(1),"Constraint issue"]="Carton ID associated with multiple orders"
    pairs.loc[pairs._HU.isin(unknown_owners),"Constraint issue"]="Carton ID also has a missing order association"
    pairs.loc[pairs._HU.eq(""),"Constraint issue"]="Carton ID missing"
    pairs["effective_ready"]=pairs[["ready_time","pair_created"]].max(axis=1)
    bad_time=pairs.actual_carton_handover.lt(pairs.effective_ready)
    pairs.loc[bad_time,"Constraint issue"]="Actual carton handover precedes readiness/creation"
    pair_groups=pairs.groupby("_Order")
    order_ready=pair_groups.agg(order_ready=("effective_ready","max"),cartons=("_HU",lambda v:len(set(v)-{""})),
        pair_issues=("Constraint issue",lambda v:"; ".join(sorted(set(v)-{""}))))
    actual_complete=source.groupby("_Order")[released_col].agg(["size","count"])
    order_ready["Actual rows complete"]=actual_complete["size"].eq(actual_complete["count"])
    return pairs,order_ready

scenario_pairs,scenario_order_ready=prepare_scenario_readiness()
SCENARIO_MODES=["X hours earlier","Within X hours of readiness"]

def scenario_deadlines(frame,definition):
    plan=frame.planned.dt.normalize()
    if definition=="A":return plan+pd.Timedelta(days=1)-pd.Timedelta(seconds=1)
    if definition=="B":return plan+pd.Timedelta(days=2)-pd.Timedelta(seconds=1)
    if definition=="C":return plan+pd.Timedelta(hours=18)
    if definition=="D":return plan+pd.Timedelta(days=1,hours=12)
    raise ValueError("Unknown late definition")

def simulate_handover(frame,hours,mode,definition):
    if hours<0 or not np.isfinite(hours):raise ValueError("Scenario hours must be finite and nonnegative.")
    if mode not in SCENARIO_MODES:raise ValueError("Unknown scenario mode")
    result=frame.copy()
    result["Earliest allowed"]=result[["created","order_ready"]].max(axis=1)
    valid=result.pair_issues.eq("")&result.created.notna()&result.order_ready.notna()&result.released.ge(result["Earliest allowed"])
    result["Constraint assessable"]=valid
    result["Reason"]=result.pair_issues.replace("","No change requested / no earlier move under this setting")
    result.loc[result.created.isna(),"Reason"]="Order creation timestamp unavailable"
    result.loc[result.released.lt(result["Earliest allowed"]),"Reason"]="Actual handover precedes readiness/creation bound"
    request=result.released-pd.Timedelta(hours=float(hours)) if mode==SCENARIO_MODES[0] else result["Earliest allowed"]+pd.Timedelta(hours=float(hours))
    proposed=pd.concat([request,result["Earliest allowed"]],axis=1).max(axis=1)
    proposed=pd.concat([proposed,result.released],axis=1).min(axis=1)
    result["Scenario handover"]=result.released.where(~valid,proposed)
    result["Hours earlier"]=(result.released-result["Scenario handover"]).dt.total_seconds()/3600
    result["Moved earlier"]=result["Hours earlier"].gt(1e-9)
    result.loc[result["Moved earlier"],"Reason"]="Moved earlier within recorded creation/readiness bounds"
    result["Readiness limited"]=valid&request.lt(result.order_ready)
    result["Creation limited"]=valid&request.lt(result.created)
    result.loc[valid&request.lt(result["Earliest allowed"]),"Reason"]="Requested shift clipped by creation/readiness bound"
    result["Actual late"]=late_mask(result,definition)
    simulated=result.copy();simulated["released"]=result["Scenario handover"]
    result["Scenario late"]=late_mask(simulated,definition)
    result["Late to on time"]=result["Actual late"]&~result["Scenario late"]
    result["Still late"]=result["Actual late"]&result["Scenario late"]
    result["On-time deadline"]=scenario_deadlines(result,definition)
    result["Hours earlier needed"]=((result.released-result["On-time deadline"]).dt.total_seconds()/3600).clip(lower=0).where(result["Actual late"],0)
    result["Deadline reachable under known bounds"]=valid&result["Earliest allowed"].le(result["On-time deadline"])
    return result

def scenario_population(start,end,flow=ALL_FILTER,forwarder=ALL_FILTER):
    selected,_=lateness_population(start,end,flow)
    if forwarder!=ALL_FILTER:selected=selected.loc[selected.Final_forwarder.eq(forwarder)].copy()
    table=selected.merge(scenario_order_ready,left_on="Order Key",right_index=True,how="left",validate="one_to_one")
    table["pair_issues"]=table.pair_issues.fillna("No identifiable carton/readiness records")
    measurable=table["Plan variants"].eq(1)&table.released.notna()&table["Actual rows complete"].fillna(False)
    exclusions=table.loc[~measurable,["Order Key","planned","released","Plan variants","Actual rows complete"]].copy()
    return table,table.loc[measurable].copy(),exclusions

def scenario_group_impact(result):
    if result.empty:return pd.DataFrame(columns=["DN type","Orders","Actual late","Scenario late","Late to on time","Moved earlier","Actual late (%)","Scenario late (%)"])
    grouped=result.groupby("DN_types",dropna=False).agg(Orders=("Order Key","size"),Actual_late=("Actual late","sum"),Scenario_late=("Scenario late","sum"),
        Fixed=("Late to on time","sum"),Moved=("Moved earlier","sum")).reset_index()
    grouped["Actual late (%)"]=100*grouped.Actual_late/grouped.Orders
    grouped["Scenario late (%)"]=100*grouped.Scenario_late/grouped.Orders
    return grouped.rename(columns={"DN_types":"DN type","Actual_late":"Actual late","Scenario_late":"Scenario late","Fixed":"Late to on time","Moved":"Moved earlier"})

def waiting_event_counts(pairs,end_column):
    if pairs.empty:return pd.Series(dtype="int64")
    start=pairs.effective_ready;end=pairs[end_column]
    valid=start.notna()&end.notna()&end.ge(start)
    events=pd.concat([pd.Series(1,index=pd.DatetimeIndex(start.loc[valid])),pd.Series(-1,index=pd.DatetimeIndex(end.loc[valid]))])
    return events.groupby(level=0).sum().sort_index().cumsum()

def scenario_backlog(result):
    eligible=result.loc[result["Constraint assessable"]]
    pairs=scenario_pairs.loc[scenario_pairs._Order.isin(eligible["Order Key"])].copy()
    if pairs.empty:return pd.DataFrame(),float("nan"),float("nan"),0
    assert pairs._HU.is_unique,"Ambiguous carton ownership escaped the constraint guard"
    shift=eligible.set_index("Order Key")["Hours earlier"]
    request=pairs.actual_carton_handover-pd.to_timedelta(pairs._Order.map(shift),unit="h")
    pairs["Scenario carton handover"]=pd.concat([request,pairs.effective_ready],axis=1).max(axis=1)
    pairs["Scenario carton handover"]=pd.concat([pairs["Scenario carton handover"],pairs.actual_carton_handover],axis=1).min(axis=1)
    actual=waiting_event_counts(pairs,"actual_carton_handover");scenario=waiting_event_counts(pairs,"Scenario carton handover")
    if actual.empty:return pd.DataFrame(),float("nan"),float("nan"),0
    points=pd.date_range(min(actual.index.min(),scenario.index.min()).floor("h"),max(actual.index.max(),scenario.index.max()).ceil("h"),freq="h")
    hourly=pd.DataFrame({"Actual waiting cartons":actual.reindex(actual.index.union(points)).sort_index().ffill().reindex(points).fillna(0),
        "Scenario waiting cartons":scenario.reindex(scenario.index.union(points)).sort_index().ffill().reindex(points).fillna(0)})
    return hourly,int(actual.max()),int(scenario.max()),len(pairs)

# Historical cohort defaults to the preceding seven days, not the latest incomplete day.
scenario_options=[(data_date_label(d),d) for d in planned_days]
scenario_default_end=max([d for d in planned_days if d<DATA_TODAY],default=DATA_TODAY)
scenario_start_target=scenario_default_end-pd.Timedelta(days=6)
scenario_default_start=min([d for d in planned_days if d>=pd.Timestamp(scenario_start_target).date() and d<=scenario_default_end],default=scenario_default_end)
scenario_from=widgets.Dropdown(options=scenario_options,value=scenario_default_start,description="Planned from:")
scenario_to=widgets.Dropdown(options=scenario_options,value=scenario_default_end,description="Planned to:")
scenario_flow=filter_widget(kpi_out["DN Type"],"DN type:")
scenario_forwarders=sorted(set(performance_base.Final_forwarder.dropna())-{"Missing / blank","Multiple values",""})
scenario_forwarder=widgets.Dropdown(options=[("All forwarders",ALL_FILTER)]+[(s,s) for s in scenario_forwarders],description="Final forwarder:")
scenario_definition=widgets.Dropdown(options=[(v,k) for k,v in LATE_DEFINITIONS.items()],value="A",description="Late definition:",layout=widgets.Layout(width="490px"))
scenario_mode=widgets.Dropdown(options=SCENARIO_MODES,value=SCENARIO_MODES[0],description="Scenario:",layout=widgets.Layout(width="440px"))
scenario_hours=widgets.FloatSlider(value=0,min=0,max=72,step=.5,continuous_update=False,description="Hours earlier:",readout_format=".1f",layout=widgets.Layout(width="650px"))
scenario_chart_show=widgets.Checkbox(value=False,description="Show scenario chart",indent=False)
scenario_chart_kind=widgets.Dropdown(options=["Late-share sensitivity","Hours earlier needed","Hourly waiting-carton backlog"],description="Chart:",layout=widgets.Layout(width="400px"))
scenario_result_filter=widgets.Dropdown(options=["All measurable orders","Moved earlier","Late to on time","Still late","Held: constraints unavailable","Deadline before known bound"],description="Inspect:",layout=widgets.Layout(width="400px"))
scenario_search=widgets.Text(description="Order contains:",continuous_update=False)
scenario_page=widgets.BoundedIntText(value=1,min=1,max=1,description="Order page:")
scenario_group_page=widgets.BoundedIntText(value=1,min=1,max=1,description="DN-type page:")
scenario_pick=widgets.Dropdown(options=[],description="Select order:",layout=widgets.Layout(width="650px"))
scenario_open=widgets.Button(description="Show actual source history")
scenario_inspector=RecordInspector(scope="Full reports")
scenario_output=widgets.Output();scenario_message=widgets.HTML();scenario_busy=False;scenario_cached_population=None;scenario_population_key=None;scenario_curve_cache={}
for control in [scenario_from,scenario_to,scenario_forwarder,scenario_definition,scenario_mode,scenario_hours,scenario_chart_kind,scenario_result_filter,scenario_search,scenario_page,scenario_group_page,scenario_pick]:
    control.style.description_width="initial";control.layout.max_width="100%"

def draw_scenario_chart(result,measurable,hourly):
    if not scenario_chart_show.value:return
    if result.empty:print("No measurable orders for a scenario chart.");return
    kind=scenario_chart_kind.value
    fig,ax=plt.subplots(figsize=(10,4.5))
    if kind=="Late-share sensitivity":
        key=(scenario_population_key,scenario_mode.value,scenario_definition.value)
        if key not in scenario_curve_cache:
            rows=[]
            for hours in range(0,73,4):
                simulated=simulate_handover(measurable,hours,scenario_mode.value,scenario_definition.value)
                rows.append({"Hours":hours,"Scenario late (%)":100*simulated["Scenario late"].mean()})
            scenario_curve_cache[key]=pd.DataFrame(rows)
        curve=scenario_curve_cache[key]
        ax.plot(curve.Hours,curve["Scenario late (%)"],marker="o",label="Scenario")
        ax.scatter([scenario_hours.value],[100*result["Scenario late"].mean()],color="#b56c28",label="Current setting",zorder=3)
        ax.axhline(100*result["Actual late"].mean(),color="#a85b4f",linestyle="--",label="Actual")
        ax.set_xlabel("Hours earlier requested" if scenario_mode.value==SCENARIO_MODES[0] else "Allowed hours after recorded readiness")
        ax.set_ylabel("Late orders (%) of the same measurable cohort");ax.legend();ax.set_title("Historical sensitivity under fixed recorded bounds")
    elif kind=="Hours earlier needed":
        values=result.loc[result["Actual late"],"Hours earlier needed"]
        if values.empty:plt.close(fig);print("No actually-late orders in this cohort.");return
        ax.hist(values,bins=min(20,max(1,len(values))),color="#176b87")
        ax.set_xlabel("Theoretical hours earlier needed to reach selected on-time deadline");ax.set_ylabel("Actually-late orders")
        ax.set_title("Required shift is not evidence that capacity or readiness permits it")
    else:
        if hourly.empty:plt.close(fig);print("No comparable identifiable cartons with valid readiness/handover intervals.");return
        for col in hourly:ax.plot(hourly.index,hourly[col],label=col)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"));ax.tick_params(axis="x",rotation=25)
        ax.set_xlabel("Historical time; hourly samples can miss event-time peaks");ax.set_ylabel("Comparable selected-cohort cartons waiting")
        ax.set_title("Recorded-ready to handover backlog — not whole-warehouse capacity");ax.legend()
    ax.grid(alpha=.2);display_operational_figure(fig)

def refresh_scenario(change=None):
    global scenario_busy,scenario_cached_population,scenario_population_key,selected_scenario_orders,scenario_all_results,selected_scenario_groups,selected_scenario_hourly
    if scenario_busy:return
    scenario_busy=True
    try:
        key=(scenario_from.value,scenario_to.value,scenario_flow.value,scenario_forwarder.value)
        if key!=scenario_population_key:
            try:scenario_cached_population=scenario_population(*key);scenario_population_key=key
            except ValueError as error:
                scenario_message.value=str(error);scenario_output.outputs=();evidence_options(scenario_pick,[]);scenario_inspector.clear();return
        scenario_message.value="";selected,measurable,excluded=scenario_cached_population
        result=simulate_handover(measurable,scenario_hours.value,scenario_mode.value,scenario_definition.value);scenario_all_results=result
        groups=scenario_group_impact(result);selected_scenario_groups=groups
        hourly,actual_peak,scenario_peak,carton_n=scenario_backlog(result);selected_scenario_hourly=hourly
        known_cartons=scenario_pairs.loc[scenario_pairs._Order.isin(result["Order Key"])&scenario_pairs._HU.ne(""),"_HU"].nunique()
        excluded_cartons=known_cartons-carton_n
        scenario_hours.description="Hours earlier:" if scenario_mode.value==SCENARIO_MODES[0] else "Hours after ready:"
        view=result.copy();choice=scenario_result_filter.value
        if choice=="Moved earlier":view=view.loc[view["Moved earlier"]]
        elif choice=="Late to on time":view=view.loc[view["Late to on time"]]
        elif choice=="Still late":view=view.loc[view["Still late"]]
        elif choice=="Held: constraints unavailable":view=view.loc[~view["Constraint assessable"]]
        elif choice=="Deadline before known bound":view=view.loc[view["Constraint assessable"]&view["Earliest allowed"].gt(view["On-time deadline"])]
        if scenario_search.value.strip():view=view.loc[view["Order Key"].astype("string").str.contains(scenario_search.value.strip(),case=False,regex=False,na=False)]
        selected_scenario_orders=view
        if change is not None and change.get("owner") is not scenario_page:scenario_page.value=1
        for control,table in [(scenario_page,view),(scenario_group_page,groups)]:control.max=max(1,(len(table)+preview_input.value-1)//preview_input.value);control.value=min(control.value,control.max)
        first=(scenario_page.value-1)*preview_input.value;shown=view.iloc[first:first+preview_input.value]
        evidence_options(scenario_pick,[(str(k),str(k)) for k in shown["Order Key"]])
        def render():
            n=len(result);before=int(result["Actual late"].sum());after=int(result["Scenario late"].sum())
            print(f"HISTORICAL WHAT-IF — planned {scenario_from.value} to {scenario_to.value}, inclusive; full-extract actual outcomes; definition {scenario_definition.value}")
            print(f"{scenario_mode.value}; X={scenario_hours.value:g}h; forwarder scope: {scenario_forwarder.label}. Not a forecast or a capacity-feasible schedule.")
            if scenario_to.value>=DATA_TODAY or scenario_from.value<=min(planned_days):print("EDGE WARNING: this selection touches an extract boundary; observed completed outcomes can be biased.")
            display(pd.DataFrame({"Metric":["Selected planned orders","Excluded outcome/plan records","Measurable comparison orders","Constraint-assessable orders","Held unchanged: constraints unavailable/inconsistent","Orders moved earlier","Late → on time","Late orders not fixed by this setting","Requested move limited by readiness","Requested move limited by creation","Known bound already after on-time deadline","Comparable cartons for backlog","Identifiable selected cartons excluded from backlog"],
                "Value":[len(selected),len(excluded),n,int(result["Constraint assessable"].sum()),int((~result["Constraint assessable"]).sum()),int(result["Moved earlier"].sum()),int(result["Late to on time"].sum()),int(result["Still late"].sum()),int(result["Readiness limited"].sum()),int(result["Creation limited"].sum()),int((result["Constraint assessable"]&result["Earliest allowed"].gt(result["On-time deadline"])).sum()),carton_n,excluded_cartons]}))
            display(pd.DataFrame({"Measure":["Late orders","Late (%)","On-time/not-late (%)","Exact peak waiting cartons (comparable subset)"],
                "Actual":[before,100*before/n if n else np.nan,100*(n-before)/n if n else np.nan,actual_peak],
                "Scenario":[after,100*after/n if n else np.nan,100*(n-after)/n if n else np.nan,scenario_peak]}).round(2))
            print(f"Late-order reduction: {before-after}. Readiness/creation limitation counts can overlap. Missing backlog coverage is unavailable, not zero; blank carton IDs prevent an exhaustive excluded-carton count.")
            reasons=result.loc[~result["Constraint assessable"],"Reason"].value_counts().rename_axis("Held unchanged reason").to_frame("Orders")
            if not reasons.empty:display(reasons)
            if not excluded.empty:
                print(f"Excluded outcome/plan evidence — first {preview_input.value} of {len(excluded)} records (unfiltered by Inspect/search)");display(excluded.head(preview_input.value))
            print("IMPACT BY DN TYPE — same complete comparison cohort")
            first=(scenario_group_page.value-1)*preview_input.value;display(groups.iloc[first:first+preview_input.value].round(2))
            draw_scenario_chart(result,measurable,hourly)
            print(f"ORDER EVIDENCE: {len(view)} matches; page {scenario_page.value}/{scenario_page.max}. These filters do not change comparison totals.")
            cols=["Order Key","DN_types","Final_forwarder","planned","released","order_ready","Earliest allowed","Scenario handover","Hours earlier","Hours earlier needed","Actual late","Scenario late","Late to on time","Reason"]
            display(shown[cols].rename(columns={"DN_types":"Flow [DN Type]","Final_forwarder":"Final [Final Forwarder]","planned":"Planned [Planned Post Date]",
                "released":"Actual handover ["+released_col+"]","order_ready":"Recorded order readiness (derived)"}).round(3))
        replace_output(scenario_output,render);scenario_inspector.clear()
    finally:scenario_busy=False

def open_scenario_history(_):
    if scenario_pick.value is not None:scenario_inspector.open("Order",scenario_pick.value,full=True,origin="Actual recorded history; scenario timestamps are separate counterfactual values.")
scenario_open.on_click(open_scenario_history)
for control in [scenario_from,scenario_to,scenario_flow,scenario_forwarder,scenario_definition,scenario_mode,scenario_hours,scenario_chart_show,scenario_chart_kind,scenario_result_filter,scenario_search,scenario_page,scenario_group_page,preview_input]:control.observe(refresh_scenario,names="value")
def scenario_row(*items):return widgets.HBox(list(items),layout=widgets.Layout(flex_flow="row wrap",gap="8px"))
display(widgets.VBox([scenario_row(scenario_from,scenario_to),scenario_row(scenario_flow,scenario_forwarder),scenario_definition,scenario_row(scenario_mode,scenario_hours),
    scenario_message,scenario_row(scenario_chart_show,scenario_chart_kind),scenario_row(scenario_result_filter,scenario_search),scenario_row(scenario_group_page,scenario_page),
    scenario_output,scenario_row(scenario_pick,scenario_open),scenario_inspector.widget]))
refresh_scenario()
